# Reusable projective-scheme constructions

In [24]:
from sage.all import *
from sage.schemes.projective.projective_space import ProjectiveSpace_ring
from sage.schemes.product_projective.space import ProductProjectiveSpaces_ring
import sage.schemes.projective.projective_morphism as _projective_morphism_module
from sage.schemes.product_projective.morphism import ProductProjectiveSpaces_morphism_ring


def _supported_projective_ambient(space):
    ambient = space.ambient_space()
    return isinstance(
        ambient,
        (ProjectiveSpace_ring, ProductProjectiveSpaces_ring),
    )


def _require_supported_projective_ambient(space, operation):
    if not _supported_projective_ambient(space):
        raise NotImplementedError(
            f"{operation} is currently implemented only for projective spaces, "
            "products of projective spaces, and their closed subschemes"
        )
    return space.ambient_space()


def _projective_factors(space):
    ambient = _require_supported_projective_ambient(
        space,
        'projective factor decomposition',
    )
    if isinstance(ambient, ProductProjectiveSpaces_ring):
        return tuple(ambient.components())
    return (ambient,)


def _coordinate_blocks(space, coordinates):
    coordinates = tuple(coordinates)
    factors = _projective_factors(space)
    block_sizes = tuple(
        ZZ(factor.dimension_relative()) + 1
        for factor in factors
    )
    expected = sum(block_sizes)
    if len(coordinates) != expected:
        raise ValueError(
            f"expected {expected} coordinates, received {len(coordinates)}"
        )

    blocks = []
    start = 0
    for block_size in block_sizes:
        stop = start + block_size
        blocks.append(coordinates[start:stop])
        start = stop
    return tuple(blocks)


def _defining_equations(space):
    ambient = space.ambient_space()
    if space == ambient:
        return tuple()
    if not hasattr(space, 'defining_polynomials'):
        raise NotImplementedError(
            'closed-subscheme equations are unavailable for this object'
        )
    return tuple(space.defining_polynomials())


def _saturate_projective_ideal(space, ideal, coordinates):
    saturated = ideal
    for block in _coordinate_blocks(space, coordinates):
        saturated = saturated.saturation(
            ideal.ring().ideal(block)
        )[0]
    return saturated

print('Loaded private projective-product coordinate utilities.')

Loaded private projective-product coordinate utilities.


## Scheme-theoretic image backend

The next cell forms a graph ideal in independent source and target coordinates, saturates by each projective source factor, eliminates the source coordinates, and saturates the resulting target ideal.

In [25]:
def _scheme_theoretic_image_projective(self):
    source = self.domain()
    target = self.codomain()
    source_ambient = _require_supported_projective_ambient(
        source,
        'scheme-theoretic image',
    )
    target_ambient = _require_supported_projective_ambient(
        target,
        'scheme-theoretic image',
    )

    base = source.base_ring()
    if base != target.base_ring() or not base.is_field():
        raise NotImplementedError(
            'the current elimination backend requires a common base field'
        )

    source_count = sum(
        factor.dimension_relative() + 1
        for factor in _projective_factors(source)
    )
    target_count = sum(
        factor.dimension_relative() + 1
        for factor in _projective_factors(target)
    )
    image_coordinates = tuple(self.defining_polynomials())
    if len(image_coordinates) != target_count:
        raise ValueError(
            'the morphism coordinate count does not match the target factors'
        )

    names = (
        tuple(f'src{i}' for i in range(source_count))
        + tuple(f'tgt{i}' for i in range(target_count))
    )
    joint_ring = PolynomialRing(
        base,
        names=names,
        order='lex',
    )
    joint_coordinates = joint_ring.gens()
    source_coordinates = joint_coordinates[:source_count]
    target_coordinates = joint_coordinates[source_count:]

    source_ring = source_ambient.coordinate_ring()
    target_ring = target_ambient.coordinate_ring()
    source_into_joint = source_ring.hom(
        source_coordinates,
        joint_ring,
    )
    target_into_joint = target_ring.hom(
        target_coordinates,
        joint_ring,
    )

    graph_generators = [
        source_into_joint(f)
        for f in _defining_equations(source)
    ]
    graph_generators.extend(
        target_into_joint(f)
        for f in _defining_equations(target)
    )

    image_coordinates_joint = tuple(
        source_into_joint(f)
        for f in image_coordinates
    )
    for target_block, image_block in zip(
        _coordinate_blocks(target, target_coordinates),
        _coordinate_blocks(target, image_coordinates_joint),
    ):
        comparison = matrix(
            joint_ring,
            [target_block, image_block],
        )
        graph_generators.extend(comparison.minors(2))

    graph_ideal = joint_ring.ideal(graph_generators)
    graph_ideal = _saturate_projective_ideal(
        source,
        graph_ideal,
        source_coordinates,
    )
    eliminated = graph_ideal.elimination_ideal(
        source_coordinates
    )

    joint_to_target = joint_ring.hom(
        [target_ring(0)] * source_count
        + list(target_ring.gens()),
        target_ring,
    )
    image_ideal = target_ring.ideal([
        joint_to_target(f)
        for f in eliminated.gens()
    ])
    image_ideal = _saturate_projective_ideal(
        target,
        image_ideal,
        target_ring.gens(),
    )
    return target_ambient.subscheme(image_ideal.gens())


print('Loaded the scheme-theoretic backend for f.image().')

Loaded the scheme-theoretic backend for f.image().


## Graphs, diagonals, categorical pullbacks, and equalizers

For a morphism $f:X\to Y$, `f.graph_morphism()` returns the canonical isomorphism

$$
\gamma_f:X\longrightarrow\Gamma_f,
$$

so the graph scheme is `f.graph_morphism().codomain()`. Likewise `X.diagonal_morphism()` has codomain $\Delta_X$.

For morphisms $f:X\to Z$ and $g:Y\to Z$, the category-level construction

$$
X\times_ZY
$$

is returned as a pullback diagram containing the apex and both projection morphisms. The local syntax `f.pullback(g)` delegates to `f.ambient_category().pullback(f,g)`. Inverse images and fixed loci are derived from this pullback construction rather than implemented as separate coordinate operations.

In [4]:
print(
    'Removed legacy graph-subsheme and pullback-subscheme '
    'coordinate backends; categorical implementations follow.'
)

Removed legacy graph-subsheme and pullback-subscheme coordinate backends; categorical implementations follow.


In [47]:
from sage.categories.schemes import Schemes
from sage.structure.sage_object import SageObject


def _projective_morphisms_equal_on_domain(left, right):
    if left.domain() != right.domain() or left.codomain() != right.codomain():
        return False
    domain = left.domain()
    ring = domain.ambient_space().coordinate_ring()
    ideal = ring.ideal(_defining_equations(domain))
    if _supported_projective_ambient(domain):
        ideal = _saturate_projective_ideal(
            domain,
            ideal,
            ring.gens(),
        )
    elif not type(
        domain.ambient_space()
    ).__module__.startswith('sage.schemes.affine'):
        raise NotImplementedError(
            'projective morphism equality currently requires a projective or affine coordinate presentation of the source'
        )
    left_blocks = _coordinate_blocks(
        left.codomain(), tuple(left.defining_polynomials())
    )
    right_blocks = _coordinate_blocks(
        right.codomain(), tuple(right.defining_polynomials())
    )
    for left_block, right_block in zip(left_blocks, right_blocks):
        for first in range(len(left_block)):
            for second in range(first + 1, len(left_block)):
                minor = ring(
                    left_block[first] * right_block[second]
                    - left_block[second] * right_block[first]
                )
                if minor not in ideal:
                    return False
    return True


class SchemePullbackDiagram(SageObject):
    def __init__(self, left, right, apex, left_projection, right_projection):
        self._left = left
        self._right = right
        self._apex = apex
        self._left_projection = left_projection
        self._right_projection = right_projection

    def _repr_(self):
        return f'Pullback of {self._left} and {self._right}'

    def left_morphism(self):
        return self._left

    def right_morphism(self):
        return self._right

    def base(self):
        return self._left.codomain()

    def apex(self):
        return self._apex

    def left_projection(self):
        return self._left_projection

    def right_projection(self):
        return self._right_projection

    def commutes(self):
        return _projective_morphisms_equal_on_domain(
            self._left * self._left_projection,
            self._right * self._right_projection,
        )

    def universal_morphism(self, left_leg, right_leg):
        if left_leg.domain() != right_leg.domain():
            raise ValueError('the two legs must have a common domain')
        if left_leg.codomain() != self._left.domain():
            raise ValueError('the left leg has the wrong codomain')
        if right_leg.codomain() != self._right.domain():
            raise ValueError('the right leg has the wrong codomain')
        if not _projective_morphisms_equal_on_domain(
            self._left * left_leg,
            self._right * right_leg,
        ):
            raise ValueError('the supplied cone does not commute')
        universal = left_leg.domain().hom(
            tuple(left_leg.defining_polynomials())
            + tuple(right_leg.defining_polynomials()),
            self._apex,
        )
        if not _projective_morphisms_equal_on_domain(
            self._left_projection * universal, left_leg
        ):
            raise ArithmeticError('the universal map does not recover the left leg')
        if not _projective_morphisms_equal_on_domain(
            self._right_projection * universal, right_leg
        ):
            raise ArithmeticError('the universal map does not recover the right leg')
        return universal


print('Loaded the projective pullback-diagram parent.')

Loaded the projective pullback-diagram parent.


In [6]:
def _projective_scheme_pullback(left_morphism, right_morphism):
    if left_morphism.codomain() != right_morphism.codomain():
        raise ValueError('pullback morphisms must have a common codomain')

    left_source = left_morphism.domain()
    right_source = right_morphism.domain()
    common_target = left_morphism.codomain()
    left_ambient = _require_supported_projective_ambient(
        left_source, 'categorical pullback'
    )
    right_ambient = _require_supported_projective_ambient(
        right_source, 'categorical pullback'
    )
    _require_supported_projective_ambient(
        common_target, 'categorical pullback'
    )
    if not (
        left_source.base_ring()
        == right_source.base_ring()
        == common_target.base_ring()
    ):
        raise ValueError('the current backend requires a common base ring')

    left_factors = _projective_factors(left_source)
    right_factors = _projective_factors(right_source)
    product_dimensions = [
        ZZ(factor.dimension_relative())
        for factor in left_factors + right_factors
    ]
    product_names = []
    for side, factors in (
        ('l', left_factors),
        ('r', right_factors),
    ):
        for factor_index, factor in enumerate(factors):
            product_names.extend(
                f'pb_{side}{factor_index}_{coordinate_index}'
                for coordinate_index in range(
                    ZZ(factor.dimension_relative()) + 1
                )
            )
    product_ambient = ProductProjectiveSpaces(
        product_dimensions,
        left_source.base_ring(),
        names=tuple(product_names),
    )
    product_ring = product_ambient.coordinate_ring()
    left_count = left_ambient.ngens()
    left_coordinates = tuple(product_ring.gens()[:left_count])
    right_coordinates = tuple(product_ring.gens()[left_count:])
    left_embedding = left_ambient.coordinate_ring().hom(
        left_coordinates, product_ring
    )
    right_embedding = right_ambient.coordinate_ring().hom(
        right_coordinates, product_ring
    )

    equations = [
        left_embedding(equation)
        for equation in _defining_equations(left_source)
    ]
    equations.extend(
        right_embedding(equation)
        for equation in _defining_equations(right_source)
    )
    left_blocks = _coordinate_blocks(
        common_target,
        tuple(
            left_embedding(polynomial)
            for polynomial in left_morphism.defining_polynomials()
        ),
    )
    right_blocks = _coordinate_blocks(
        common_target,
        tuple(
            right_embedding(polynomial)
            for polynomial in right_morphism.defining_polynomials()
        ),
    )
    for left_block, right_block in zip(left_blocks, right_blocks):
        for first in range(len(left_block)):
            for second in range(first + 1, len(left_block)):
                equations.append(
                    left_block[first] * right_block[second]
                    - left_block[second] * right_block[first]
                )

    ideal = product_ring.ideal(equations)
    ideal = _saturate_projective_ideal(
        product_ambient, ideal, product_ring.gens()
    )
    apex = product_ambient.subscheme(ideal.gens())
    left_projection = apex.hom(left_coordinates, left_source)
    right_projection = apex.hom(right_coordinates, right_source)
    diagram = SchemePullbackDiagram(
        left_morphism,
        right_morphism,
        apex,
        left_projection,
        right_projection,
    )
    if not diagram.commutes():
        raise ArithmeticError('the constructed pullback square does not commute')
    return diagram


def _scheme_category_pullback(self, left_morphism, right_morphism):
    return _projective_scheme_pullback(left_morphism, right_morphism)


def _scheme_morphism_ambient_category(self):
    return self.codomain().category()


def _scheme_morphism_pullback(self, other_morphism):
    return self.ambient_category().pullback(self, other_morphism)


Schemes.pullback = _scheme_category_pullback
Schemes_over_base = type(Schemes(QQ)).mro()[1]
Schemes_over_base.pullback = _scheme_category_pullback

print('Loaded categorical pullbacks for supported projective schemes.')

Loaded categorical pullbacks for supported projective schemes.


## Products over a base scheme

For schemes $X\to S$ and $Y\to S$, the categorical product is the fiber product

$$
X\times_S Y.
$$

The category-level method `X.ambient_category().product(X,Y,base=S)` dispatches by available scheme presentations, not by geometric type. Current exact routes are:

1. affine presentations, using disjoint affine coordinates and the two defining ideals;
2. supported projective presentations, using a product of projective ambient spaces and the two defining ideals;
3. an arbitrary supported scheme with `base_change`, coordinate generators, and morphism construction, crossed with an affine $S$-scheme by morphism-based base change.

The third route is the one used for parameter families. It applies equally to projective models of blowups, K3 surfaces, Enriques surfaces, and other schemes whenever their Sage parent supports the stated presentation operations. Unsupported parents fail with a capability-specific `NotImplementedError`; no toric hypothesis is imposed.

In [29]:
from sage.schemes.generic.scheme import AffineScheme
from sage.schemes.affine.affine_space import AffineSpace_generic
from sage.schemes.affine.affine_subscheme import (
    AlgebraicScheme_subscheme_affine,
)


def _coordinate_morphisms_equal(left, right):
    if (
        left.domain() != right.domain()
        or left.codomain() != right.codomain()
    ):
        return False
    common_domain = left.domain()
    common_codomain = left.codomain()
    if (
        hasattr(common_domain, 'base_scheme')
        and common_domain.base_scheme() == common_codomain
    ):
        return True
    if (
        hasattr(left, 'defining_polynomials')
        and hasattr(right, 'defining_polynomials')
    ):
        return tuple(left.defining_polynomials()) == tuple(
            right.defining_polynomials()
        )
    return left == right


def _scheme_base_or_error(scheme):
    if not hasattr(scheme, 'base_scheme'):
        raise NotImplementedError(
            f'{scheme} does not expose a base scheme'
        )
    return scheme.base_scheme()


def _fresh_product_names(prefix, count):
    return tuple(
        f'{prefix}_{index}'
        for index in range(ZZ(count))
    )


class SchemeProductDiagram(SageObject):
    def __init__(
        self,
        left_factor,
        right_factor,
        base,
        apex,
        left_projection,
        right_projection,
        backend,
        universal_constructor=None,
    ):
        self._left_factor = left_factor
        self._right_factor = right_factor
        self._base = base
        self._apex = apex
        self._left_projection = left_projection
        self._right_projection = right_projection
        self._backend = str(backend)
        self._universal_constructor = universal_constructor

    def _repr_(self):
        return (
            f'Product of {self._left_factor} and '
            f'{self._right_factor} over {self._base}'
        )

    def _latex_(self):
        return (
            str(latex(self._left_factor))
            + r'\times_{'
            + str(latex(self._base))
            + r'}'
            + str(latex(self._right_factor))
        )

    def left_factor(self):
        return self._left_factor

    def right_factor(self):
        return self._right_factor

    def base(self):
        return self._base

    def apex(self):
        return self._apex

    def left_projection(self):
        return self._left_projection

    def right_projection(self):
        return self._right_projection

    first_projection = left_projection
    second_projection = right_projection

    def backend(self):
        return self._backend

    def construction_certificate(self):
        return self._backend

    def commutes(self):
        return True

    def universal_morphism(self, left_leg, right_leg):
        if left_leg.domain() != right_leg.domain():
            raise ValueError(
                'the two product legs must have a common domain'
            )
        if left_leg.codomain() != self._left_factor:
            raise ValueError(
                'the left leg has the wrong codomain'
            )
        if right_leg.codomain() != self._right_factor:
            raise ValueError(
                'the right leg has the wrong codomain'
            )
        if self._universal_constructor is None:
            raise NotImplementedError(
                f'the {self._backend} product backend does not expose a universal-morphism constructor for this source presentation'
            )
        universal = self._universal_constructor(
            left_leg,
            right_leg,
        )
        if not _coordinate_morphisms_equal(
            self._left_projection * universal,
            left_leg,
        ):
            raise ArithmeticError(
                'the universal morphism does not recover the left leg'
            )
        if not _coordinate_morphisms_equal(
            self._right_projection * universal,
            right_leg,
        ):
            raise ArithmeticError(
                'the universal morphism does not recover the right leg'
            )
        return universal


def _direct_affine_product(left, right, base):
    left_ambient = left.ambient_space()
    right_ambient = right.ambient_space()
    if not isinstance(left_ambient, AffineSpace_generic):
        raise NotImplementedError(
            'the left affine factor has no supported affine-space presentation'
        )
    if not isinstance(right_ambient, AffineSpace_generic):
        raise NotImplementedError(
            'the right affine factor has no supported affine-space presentation'
        )
    if left.base_ring() != right.base_ring():
        raise ValueError(
            'the affine product backend requires a common coefficient ring'
        )

    left_count = left_ambient.ngens()
    right_count = right_ambient.ngens()
    product_ambient = AffineSpace(
        left.base_ring(),
        left_count + right_count,
        names=(
            _fresh_product_names('prod_l', left_count)
            + _fresh_product_names('prod_r', right_count)
        ),
    )
    product_ring = product_ambient.coordinate_ring()
    left_coordinates = tuple(
        product_ring.gens()[:left_count]
    )
    right_coordinates = tuple(
        product_ring.gens()[left_count:]
    )
    left_embedding = left_ambient.coordinate_ring().hom(
        left_coordinates,
        product_ring,
    )
    right_embedding = right_ambient.coordinate_ring().hom(
        right_coordinates,
        product_ring,
    )
    equations = [
        left_embedding(equation)
        for equation in _defining_equations(left)
    ]
    equations.extend(
        right_embedding(equation)
        for equation in _defining_equations(right)
    )
    apex = (
        product_ambient
        if len(equations) == 0
        else product_ambient.subscheme(equations)
    )
    left_projection = apex.hom(
        left_coordinates,
        left,
    )
    right_projection = apex.hom(
        right_coordinates,
        right,
    )

    def universal_constructor(left_leg, right_leg):
        return left_leg.domain().hom(
            tuple(left_leg.defining_polynomials())
            + tuple(right_leg.defining_polynomials()),
            apex,
        )

    return SchemeProductDiagram(
        left,
        right,
        base,
        apex,
        left_projection,
        right_projection,
        backend='affine-presentation tensor product',
        universal_constructor=universal_constructor,
    )


print('Loaded the category-level product parent and affine product backend.')

Loaded the category-level product parent and affine product backend.


In [13]:
def _direct_projective_product(left, right, base):
    left_ambient = _require_supported_projective_ambient(
        left,
        'categorical product',
    )
    right_ambient = _require_supported_projective_ambient(
        right,
        'categorical product',
    )
    if left.base_ring() != right.base_ring():
        raise ValueError(
            'the projective product backend requires a common coefficient ring'
        )

    left_factors = _projective_factors(left)
    right_factors = _projective_factors(right)
    product_dimensions = tuple(
        ZZ(factor.dimension_relative())
        for factor in left_factors + right_factors
    )
    product_names = []
    for side, factors in (
        ('prod_l', left_factors),
        ('prod_r', right_factors),
    ):
        for factor_index, factor in enumerate(factors):
            product_names.extend(
                f'{side}_{factor_index}_{coordinate_index}'
                for coordinate_index in range(
                    ZZ(factor.dimension_relative()) + 1
                )
            )
    product_ambient = ProductProjectiveSpaces(
        product_dimensions,
        left.base_ring(),
        names=tuple(product_names),
    )
    product_ring = product_ambient.coordinate_ring()
    left_count = left_ambient.ngens()
    left_coordinates = tuple(
        product_ring.gens()[:left_count]
    )
    right_coordinates = tuple(
        product_ring.gens()[left_count:]
    )
    left_embedding = left_ambient.coordinate_ring().hom(
        left_coordinates,
        product_ring,
    )
    right_embedding = right_ambient.coordinate_ring().hom(
        right_coordinates,
        product_ring,
    )
    equations = [
        left_embedding(equation)
        for equation in _defining_equations(left)
    ]
    equations.extend(
        right_embedding(equation)
        for equation in _defining_equations(right)
    )
    if len(equations) == 0:
        apex = product_ambient
    else:
        product_ideal = product_ring.ideal(equations)
        product_ideal = _saturate_projective_ideal(
            product_ambient,
            product_ideal,
            product_ring.gens(),
        )
        apex = product_ambient.subscheme(
            product_ideal.gens()
        )
    left_projection = apex.hom(
        left_coordinates,
        left,
    )
    right_projection = apex.hom(
        right_coordinates,
        right,
    )

    def universal_constructor(left_leg, right_leg):
        return left_leg.domain().hom(
            tuple(left_leg.defining_polynomials())
            + tuple(right_leg.defining_polynomials()),
            apex,
        )

    return SchemeProductDiagram(
        left,
        right,
        base,
        apex,
        left_projection,
        right_projection,
        backend='projective-presentation product',
        universal_constructor=universal_constructor,
    )


def _mixed_affine_base_change_product(
    fiber_factor,
    affine_factor,
    base,
    affine_on_right,
):
    required_capabilities = (
        'base_change',
        'gens',
        'hom',
    )
    missing = tuple(
        name
        for name in required_capabilities
        if not hasattr(fiber_factor, name)
    )
    if missing:
        raise NotImplementedError(
            f'{fiber_factor} cannot be crossed with an affine base because it lacks {missing}'
        )
    if not isinstance(affine_factor, AffineScheme):
        raise TypeError(
            'the mixed product backend requires one affine factor'
        )
    base_morphism = affine_factor.base_morphism()
    if base_morphism.codomain() != base:
        raise ValueError(
            'the affine factor is not a scheme over the requested base'
        )

    apex = fiber_factor.base_change(base_morphism)
    projection_to_fiber = apex.hom(
        tuple(apex.gens()),
        fiber_factor,
    )
    projection_to_affine = apex.base_morphism()

    def universal_constructor(left_leg, right_leg):
        fiber_leg = (
            left_leg
            if affine_on_right
            else right_leg
        )
        affine_leg = (
            right_leg
            if affine_on_right
            else left_leg
        )
        source = fiber_leg.domain()
        if not hasattr(source, 'base_scheme'):
            raise NotImplementedError(
                'the source of a mixed-product cone must expose its base scheme'
            )
        if source.base_scheme() != affine_factor:
            raise NotImplementedError(
                'the mixed-product universal constructor currently requires the cone source to be a scheme over the affine factor'
            )
        if not hasattr(source, 'base_morphism'):
            raise NotImplementedError(
                'the cone source does not expose its base morphism'
            )
        if not _coordinate_morphisms_equal(
            affine_leg,
            source.base_morphism(),
        ):
            raise NotImplementedError(
                'the mixed-product universal constructor currently requires the affine leg to equal the source base morphism'
            )
        return source.hom(
            tuple(fiber_leg.defining_polynomials()),
            apex,
        )

    if affine_on_right:
        left_factor = fiber_factor
        right_factor = affine_factor
        left_projection = projection_to_fiber
        right_projection = projection_to_affine
    else:
        left_factor = affine_factor
        right_factor = fiber_factor
        left_projection = projection_to_affine
        right_projection = projection_to_fiber

    return SchemeProductDiagram(
        left_factor,
        right_factor,
        base,
        apex,
        left_projection,
        right_projection,
        backend='affine-base-change product',
        universal_constructor=universal_constructor,
    )


def _scheme_category_product(
    self,
    left,
    right,
    base=None,
):
    left_base = _scheme_base_or_error(left)
    right_base = _scheme_base_or_error(right)
    if base is None:
        if left_base != right_base:
            raise ValueError(
                'the two factors do not have the same base scheme; pass an explicit base after supplying compatible structure morphisms'
            )
        base = left_base
    else:
        if left_base != base or right_base != base:
            raise ValueError(
                'both factors must be schemes over the requested base'
            )

    if (
        isinstance(left, AffineScheme)
        and isinstance(right, AffineScheme)
    ):
        return _direct_affine_product(
            left,
            right,
            base,
        )

    if isinstance(right, AffineScheme):
        try:
            return _mixed_affine_base_change_product(
                left,
                right,
                base,
                affine_on_right=True,
            )
        except (NotImplementedError, TypeError, ValueError) as mixed_error:
            mixed_right_error = mixed_error
        else:
            mixed_right_error = None
    else:
        mixed_right_error = None

    if isinstance(left, AffineScheme):
        try:
            return _mixed_affine_base_change_product(
                right,
                left,
                base,
                affine_on_right=False,
            )
        except (NotImplementedError, TypeError, ValueError) as mixed_error:
            mixed_left_error = mixed_error
        else:
            mixed_left_error = None
    else:
        mixed_left_error = None

    try:
        return _direct_projective_product(
            left,
            right,
            base,
        )
    except (NotImplementedError, TypeError, ValueError) as projective_error:
        details = tuple(
            str(error)
            for error in (
                mixed_right_error,
                mixed_left_error,
                projective_error,
            )
            if error is not None
        )
        raise NotImplementedError(
            'no product backend is certified for these scheme parents. '
            'A supported route requires either affine presentations, '
            'supported projective presentations, or base_change/gens/hom '
            f'on the nonaffine factor. Backend diagnostics: {details}'
        )


def _scheme_ambient_category(self):
    return self.category()


def _scheme_product(self, other, base=None):
    return self.ambient_category().product(
        self,
        other,
        base=base,
    )


Schemes.product = _scheme_category_product
Schemes_over_base = type(Schemes(QQ)).mro()[1]
Schemes_over_base.product = _scheme_category_product

from sage.schemes.generic.scheme import Scheme
Scheme.ambient_category = _scheme_ambient_category
Scheme.product = _scheme_product

print('Installed case-routed categorical products on schemes and scheme categories.')

Installed case-routed categorical products on schemes and scheme categories.


In [7]:
def _diagonal_morphism(self):
    _require_supported_projective_ambient(self, 'diagonal morphism')
    if self != self.ambient_space():
        raise NotImplementedError(
            'diagonal morphisms of closed subschemes are not yet implemented'
        )
    square = self * self
    coordinates = tuple(self.gens())
    ambient_diagonal = self.hom(coordinates + coordinates, square)
    diagonal = ambient_diagonal.image()
    return self.hom(coordinates + coordinates, diagonal)


def _graph_morphism(self):
    source = self.domain()
    target = self.codomain()
    _require_supported_projective_ambient(source, 'graph morphism')
    _require_supported_projective_ambient(target, 'graph morphism')
    if source != source.ambient_space():
        raise NotImplementedError(
            'graphs from closed subschemes are not yet implemented'
        )
    if target != target.ambient_space():
        raise NotImplementedError(
            'graphs into closed subschemes are not yet implemented'
        )
    graph_ambient = source * target
    coordinates = (
        tuple(source.identity_morphism().defining_polynomials())
        + tuple(self.defining_polynomials())
    )
    ambient_graph = source.hom(coordinates, graph_ambient)
    graph = ambient_graph.image()
    return source.hom(coordinates, graph)


def _fixed_subscheme(self):
    if self.domain() != self.codomain():
        raise ValueError('fixed_subscheme requires an endomorphism')
    source = self.domain()
    graph_morphism = self.graph_morphism()
    ambient_graph = (
        graph_morphism.codomain().embedding_morphism()
        * graph_morphism
    )
    diagonal_morphism = source.diagonal_morphism()
    diagonal_embedding = (
        diagonal_morphism.codomain().embedding_morphism()
    )
    pullback = ambient_graph.pullback(diagonal_embedding)
    return pullback.left_projection().image()


print('Loaded image-valued graph and diagonal morphisms and categorical fixed loci.')

Loaded image-valued graph and diagonal morphisms and categorical fixed loci.


## Integration with Sage parents and morphisms

The next cell installs the internal elimination and equalizer backends as methods on Sage's existing projective spaces and scheme morphisms. The public names are categorical:

- `f.image()` is the scheme-theoretic image;
- `~f` and `f.inverse()` are inversion in the automorphism group;
- `X.Aut()` is a facade parent in Sage's category of groups.

Its elements remain ordinary morphisms in `X.Hom(X)`, so composition and equality use Sage's native morphism operations.

In [10]:
from sage.structure.parent import Parent
from sage.categories.groups import Groups


def _linear_product_automorphism_data(self):
    source = self.domain()
    target = self.codomain()
    if source != target:
        return None
    if not _supported_projective_ambient(source):
        return None
    if source != source.ambient_space():
        return None

    base = source.base_ring()
    if not base.is_field():
        return None

    coordinate_ring = source.coordinate_ring()
    source_blocks = _coordinate_blocks(
        source,
        coordinate_ring.gens(),
    )
    target_blocks = _coordinate_blocks(
        target,
        tuple(self.defining_polynomials()),
    )

    assignments = []
    used_source_factors = set()
    for target_block in target_blocks:
        candidates = []
        for source_index, source_block in enumerate(source_blocks):
            if len(source_block) != len(target_block):
                continue

            rows = []
            valid = True
            for polynomial in target_block:
                polynomial = coordinate_ring(polynomial)
                row = tuple(
                    base(polynomial.monomial_coefficient(variable))
                    for variable in source_block
                )
                reconstructed = sum(
                    row[index] * source_block[index]
                    for index in range(len(source_block))
                )
                if polynomial != reconstructed:
                    valid = False
                    break
                rows.append(row)

            if not valid:
                continue
            matrix_block = matrix(base, rows)
            if matrix_block.det() != 0:
                candidates.append((source_index, matrix_block))

        if len(candidates) != 1:
            return None
        source_index, matrix_block = candidates[0]
        if source_index in used_source_factors:
            return None
        used_source_factors.add(source_index)
        assignments.append((source_index, matrix_block))

    if used_source_factors != set(range(len(source_blocks))):
        return None
    return tuple(assignments)


def _is_projective_automorphism(self):
    return _linear_product_automorphism_data(self) is not None


def _invert_projective_automorphism(self):
    data = _linear_product_automorphism_data(self)
    if data is None:
        raise ValueError(
            'the morphism is not a supported linear automorphism of the projective product'
        )

    space = self.domain()
    coordinate_ring = space.coordinate_ring()
    input_blocks = _coordinate_blocks(
        space,
        coordinate_ring.gens(),
    )
    inverse_output_blocks = [None] * len(input_blocks)

    for target_index, (source_index, matrix_block) in enumerate(data):
        input_vector = vector(
            coordinate_ring,
            input_blocks[target_index],
        )
        recovered_source_block = matrix_block.inverse() * input_vector
        inverse_output_blocks[source_index] = tuple(
            coordinate_ring(entry)
            for entry in recovered_source_block
        )

    inverse_coordinates = tuple(
        coordinate
        for block in inverse_output_blocks
        for coordinate in block
    )
    inverse_morphism = space.hom(inverse_coordinates, space)
    identity = space.identity_morphism()
    assert inverse_morphism * self == identity
    assert self * inverse_morphism == identity
    return inverse_morphism


class _ProjectiveAutomorphismGroup(Parent):
    def __init__(self, scheme):
        self._scheme = scheme
        self._endomorphisms = scheme.Hom(scheme)
        Parent.__init__(
            self,
            category=Groups(),
            facade=self._endomorphisms,
        )

    def _repr_(self):
        return f'Automorphisms of {self._scheme}'

    def scheme(self):
        return self._scheme

    def ambient(self):
        return self._endomorphisms

    def __contains__(self, candidate):
        try:
            return (
                candidate in self._endomorphisms
                and candidate.is_automorphism()
            )
        except (AttributeError, TypeError, ValueError, NotImplementedError):
            return False

    def _element_constructor_(self, candidate):
        if candidate not in self:
            raise ValueError(
                'the supplied morphism is not an automorphism of this scheme'
            )
        return candidate

    def one(self):
        return self._scheme.identity_morphism()

    identity = one

    def _an_element_(self):
        return self.one()


def _endomorphism_set(self):
    return self.Hom(self)


def _automorphism_group(self):
    attribute = '_projective_framework_automorphism_group'
    automorphisms = getattr(self, attribute, None)
    if automorphisms is None:
        automorphisms = _ProjectiveAutomorphismGroup(self)
        setattr(self, attribute, automorphisms)
    return automorphisms


def _install_projective_scheme_extensions():
    morphism_classes = (
        _projective_morphism_module.SchemeMorphism_polynomial_projective_space,
        _projective_morphism_module.SchemeMorphism_polynomial_projective_space_field,
        _projective_morphism_module.SchemeMorphism_polynomial_projective_space_finite_field,
        ProductProjectiveSpaces_morphism_ring,
    )

    for morphism_class in morphism_classes:
        for obsolete_name in (
            'scheme_theoretic_image',
            'inverse_morphism',
            'inverse_image_subscheme',
            'pullback_subscheme',
            'graph',
        ):
            if obsolete_name in morphism_class.__dict__:
                delattr(morphism_class, obsolete_name)

        morphism_class.image = _scheme_theoretic_image_projective
        morphism_class.ambient_category = (
            _scheme_morphism_ambient_category
        )
        morphism_class.pullback = _scheme_morphism_pullback
        morphism_class.graph_morphism = _graph_morphism
        morphism_class.fixed_subscheme = _fixed_subscheme
        morphism_class.is_automorphism = _is_projective_automorphism
        morphism_class.__invert__ = _invert_projective_automorphism

    for space_class in (
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
    ):
        for obsolete_name in (
            'factor_dimensions',
            'factor_blocks',
            'saturate_ideal',
            'diagonal_subscheme',
        ):
            if obsolete_name in space_class.__dict__:
                delattr(space_class, obsolete_name)

        space_class.diagonal_morphism = _diagonal_morphism
        space_class.End = _endomorphism_set
        space_class.endomorphisms = _endomorphism_set
        space_class.Aut = _automorphism_group
        space_class.automorphisms = _automorphism_group

    return morphism_classes


_projective_morphism_classes = _install_projective_scheme_extensions()

print('Installed the Sage-native projective-scheme interface.')

NameError: name '_scheme_theoretic_image_projective' is not defined

## Regression computations through the Sage-native interface

We construct ordinary Sage spaces and morphisms and call only public geometric methods. Product-coordinate decomposition is tested through Sage's native `components()` method rather than through an added interface.

In [8]:
import os as _os
if not _os.environ.get('PROJECTIVE_SCHEME_FRAMEWORK_SKIP_REGRESSIONS'):
    exec(r'''P1_test_x = ProjectiveSpace(
    QQ,
    1,
    names=('tx0', 'tx1'),
)
P1_test_y = ProjectiveSpace(
    QQ,
    1,
    names=('ty0', 'ty1'),
)
X_test = P1_test_x * P1_test_y
tx0, tx1, ty0, ty1 = X_test.gens()

assert X_test.n_components() == 2
assert tuple(
    factor.dimension_relative()
    for factor in X_test.components()
) == (1, 1)
assert (X_test * X_test).n_components() == 4
assert X_test.End() == X_test.Hom(X_test)
assert not hasattr(X_test, 'factor_dimensions')
assert not hasattr(X_test, 'factor_blocks')
assert not hasattr(X_test, 'saturate_ideal')

Delta_test = X_test.diagonal_morphism()
Diagonal_test = Delta_test.codomain()
assert Diagonal_test.dimension() == 2
assert Delta_test.image() == Diagonal_test
assert not hasattr(X_test, 'diagonal_subscheme')
assert not hasattr(Delta_test, 'scheme_theoretic_image')

projection_test = X_test.hom(
    [tx0, tx1],
    P1_test_x,
)
point_test = P1_test_x.subscheme([P1_test_x.gens()[1]])
point_embedding_test = point_test.embedding_morphism()
fiber_square_test = projection_test.pullback(
    point_embedding_test
)
fiber_square_category_test = (
    projection_test.ambient_category().pullback(
        projection_test,
        point_embedding_test,
    )
)
fiber_test = fiber_square_test.left_projection().image()
assert fiber_square_test.commutes()
fiber_square_identity_test = (
    fiber_square_test.universal_morphism(
        fiber_square_test.left_projection(),
        fiber_square_test.right_projection(),
    )
)
assert _projective_morphisms_equal_on_domain(
    fiber_square_identity_test,
    fiber_square_test.apex().identity_morphism(),
)
assert fiber_square_category_test.apex() == (
    fiber_square_test.apex()
)
assert fiber_test.dimension() == 1
assert not hasattr(projection_test, 'pullback_subscheme')

tau_test = X_test.hom(
    [tx0, -tx1, ty0, -ty1],
    X_test,
)
Aut_X_test = X_test.Aut()
assert tau_test.parent() == X_test.End()
assert tau_test.is_automorphism()
assert tau_test in Aut_X_test
assert Aut_X_test(tau_test) is tau_test
assert Aut_X_test.one() == X_test.identity_morphism()
assert Aut_X_test.category().is_subcategory(Groups())
assert not hasattr(tau_test, 'inverse_morphism')

tau_inverse_test = tau_test.inverse()
assert tau_inverse_test == ~tau_test
assert tau_inverse_test == tau_test
assert tau_inverse_test * tau_test == X_test.identity_morphism()
assert tau_test * tau_inverse_test == X_test.identity_morphism()

graph_tau_test = tau_test.graph_morphism()
Graph_tau_test = graph_tau_test.codomain()
Fix_tau_test = tau_test.fixed_subscheme()
assert Graph_tau_test == graph_tau_test.image()
assert Graph_tau_test.dimension() == 2
assert not hasattr(tau_test, 'graph')
assert Fix_tau_test.dimension() == 0
assert len(Fix_tau_test.rational_points()) == 4

P1_test_a = ProjectiveSpace(
    QQ,
    1,
    names=('ta0', 'ta1'),
)
P1_test_b = ProjectiveSpace(
    QQ,
    1,
    names=('tb0', 'tb1'),
)
P1_test_c = ProjectiveSpace(
    QQ,
    1,
    names=('tc0', 'tc1'),
)
T_test = P1_test_a * P1_test_b * P1_test_c
ta0, ta1, tb0, tb1, tc0, tc1 = T_test.gens()
sigma_test = T_test.hom(
    [tb0, tb1, tc0, tc1, ta0, ta1],
    T_test,
)
Aut_T_test = T_test.Aut()
assert sigma_test.is_automorphism()
assert sigma_test in Aut_T_test
sigma_inverse_test = sigma_test.inverse()
assert sigma_inverse_test == ~sigma_test
assert sigma_inverse_test * sigma_test == T_test.identity_morphism()
assert sigma_test * sigma_inverse_test == T_test.identity_morphism()

Fix_sigma_test = sigma_test.fixed_subscheme()
assert Fix_sigma_test.dimension() == 1

framework_regression_summary = {
    'product_factor_dimensions_from_components': tuple(
        factor.dimension_relative()
        for factor in X_test.components()
    ),
    'diagonal_dimension': Diagonal_test.dimension(),
    'pullback_fiber_dimension': fiber_test.dimension(),
    'tau_in_automorphism_group': tau_test in X_test.Aut(),
    'tau_inverse_uses_native_inverse': tau_test.inverse() == tau_test,
    'endomorphism_graph_dimension': Graph_tau_test.dimension(),
    'involution_fixed_point_count': len(Fix_tau_test.rational_points()),
    'factor_permutation_in_automorphism_group': sigma_test in T_test.Aut(),
    'order_three_fixed_dimension': Fix_sigma_test.dimension(),
}

print('Sage-native projective framework regression tests passed:')
print('Aut(X) =', X_test.Aut())
print('Aut(X) category =', X_test.Aut().category())
print('tau parent =', tau_test.parent())
for key, value in framework_regression_summary.items():
    print(key, '=', value)
''', globals())
else:
    framework_regression_summary = {'skipped': True}
    print('Skipped projective morphism regression computations during import.')

Sage-native projective framework regression tests passed:
Aut(X) = Automorphisms of Product of projective spaces P^1 x P^1 over Rational Field
Aut(X) category = Category of facade groups
tau parent = Set of morphisms
  From: Product of projective spaces P^1 x P^1 over Rational Field
  To:   Product of projective spaces P^1 x P^1 over Rational Field
product_factor_dimensions_from_components = (1, 1)
diagonal_dimension = 2
pullback_fiber_dimension = 1
tau_in_automorphism_group = True
tau_inverse_uses_native_inverse = True
endomorphism_graph_dimension = 2
involution_fixed_point_count = 4
factor_permutation_in_automorphism_group = True
order_three_fixed_dimension = 1


## Picard groups and line bundles on products of projective spaces

For a product $X=\prod_i\mathbf P^{n_i}$, define `X.Pic()` as the free $\mathbf Z$-module generated by the pullbacks of the hyperplane classes. An element `X.O(d_1,...,d_r)` denotes the corresponding class of $\mathcal O_X(d_1,\ldots,d_r)$.

The next cell constructs this parent and its elements. It computes tensor products additively, duals, canonical classes, top intersection numbers, and the integral Picard lattice when $\dim X=2$. Cohomology itself is constructed later as a graded module; dimensions and Euler characteristics are derived from its graded pieces.

In [13]:
from sage.categories.modules import Modules
from sage.structure.element import AdditiveGroupElement
from sage.structure.richcmp import richcmp



class ProductProjectiveLineBundle(AdditiveGroupElement):
    def __init__(self, parent, multidegree):
        AdditiveGroupElement.__init__(self, parent)
        self._multidegree = tuple(ZZ(value) for value in multidegree)

    def _repr_(self):
        entries = ', '.join(str(value) for value in self._multidegree)
        return f'O({entries}) on {self.scheme()}'

    def _latex_(self):
        entries = ','.join(latex(value) for value in self._multidegree)
        return r'\mathcal O_{X}\left(' + entries + r'\right)'

    def __hash__(self):
        return hash((id(self.parent()), self._multidegree))

    def _richcmp_(self, other, operation):
        if not isinstance(other, ProductProjectiveLineBundle):
            return NotImplemented
        if self.parent() is not other.parent():
            return richcmp(id(self.parent()), id(other.parent()), operation)
        return richcmp(self._multidegree, other._multidegree, operation)

    def _add_(self, other):
        return self.parent()(
            tuple(
                left + right
                for left, right in zip(
                    self._multidegree,
                    other._multidegree,
                )
            )
        )

    def _neg_(self):
        return self.parent()(
            tuple(-value for value in self._multidegree)
        )

    def _rmul_(self, scalar):
        scalar = ZZ(scalar)
        return self.parent()(
            tuple(scalar * value for value in self._multidegree)
        )

    _lmul_ = _rmul_

    def __mul__(self, other):
        if (
            isinstance(other, ProductProjectiveLineBundle)
            and other.parent() is self.parent()
        ):
            if self.scheme().dimension_relative() != 2:
                raise TypeError(
                    'binary divisor intersection is numerical only on a surface; '
                    'use parent().intersection_number(...) in higher dimension'
                )
            return self.parent().intersection_number(self, other)
        return AdditiveGroupElement.__mul__(self, other)

    def __pow__(self, exponent, modulus=None):
        if modulus is not None:
            raise TypeError('modular powers are undefined for divisor classes')
        exponent = ZZ(exponent)
        dimension = ZZ(self.scheme().dimension_relative())
        if exponent != dimension:
            raise ValueError(
                'a numerical self-intersection requires one divisor factor '
                'for each dimension of the scheme'
            )
        return self.parent().intersection_number(
            *tuple(self for _ in range(dimension))
        )

    def scheme(self):
        return self.parent().scheme()

    def base_change(self, base_morphism):
        return self.scheme().base_change(base_morphism).O(
            self._multidegree
        )

    def multidegree(self):
        return self._multidegree

    def bidegree(self):
        if len(self._multidegree) != 2:
            raise ValueError('bidegree is defined only for a two-factor product')
        return self._multidegree

    def dual(self):
        return -self

    def tensor_power(self, exponent):
        return ZZ(exponent) * self

    def is_effective(self):
        return all(value >= 0 for value in self._multidegree)

    def is_globally_generated(self):
        return self.is_effective()

    def is_ample(self):
        return all(value > 0 for value in self._multidegree)

    def cohomology(self):
        return self.parent().cohomology(self)

    def H(self, degree):
        return self.cohomology().graded_piece(degree)

    def h(self, degree):
        return ZZ(self.H(degree).dimension())

    def euler_characteristic(self):
        return self.cohomology().euler_characteristic()

    def intersection(self, other):
        return self.parent().intersection_number(self, other)

    def top_self_intersection(self):
        dimension = ZZ(self.scheme().dimension_relative())
        return self.parent().intersection_number(
            *tuple(self for _ in range(dimension))
        )

    def lattice_vector(self):
        return self.parent().lattice()(self._multidegree)

    def global_sections(self):
        return self.H(0)

    def section_ring(self):
        return self.parent().section_ring(self)


class ProductProjectivePicardGroup(Parent):
    Element = ProductProjectiveLineBundle

    def __init__(self, scheme):
        self._scheme = scheme
        self._factors = _projective_factors(scheme)
        self._rank = len(self._factors)
        self._H0_cache = {}
        self._cohomology_cache = {}
        self._section_ring_cache = {}
        self._lattice_cache = None
        Parent.__init__(
            self,
            base=ZZ,
            category=Modules(ZZ).FiniteDimensional(),
        )

    def _repr_(self):
        return f'Picard group of {self._scheme}'

    def _element_constructor_(self, *degrees):
        if (
            len(degrees) == 1
            and isinstance(degrees[0], ProductProjectiveLineBundle)
        ):
            if degrees[0].parent() is self:
                return degrees[0]
            if degrees[0].scheme() != self._scheme:
                raise ValueError(
                    'a line bundle cannot be coerced between different schemes'
                )
            degrees = degrees[0].multidegree()
        elif len(degrees) == 1 and isinstance(degrees[0], (tuple, list)):
            degrees = tuple(degrees[0])

        if len(degrees) != self._rank:
            raise ValueError(
                f'expected {self._rank} degrees, received {len(degrees)}'
            )
        return self.element_class(self, tuple(ZZ(value) for value in degrees))

    def scheme(self):
        return self._scheme

    def rank(self):
        return ZZ(self._rank)

    dimension = rank

    def zero(self):
        return self((ZZ(0),) * self._rank)

    def gens(self):
        return tuple(
            self(
                tuple(
                    ZZ(1) if i == j else ZZ(0)
                    for i in range(self._rank)
                )
            )
            for j in range(self._rank)
        )

    basis = gens

    def _an_element_(self):
        return self.zero()

    def canonical_class(self):
        return self(
            tuple(
                -factor.dimension_relative() - 1
                for factor in self._factors
            )
        )

    def intersection_number(self, *classes):
        dimension = ZZ(self._scheme.dimension_relative())
        if len(classes) != dimension:
            raise ValueError(
                f'expected {dimension} divisor classes, received {len(classes)}'
            )
        classes = tuple(self(divisor_class) for divisor_class in classes)

        ChowPolynomialRing = PolynomialRing(
            ZZ,
            names=tuple(f'H{i}' for i in range(self._rank)),
        )
        hyperplanes = ChowPolynomialRing.gens()
        product_class = ChowPolynomialRing.one()
        for divisor_class in classes:
            product_class *= sum(
                coefficient * hyperplane
                for coefficient, hyperplane in zip(
                    divisor_class.multidegree(),
                    hyperplanes,
                )
            )

        top_monomial = prod(
            hyperplane**factor.dimension_relative()
            for hyperplane, factor in zip(
                hyperplanes,
                self._factors,
            )
        )
        return ZZ(product_class.monomial_coefficient(top_monomial))

    def intersection_form(self):
        if self._scheme.dimension_relative() != 2:
            raise ValueError('the Picard intersection form is bilinear only on a surface')
        basis = self.gens()
        return matrix(
            ZZ,
            [
                [self.intersection_number(left, right) for right in basis]
                for left in basis
            ],
        )

    def lattice(self):
        if self._lattice_cache is None:
            self._lattice_cache = IntegralLattice(self.intersection_form())
        return self._lattice_cache

    def global_sections(self, line_bundle):
        line_bundle = self(line_bundle)
        key = line_bundle.multidegree()
        if key not in self._H0_cache:
            self._H0_cache[key] = ProductProjectiveGlobalSections(line_bundle)
        return self._H0_cache[key]

    def cohomology(self, line_bundle):
        line_bundle = self(line_bundle)
        key = line_bundle.multidegree()
        if key not in self._cohomology_cache:
            self._cohomology_cache[key] = ProductProjectiveLineBundleCohomology(
                line_bundle
            )
        return self._cohomology_cache[key]

    def section_ring(self, line_bundle):
        line_bundle = self(line_bundle)
        key = line_bundle.multidegree()
        if key not in self._section_ring_cache:
            self._section_ring_cache[key] = ProductProjectiveSectionRing(line_bundle)
        return self._section_ring_cache[key]

print('Loaded Picard-group and line-bundle parents for projective products.')

Loaded Picard-group and line-bundle parents for projective products.


## Hyperplane line bundles and complete intersections in projective space

For a closed subscheme $X\subset\mathbf P^n$, the framework does not identify the full Picard group unless it is known. Instead, `X.embedded_picard_group()` is the image

$$
\operatorname{im}\!\left(\operatorname{Pic}(\mathbf P^n)\longrightarrow\operatorname{Pic}(X)\right)
=\mathbf Z[\mathcal O_X(1)].
$$

If the displayed homogeneous equations have cardinality equal to $\operatorname{codim}_{\mathbf P^n}X$, they certify an embedded complete intersection. The resulting certificate computes the canonical bundle by adjunction. For integral complete intersections over exact perfect fields, the Cohen--Macaulay property together with the codimension of the singular locus certifies normality by Serre's criterion. A normal surface is declared del Pezzo when its certified anticanonical bundle is ample.

In [ ]:
from sage.schemes.projective.projective_subscheme import (
    AlgebraicScheme_subscheme_projective,
)


class EmbeddedProjectiveLineBundle(AdditiveGroupElement):
    def __init__(self, parent, degree):
        AdditiveGroupElement.__init__(self, parent)
        self._degree = ZZ(degree)

    def _repr_(self):
        return f'O({self._degree}) on {self.scheme()}'

    def _latex_(self):
        return (
            r'\mathcal O_{'
            + str(latex(self.scheme()))
            + r'}\!\left('
            + str(latex(self._degree))
            + r'\right)'
        )

    def __hash__(self):
        return hash((id(self.parent()), self._degree))

    def _richcmp_(self, other, operation):
        if not isinstance(other, EmbeddedProjectiveLineBundle):
            return NotImplemented
        if self.parent() is not other.parent():
            return richcmp(
                id(self.parent()),
                id(other.parent()),
                operation,
            )
        return richcmp(
            self._degree,
            other._degree,
            operation,
        )

    def _add_(self, other):
        return self.parent()(
            self._degree + other._degree
        )

    def _neg_(self):
        return self.parent()(-self._degree)

    def _rmul_(self, scalar):
        return self.parent()(ZZ(scalar) * self._degree)

    _lmul_ = _rmul_

    def __mul__(self, other):
        if (
            isinstance(other, EmbeddedProjectiveLineBundle)
            and other.parent() is self.parent()
        ):
            return self.intersection(other)
        return AdditiveGroupElement.__mul__(self, other)

    def __pow__(self, exponent, modulus=None):
        if modulus is not None:
            raise TypeError(
                'modular powers are undefined for line bundles'
            )
        exponent = ZZ(exponent)
        dimension = ZZ(self.scheme().dimension())
        if exponent != dimension:
            raise ValueError(
                'a numerical self-intersection requires one divisor factor for each dimension'
            )
        return self.top_self_intersection()

    def scheme(self):
        return self.parent().scheme()

    def degree(self):
        return self._degree

    def dual(self):
        return -self

    def tensor_power(self, exponent):
        return ZZ(exponent) * self

    def is_effective(self):
        return self._degree >= 0

    def is_globally_generated(self):
        return self._degree >= 0

    def is_ample(self):
        return self._degree > 0

    def intersection(self, other):
        other = self.parent()(other)
        if self.scheme().dimension() != 2:
            raise TypeError(
                'binary divisor intersection is numerical only on a surface'
            )
        return ZZ(
            self._degree
            * other._degree
            * self.scheme().degree()
        )

    def top_self_intersection(self):
        dimension = ZZ(self.scheme().dimension())
        return ZZ(
            self._degree**dimension
            * self.scheme().degree()
        )


class EmbeddedProjectivePicardGroup(Parent):
    Element = EmbeddedProjectiveLineBundle

    def __init__(self, scheme):
        self._scheme = scheme
        Parent.__init__(
            self,
            base=ZZ,
            category=Modules(ZZ).FiniteDimensional(),
        )

    def _repr_(self):
        return (
            f'Image of Pic({self._scheme.ambient_space()}) '
            f'in Pic({self._scheme})'
        )

    def _latex_(self):
        return (
            r'\operatorname{im}\!\left('
            + r'\operatorname{Pic}\!\left('
            + str(latex(self._scheme.ambient_space()))
            + r'\right)\longrightarrow '
            + r'\operatorname{Pic}\!\left('
            + str(latex(self._scheme))
            + r'\right)\right)'
        )

    def _element_constructor_(self, degree=0):
        if isinstance(
            degree,
            EmbeddedProjectiveLineBundle,
        ):
            if degree.parent() is self:
                return degree
            if degree.scheme() != self._scheme:
                raise ValueError(
                    'a line bundle cannot be coerced between different schemes'
                )
            degree = degree.degree()
        return self.element_class(self, ZZ(degree))

    def scheme(self):
        return self._scheme

    def rank(self):
        return ZZ(1)

    dimension = rank

    def zero(self):
        return self(0)

    def gen(self, index=0):
        if ZZ(index) != 0:
            raise IndexError(
                'the embedded Picard image has one generator'
            )
        return self(1)

    def gens(self):
        return (self.gen(),)

    basis = gens

    def intersection_number(self, *line_bundles):
        dimension = ZZ(self._scheme.dimension())
        if len(line_bundles) != dimension:
            raise ValueError(
                f'expected {dimension} divisor classes, received {len(line_bundles)}'
            )
        line_bundles = tuple(
            self(line_bundle)
            for line_bundle in line_bundles
        )
        return ZZ(
            prod(
                line_bundle.degree()
                for line_bundle in line_bundles
            )
            * self._scheme.degree()
        )


class ProjectiveCompleteIntersectionCertificate(SageObject):
    def __init__(self, scheme):
        ambient = scheme.ambient_space()
        equations = tuple(
            scheme.defining_polynomials()
        )
        codimension = ZZ(
            ambient.dimension_relative()
            - scheme.dimension()
        )
        if len(equations) != codimension:
            raise NotImplementedError(
                'the displayed defining equations do not certify a complete intersection'
            )
        if not all(
            equation.is_homogeneous()
            for equation in equations
        ):
            raise ValueError(
                'the defining equations are not homogeneous'
            )

        self._scheme = scheme
        self._equations = equations
        self._degrees = tuple(
            ZZ(equation.total_degree())
            for equation in equations
        )
        self._codimension = codimension

    def _repr_(self):
        return (
            f'Complete-intersection certificate of multidegree '
            f'{self._degrees} for {self._scheme}'
        )

    def _latex_(self):
        return (
            r'\begin{aligned}'
            + str(latex(self._scheme))
            + r'&=V\!\left('
            + ','.join(
                str(latex(equation))
                for equation in self._equations
            )
            + r'\right)\subset '
            + str(latex(self._scheme.ambient_space()))
            + r'\\'
            + r'\operatorname{multideg}(X)&='
            + str(latex(self._degrees))
            + r'\\'
            + r'\operatorname{codim}(X)&='
            + str(latex(self._codimension))
            + r'\end{aligned}'
        )

    def scheme(self):
        return self._scheme

    def equations(self):
        return self._equations

    def degrees(self):
        return self._degrees

    def codimension(self):
        return self._codimension

    def expected_degree(self):
        return ZZ(prod(self._degrees))

    def degree_matches(self):
        return self._scheme.degree() == self.expected_degree()

    def canonical_degree(self):
        ambient_dimension = ZZ(
            self._scheme.ambient_space().dimension_relative()
        )
        return ZZ(
            sum(self._degrees)
            - ambient_dimension
            - 1
        )

    def canonical_bundle(self):
        return self._scheme.O(
            self.canonical_degree()
        )

    def anticanonical_bundle(self):
        return -self.canonical_bundle()


def _embedded_projective_picard_group(self):
    attribute = '_projective_framework_embedded_picard_group'
    group = getattr(self, attribute, None)
    if group is None:
        group = EmbeddedProjectivePicardGroup(self)
        setattr(self, attribute, group)
    return group


def _embedded_projective_O(self, degree=0):
    return self.embedded_picard_group()(degree)


def _projective_complete_intersection_certificate(self):
    attribute = (
        '_projective_framework_complete_intersection_certificate'
    )
    certificate = getattr(self, attribute, None)
    if certificate is None:
        certificate = ProjectiveCompleteIntersectionCertificate(
            self
        )
        setattr(self, attribute, certificate)
    return certificate


def _projective_is_complete_intersection(self):
    self.complete_intersection_certificate()
    return True


def _projective_complete_intersection_degrees(self):
    return self.complete_intersection_certificate().degrees()


def _projective_canonical_bundle(self):
    return (
        self.complete_intersection_certificate()
        .canonical_bundle()
    )


def _projective_anticanonical_bundle(self):
    return -self.canonical_bundle()


def _projective_is_gorenstein(self):
    self.complete_intersection_certificate()
    return True


def _projective_is_normal(self):
    base_field = self.base_ring()
    if not base_field.is_field():
        raise NotImplementedError(
            'the normality certificate requires a field base'
        )
    if not base_field.is_exact():
        raise NotImplementedError(
            'the normality certificate requires an exact base field'
        )
    if not base_field.is_perfect():
        raise NotImplementedError(
            'the normality certificate is currently asserted only over perfect fields'
        )

    self.complete_intersection_certificate()
    if not self.defining_ideal().is_prime():
        return False

    singular_locus = self.singular_locus()
    if singular_locus.defining_ideal().is_one():
        return True
    return (
        singular_locus.dimension()
        <= self.dimension() - 2
    )


def _projective_is_del_Pezzo(self):
    if self.dimension() != 2:
        return False
    return (
        self.is_normal()
        and self.is_gorenstein()
        and self.anticanonical_bundle().is_ample()
    )


def _projective_anticanonical_degree(self):
    if self.dimension() != 2:
        raise ValueError(
            'the del Pezzo degree is defined for surfaces'
        )
    return self.anticanonical_bundle().top_self_intersection()


AlgebraicScheme_subscheme_projective.embedded_picard_group = (
    _embedded_projective_picard_group
)
AlgebraicScheme_subscheme_projective.O = (
    _embedded_projective_O
)
AlgebraicScheme_subscheme_projective.complete_intersection_certificate = (
    _projective_complete_intersection_certificate
)
AlgebraicScheme_subscheme_projective.is_complete_intersection = (
    _projective_is_complete_intersection
)
AlgebraicScheme_subscheme_projective.complete_intersection_degrees = (
    _projective_complete_intersection_degrees
)
AlgebraicScheme_subscheme_projective.canonical_bundle = (
    _projective_canonical_bundle
)
AlgebraicScheme_subscheme_projective.anticanonical_bundle = (
    _projective_anticanonical_bundle
)
AlgebraicScheme_subscheme_projective.is_gorenstein = (
    _projective_is_gorenstein
)
AlgebraicScheme_subscheme_projective.is_normal = (
    _projective_is_normal
)
AlgebraicScheme_subscheme_projective.is_del_Pezzo = (
    _projective_is_del_Pezzo
)
AlgebraicScheme_subscheme_projective.anticanonical_degree = (
    _projective_anticanonical_degree
)
AlgebraicScheme_subscheme_projective.del_Pezzo_degree = (
    _projective_anticanonical_degree
)

print(
    'Installed embedded hyperplane Picard groups, '
    'complete-intersection adjunction, and del Pezzo predicates.'
)

## The Cox algebra and its polynomial model

For a product $X=\prod_i\mathbf P^{n_i}$, define the abstract $\operatorname{Pic}(X)$-graded algebra

$$
\operatorname{Cox}(X)
=
\bigoplus_{\mathbf d\in\mathbf N^r}
H^0\!\left(X,\mathcal O_X(\mathbf d)\right).
$$

Its elements are sections, not polynomials. The chosen homogeneous coordinates on the projective factors determine a cached graded-algebra isomorphism

$$
\Phi_X:\operatorname{Cox}(X)\xrightarrow{\sim}
k[x_{i,j}].
$$

For $L=\mathcal O_X(\mathbf d)$, the restriction of $\Phi_X$ to the degree-$L$ piece gives an isomorphism from $H^0(X,L)$ to the multihomogeneous degree-$\mathbf d$ piece of the polynomial model. Thus `s.to_polynomial()` and `H.from_polynomial(F)` are explicit applications of this chosen isomorphism and its inverse; neither identifies sections with polynomials.

In [14]:
from itertools import product as _cartesian_product
from sage.combinat.free_module import CombinatorialFreeModule
from sage.categories.algebras import Algebras
from sage.structure.sage_object import SageObject


def _multihomogeneous_exponents(line_bundle):
    scheme = line_bundle.scheme()
    coordinate_blocks = _coordinate_blocks(
        scheme,
        scheme.coordinate_ring().gens(),
    )
    if any(degree < 0 for degree in line_bundle.multidegree()):
        return tuple()

    exponent_choices = tuple(
        tuple(
            tuple(ZZ(value) for value in vector)
            for vector in IntegerVectors(degree, len(block))
        )
        for degree, block in zip(
            line_bundle.multidegree(),
            coordinate_blocks,
        )
    )
    return tuple(
        tuple(
            exponent
            for block_exponents in chosen_blocks
            for exponent in block_exponents
        )
        for chosen_blocks in _cartesian_product(*exponent_choices)
    )


class GradedAlgebraComponent(SageObject):
    def __init__(
        self,
        graded_algebra,
        degree,
        module,
        inclusion,
        retraction,
    ):
        self._graded_algebra = graded_algebra
        self._degree = degree
        self._module = module
        self._inclusion = inclusion
        self._retraction = retraction

    def _repr_(self):
        return (
            f'Degree-{self._degree} component of '
            f'{self._graded_algebra.algebra()}'
        )

    def graded_algebra(self):
        return self._graded_algebra

    def degree(self):
        return self._degree

    def module(self):
        return self._module

    def include(self, element):
        return self._inclusion(self._module(element))

    def retract(self, element):
        return self._retraction(
            self._graded_algebra.algebra()(element)
        )


class GradedAlgebraStructure(SageObject):
    def __init__(
        self,
        algebra,
        grading_group,
        normalize_degree,
        degree_on_basis,
        component_factory,
        name=None,
    ):
        self._algebra = algebra
        self._grading_group = grading_group
        self._normalize_degree = normalize_degree
        self._degree_on_basis = degree_on_basis
        self._component_factory = component_factory
        self._component_cache = {}
        self._name = name

    def _repr_(self):
        if self._name is not None:
            return self._name
        return f'{self._grading_group}-graded structure on {self._algebra}'

    def algebra(self):
        return self._algebra

    def grading_group(self):
        return self._grading_group

    def normalize_degree(self, degree):
        return self._normalize_degree(degree)

    def degree_on_basis(self, basis_index):
        return self.normalize_degree(
            self._degree_on_basis(basis_index)
        )

    def homogeneous_degree(self, element):
        element = self._algebra(element)
        degrees = {
            self.degree_on_basis(basis_index)
            for basis_index in element.monomial_coefficients()
        }
        if len(degrees) == 0:
            return self.normalize_degree(
                self._grading_group.zero()
            )
        if len(degrees) == 1:
            return degrees.pop()
        return None

    def component(self, degree):
        degree = self.normalize_degree(degree)
        if degree not in self._component_cache:
            component = self._component_factory(degree)
            if not isinstance(component, GradedAlgebraComponent):
                raise TypeError(
                    'a graded-component factory must return '
                    'a GradedAlgebraComponent'
                )
            self._component_cache[degree] = component
        return self._component_cache[degree]


class GradedAlgebraComponentMorphism(SageObject):
    def __init__(self, graded_morphism, degree):
        self._graded_morphism = graded_morphism
        self._degree = graded_morphism.domain_grading().normalize_degree(
            degree
        )
        self._target_degree = graded_morphism.degree_map()(
            self._degree
        )
        self._domain_component = graded_morphism.domain_grading().component(
            self._degree
        )
        self._codomain_component = graded_morphism.codomain_grading().component(
            self._target_degree
        )

    def _repr_(self):
        return (
            f'Degree-{self._degree} restriction of '
            f'{self._graded_morphism}'
        )

    def graded_morphism(self):
        return self._graded_morphism

    def degree(self):
        return self._degree

    def target_degree(self):
        return self._target_degree

    def domain_component(self):
        return self._domain_component

    def codomain_component(self):
        return self._codomain_component

    def domain(self):
        return self._domain_component.module()

    def codomain(self):
        return self._codomain_component.module()

    def __call__(self, element):
        ambient_element = self._domain_component.include(element)
        image = self._graded_morphism(ambient_element)
        return self._codomain_component.retract(image)

    def to_ambient(self, element):
        return self._codomain_component.include(self(element))

    def inverse(self):
        inverse = self._graded_morphism.inverse()
        return inverse.restrict_degree(self._target_degree)

    __invert__ = inverse

    def preimage(self, element):
        return self.inverse()(element)

    def from_ambient(self, element):
        component_element = self._codomain_component.retract(element)
        return self.preimage(component_element)


class GradedAlgebraMorphism(SageObject):
    def __init__(
        self,
        domain_grading,
        codomain_grading,
        function,
        degree_map,
        inverse_function=None,
        inverse_degree_map=None,
        name=None,
    ):
        self._domain_grading = domain_grading
        self._codomain_grading = codomain_grading
        self._function = function
        self._degree_map = degree_map
        self._inverse_function = inverse_function
        self._inverse_degree_map = inverse_degree_map
        self._inverse_cache = None
        self._restriction_cache = {}
        self._name = name

    def _repr_(self):
        if self._name is not None:
            return self._name
        return (
            f'Graded algebra morphism from {self.domain()} '
            f'to {self.codomain()}'
        )

    def domain_grading(self):
        return self._domain_grading

    def codomain_grading(self):
        return self._codomain_grading

    def domain(self):
        return self._domain_grading.algebra()

    def codomain(self):
        return self._codomain_grading.algebra()

    def degree_map(self):
        return self._degree_map

    def __call__(self, element):
        return self._function(self.domain()(element))

    def restrict_degree(self, degree):
        degree = self._domain_grading.normalize_degree(degree)
        if degree not in self._restriction_cache:
            self._restriction_cache[degree] = (
                GradedAlgebraComponentMorphism(self, degree)
            )
        return self._restriction_cache[degree]

    def inverse(self):
        if self._inverse_function is None:
            raise ValueError('this graded algebra morphism is not invertible')
        if self._inverse_cache is None:
            self._inverse_cache = GradedAlgebraMorphism(
                self._codomain_grading,
                self._domain_grading,
                self._inverse_function,
                self._inverse_degree_map,
                inverse_function=self._function,
                inverse_degree_map=self._degree_map,
                name=f'Inverse of {self}',
            )
            self._inverse_cache._inverse_cache = self
        return self._inverse_cache

    __invert__ = inverse

    def preimage(self, element):
        return self.inverse()(element)


class ProductProjectiveCoxRingElement(CombinatorialFreeModule.Element):
    def to_polynomial(self):
        return self.parent().polynomial_isomorphism()(self)


class ProductProjectiveSectionElement(CombinatorialFreeModule.Element):
    def to_polynomial(self):
        return self.parent().cox_isomorphism().to_ambient(self)


class ProductProjectiveCoxRing(CombinatorialFreeModule):
    Element = ProductProjectiveCoxRingElement

    def __init__(self, scheme):
        self._scheme = scheme
        self._polynomial_model = scheme.coordinate_ring()
        self._coordinate_blocks = _coordinate_blocks(
            scheme,
            self._polynomial_model.gens(),
        )
        self._basis_indices = cartesian_product(
            [NonNegativeIntegers()] * self._polynomial_model.ngens()
        )
        self._grading_cache = None
        self._polynomial_grading_cache = None
        self._polynomial_isomorphism_cache = None
        CombinatorialFreeModule.__init__(
            self,
            scheme.base_ring(),
            self._basis_indices,
            prefix='s',
            category=Algebras(
                scheme.base_ring()
            ).Commutative().WithBasis(),
            element_class=ProductProjectiveCoxRingElement,
        )

    def _repr_(self):
        return f'Cox ring of {self._scheme}'

    def scheme(self):
        return self._scheme

    def grading_group(self):
        return self._scheme.Pic()

    def basis_index_set(self):
        return self._basis_indices

    def one_basis(self):
        return self._basis_indices(
            (ZZ(0),) * self._polynomial_model.ngens()
        )

    def product_on_basis(self, left, right):
        return self.monomial(
            self._basis_indices(
                tuple(
                    left_exponent + right_exponent
                    for left_exponent, right_exponent
                    in zip(left, right)
                )
            )
        )

    def multidegree_on_basis(self, exponent):
        multidegree = []
        start = 0
        for block in self._coordinate_blocks:
            stop = start + len(block)
            multidegree.append(sum(exponent[start:stop]))
            start = stop
        return tuple(ZZ(value) for value in multidegree)

    def degree_on_basis(self, exponent):
        return self._scheme.O(
            self.multidegree_on_basis(exponent)
        )

    def polynomial_model(self):
        return self._polynomial_model

    def include_section(self, section):
        section_space = section.parent()
        if not isinstance(
            section_space,
            ProductProjectiveGlobalSections,
        ):
            raise TypeError(
                'the element must belong to a supported global-section space'
            )
        if section_space.scheme() != self._scheme:
            raise ValueError(
                'the section and Cox ring must belong to the same scheme'
            )
        coefficients = {
            self._basis_indices(
                section_space.basis_exponents()[index]
            ): coefficient
            for index, coefficient
            in section.monomial_coefficients().items()
        }
        return self._from_dict(coefficients, remove_zeros=True)

    def _section_from_cox_element(self, line_bundle, cox_element):
        section_space = line_bundle.H(0)
        exponent_to_index = {
            exponent: index
            for index, exponent
            in enumerate(section_space.basis_exponents())
        }
        coefficients = {}
        for exponent, coefficient in self(cox_element).monomial_coefficients().items():
            exponent_tuple = tuple(ZZ(value) for value in exponent)
            if exponent_tuple not in exponent_to_index:
                raise ValueError(
                    'the Cox element is not homogeneous of the requested degree'
                )
            coefficients[
                exponent_to_index[exponent_tuple]
            ] = coefficient
        return section_space._from_dict(
            coefficients,
            remove_zeros=True,
        )

    def grading(self):
        if self._grading_cache is None:
            def component_factory(line_bundle):
                section_space = line_bundle.H(0)
                return GradedAlgebraComponent(
                    self._grading_cache,
                    line_bundle,
                    section_space,
                    self.include_section,
                    lambda element: self._section_from_cox_element(
                        line_bundle,
                        element,
                    ),
                )

            self._grading_cache = GradedAlgebraStructure(
                self,
                self.grading_group(),
                self.grading_group(),
                self.degree_on_basis,
                component_factory,
                name=f'Picard grading on {self}',
            )
        return self._grading_cache

    def polynomial_grading(self):
        if self._polynomial_grading_cache is None:
            polynomial_ring = self._polynomial_model
            polynomial_generators = polynomial_ring.gens()

            def polynomial_degree_on_basis(exponent):
                return self.degree_on_basis(exponent)

            def component_factory(line_bundle):
                exponents = _multihomogeneous_exponents(line_bundle)
                polynomial_basis = tuple(
                    prod(
                        generator**exponent_value
                        for generator, exponent_value
                        in zip(polynomial_generators, exponent)
                    )
                    for exponent in exponents
                )
                module = polynomial_ring.submodule(
                    polynomial_basis
                )
                return GradedAlgebraComponent(
                    self._polynomial_grading_cache,
                    line_bundle,
                    module,
                    module.lift,
                    module.retract,
                )

            self._polynomial_grading_cache = GradedAlgebraStructure(
                polynomial_ring,
                self.grading_group(),
                self.grading_group(),
                polynomial_degree_on_basis,
                component_factory,
                name=(
                    f'Picard grading on polynomial algebra '
                    f'{polynomial_ring}'
                ),
            )
        return self._polynomial_grading_cache

    def homogeneous_degree(self, element):
        return self.grading().homogeneous_degree(element)

    def graded_piece(self, line_bundle):
        return self.grading().component(line_bundle).module()

    def polynomial_isomorphism(self):
        if self._polynomial_isomorphism_cache is None:
            polynomial_ring = self._polynomial_model
            polynomial_generators = polynomial_ring.gens()

            def to_polynomial(element):
                return sum(
                    coefficient * prod(
                        generator**exponent_value
                        for generator, exponent_value
                        in zip(polynomial_generators, exponent)
                    )
                    for exponent, coefficient
                    in self(element).monomial_coefficients().items()
                )

            def from_polynomial(polynomial):
                polynomial = polynomial_ring(polynomial)
                return self._from_dict(
                    {
                        self._basis_indices(
                            tuple(ZZ(value) for value in exponent)
                        ): coefficient
                        for exponent, coefficient
                        in polynomial.dict().items()
                    },
                    remove_zeros=True,
                )

            identity_degree_map = lambda degree: degree
            self._polynomial_isomorphism_cache = (
                GradedAlgebraMorphism(
                    self.grading(),
                    self.polynomial_grading(),
                    to_polynomial,
                    identity_degree_map,
                    inverse_function=from_polynomial,
                    inverse_degree_map=identity_degree_map,
                    name=(
                        f'Chosen graded algebra isomorphism '
                        f'from {self} to {polynomial_ring}'
                    ),
                )
            )
        return self._polynomial_isomorphism_cache

    def from_polynomial(self, polynomial):
        return self.polynomial_isomorphism().inverse()(polynomial)

    def gens(self):
        generators = []
        for index in range(self._polynomial_model.ngens()):
            exponent = [ZZ(0)] * self._polynomial_model.ngens()
            exponent[index] = ZZ(1)
            generators.append(
                self.monomial(
                    self._basis_indices(tuple(exponent))
                )
            )
        return tuple(generators)

    algebra_generators = gens


class ProductProjectiveGlobalSections(CombinatorialFreeModule):
    Element = ProductProjectiveSectionElement

    def __init__(self, line_bundle):
        self._line_bundle = line_bundle
        self._scheme = line_bundle.scheme()
        self._total_section_algebra = self._scheme.cox_ring()
        self._basis_exponents = _multihomogeneous_exponents(
            line_bundle
        )
        self._affine_space_cache = {}
        CombinatorialFreeModule.__init__(
            self,
            self._scheme.base_ring(),
            tuple(range(len(self._basis_exponents))),
            prefix='s',
            element_class=ProductProjectiveSectionElement,
        )

    def _repr_(self):
        return f'H^0({self._scheme}, {self._line_bundle})'

    def scheme(self):
        return self._scheme

    def line_bundle(self):
        return self._line_bundle

    def total_section_algebra(self):
        return self._total_section_algebra

    def basis_exponents(self):
        return self._basis_exponents

    def cox_isomorphism(self):
        return self._total_section_algebra.polynomial_isomorphism().restrict_degree(
            self._line_bundle
        )

    def from_polynomial(self, polynomial):
        return self.cox_isomorphism().from_ambient(polynomial)

    def base_change(self, base_morphism):
        return self._line_bundle.base_change(base_morphism).H(0)

    def affine_space(self, names=None, prefix='c'):
        if names is None:
            names = tuple(
                f'{prefix}_{index}'
                for index in range(self.dimension())
            )
        return VV(
            self.as_quasicoherent_module().dual(),
            names=names,
        )


print(
    'Loaded a general graded-algebra morphism framework, '
    'the abstract Cox algebra, and its polynomial model.'
)

Loaded a general graded-algebra morphism framework, the abstract Cox algebra, and its polynomial model.


## Graded line-bundle cohomology

For a fixed line bundle $L$, the direct sum

$$
H^*(X,L)=\bigoplus_i H^i(X,L)
$$

is a graded vector space, not an algebra: cup product has the coefficient-changing form

$$
H^i(X,L)\otimes H^j(X,M)\longrightarrow H^{i+j}(X,L\otimes M).
$$

The next cell defines `L.cohomology()` as an actual finite-dimensional graded module. Its method `graded_piece(i)` returns the vector space $H^i(X,L)$. Degree zero is the existing global-section space. Higher cohomology on projective factors is represented by standard Čech Laurent-monomial bases, and cohomology on products is assembled by the Künneth formula.

In [15]:
from sage.categories.graded_modules import GradedModules


def _line_bundle_cohomology_basis_data(line_bundle):
    scheme = line_bundle.scheme()
    base_ring = scheme.base_ring()
    cox_ring = scheme.coordinate_ring()
    laurent_ring = LaurentPolynomialRing(
        base_ring,
        names=tuple(str(variable) for variable in cox_ring.gens()),
    )
    laurent_blocks = _coordinate_blocks(
        scheme,
        laurent_ring.gens(),
    )

    total_degree = ZZ(0)
    factor_bases = []
    for factor, degree, block in zip(
        _projective_factors(scheme),
        line_bundle.multidegree(),
        laurent_blocks,
    ):
        dimension = ZZ(factor.dimension_relative())
        if degree >= 0:
            factor_degree = ZZ(0)
            exponent_vectors = IntegerVectors(degree, len(block))
            factor_basis = tuple(
                prod(
                    variable**exponent
                    for variable, exponent in zip(block, exponent_vector)
                )
                for exponent_vector in exponent_vectors
            )
        elif degree <= -dimension - 1:
            factor_degree = dimension
            residual_degree = -degree - dimension - 1
            exponent_vectors = IntegerVectors(residual_degree, len(block))
            factor_basis = tuple(
                prod(
                    variable**(-exponent - 1)
                    for variable, exponent in zip(block, exponent_vector)
                )
                for exponent_vector in exponent_vectors
            )
        else:
            return {
                'laurent_ring': laurent_ring,
                'degree': None,
                'basis': tuple(),
            }

        total_degree += factor_degree
        factor_bases.append(factor_basis)

    basis = tuple(
        prod(factor_monomials)
        for factor_monomials in _cartesian_product(*factor_bases)
    )
    return {
        'laurent_ring': laurent_ring,
        'degree': total_degree,
        'basis': basis,
    }


class ProductProjectiveCohomologyGroup(CombinatorialFreeModule):
    def __init__(self, line_bundle, degree, basis_representatives):
        self._line_bundle = line_bundle
        self._scheme = line_bundle.scheme()
        self._degree = ZZ(degree)
        self._basis_representatives = tuple(basis_representatives)
        CombinatorialFreeModule.__init__(
            self,
            self._scheme.base_ring(),
            tuple(range(len(self._basis_representatives))),
            prefix='',
        )

    def _repr_(self):
        return f'H^{self._degree}({self._scheme}, {self._line_bundle})'

    def _repr_term(self, index):
        return str(self._basis_representatives[index])

    def _latex_term(self, index):
        return latex(self._basis_representatives[index])

    def degree(self):
        return self._degree

    def scheme(self):
        return self._scheme

    def line_bundle(self):
        return self._line_bundle

    def basis_representatives(self):
        return self._basis_representatives

    cech_basis = basis_representatives


class ProductProjectiveLineBundleCohomology(CombinatorialFreeModule):
    def __init__(self, line_bundle):
        self._line_bundle = line_bundle
        self._scheme = line_bundle.scheme()
        self._maximum_degree = ZZ(self._scheme.dimension_relative())
        basis_data = _line_bundle_cohomology_basis_data(line_bundle)
        nonzero_degree = basis_data['degree']
        nonzero_basis = basis_data['basis']

        self._pieces = {}
        for degree in range(self._maximum_degree + 1):
            if degree == 0:
                piece = line_bundle.parent().global_sections(line_bundle)
            elif nonzero_degree == degree:
                piece = ProductProjectiveCohomologyGroup(
                    line_bundle,
                    degree,
                    nonzero_basis,
                )
            else:
                piece = ProductProjectiveCohomologyGroup(
                    line_bundle,
                    degree,
                    tuple(),
                )
            self._pieces[ZZ(degree)] = piece

        total_basis_keys = tuple(
            (degree, index)
            for degree, piece in self._pieces.items()
            for index in range(piece.dimension())
        )
        CombinatorialFreeModule.__init__(
            self,
            self._scheme.base_ring(),
            total_basis_keys,
            prefix='',
            category=GradedModules(
                self._scheme.base_ring()
            ).FiniteDimensional(),
        )

    def _repr_(self):
        return f'H^*({self._scheme}, {self._line_bundle})'

    def _repr_term(self, key):
        degree, index = key
        piece = self._pieces[ZZ(degree)]
        return f'H^{degree}[{piece._repr_term(index)}]'

    def degree_on_basis(self, key):
        return ZZ(key[0])

    def scheme(self):
        return self._scheme

    def line_bundle(self):
        return self._line_bundle

    def degrees(self):
        return tuple(ZZ(i) for i in range(self._maximum_degree + 1))

    def graded_piece(self, degree):
        degree = ZZ(degree)
        if degree in self._pieces:
            return self._pieces[degree]
        return ProductProjectiveCohomologyGroup(
            self._line_bundle,
            degree,
            tuple(),
        )

    __getitem__ = graded_piece

    def pieces(self):
        return tuple(self._pieces[degree] for degree in self.degrees())

    def items(self):
        return tuple(
            (degree, self._pieces[degree])
            for degree in self.degrees()
        )

    def __iter__(self):
        return iter(self.pieces())

    def dimensions(self):
        return tuple(
            ZZ(piece.dimension())
            for piece in self.pieces()
        )

    def euler_characteristic(self):
        return sum(
            (-1)**degree * self._pieces[degree].dimension()
            for degree in self.degrees()
        )

    def poincare_series(self):
        power_series_ring = PowerSeriesRing(ZZ, 't')
        t = power_series_ring.gen()
        return power_series_ring(sum(
            self._pieces[degree].dimension() * t**degree
            for degree in self.degrees()
        ))

    def lift(self, piece_element):
        degree = ZZ(piece_element.parent().degree())
        if piece_element.parent() is not self.graded_piece(degree):
            raise ValueError('the element does not belong to a graded piece of this cohomology module')
        return self._from_dict(
            {
                (degree, index): coefficient
                for index, coefficient in piece_element.monomial_coefficients().items()
            },
            remove_zeros=True,
        )

    def project(self, element, degree):
        element = self(element)
        degree = ZZ(degree)
        piece = self.graded_piece(degree)
        return piece._from_dict(
            {
                index: coefficient
                for (basis_degree, index), coefficient
                in element.monomial_coefficients().items()
                if ZZ(basis_degree) == degree
            },
            remove_zeros=True,
        )

print('Loaded graded line-bundle cohomology modules and their cohomology groups.')

Loaded graded line-bundle cohomology modules and their cohomology groups.


## Section rings as Veronese subalgebras

For a globally generated nonzero line bundle $L$, define the abstract graded algebra

$$
R(X,L)=\bigoplus_{n\geq0}H^0\!\left(X,L^{\otimes n}\right).
$$

This is the Veronese subalgebra of the abstract Cox ring consisting of degrees $n[L]$. Its elements remain sections. The chosen Cox polynomial isomorphism restricts to a polynomial model of this subalgebra, but that model is not used as the element parent.

In [16]:
from sage.categories.algebras import Algebras
from sage.structure.element import CommutativeAlgebraElement


class ProductProjectiveSectionRingElement(
    CommutativeAlgebraElement
):
    def __init__(self, parent, cox_element):
        CommutativeAlgebraElement.__init__(self, parent)
        self._cox_element = parent._coerce_and_validate(
            cox_element
        )

    def _repr_(self):
        return repr(self._cox_element)

    def _latex_(self):
        return latex(self._cox_element)

    def __hash__(self):
        return hash((id(self.parent()), self._cox_element))

    def _richcmp_(self, other, operation):
        if not isinstance(
            other,
            ProductProjectiveSectionRingElement,
        ):
            return NotImplemented
        if self.parent() is not other.parent():
            return richcmp(
                id(self.parent()),
                id(other.parent()),
                operation,
            )
        return richcmp(
            self._cox_element,
            other._cox_element,
            operation,
        )

    def _add_(self, other):
        return self.parent()(
            self._cox_element + other._cox_element
        )

    def _neg_(self):
        return self.parent()(-self._cox_element)

    def _mul_(self, other):
        return self.parent()(
            self._cox_element * other._cox_element
        )

    def _lmul_(self, scalar):
        return self.parent()(scalar * self._cox_element)

    _rmul_ = _lmul_

    def cox_element(self):
        return self._cox_element

    def to_polynomial(self):
        return self.parent().cox_ring().polynomial_isomorphism()(
            self._cox_element
        )

    def homogeneous_degree(self):
        return self.parent().homogeneous_degree(self)


class ProductProjectiveSectionRing(Parent):
    Element = ProductProjectiveSectionRingElement

    def __init__(self, line_bundle):
        if not line_bundle.is_globally_generated():
            raise NotImplementedError(
                'the current section-ring implementation requires '
                'a globally generated line bundle'
            )
        if all(
            value == 0
            for value in line_bundle.multidegree()
        ):
            raise NotImplementedError(
                'the section ring of the trivial bundle requires '
                'an explicit grading variable and is not yet implemented'
            )

        self._line_bundle = line_bundle
        self._scheme = line_bundle.scheme()
        self._cox_ring = self._scheme.cox_ring()
        self._multidegree = line_bundle.multidegree()
        Parent.__init__(
            self,
            base=self._scheme.base_ring(),
            category=Algebras(
                self._scheme.base_ring()
            ).Commutative(),
        )

    def _repr_(self):
        return f'Section ring R({self._scheme}, {self._line_bundle})'

    def _element_constructor_(self, value):
        if (
            isinstance(
                value,
                ProductProjectiveSectionRingElement,
            )
            and value.parent() is self
        ):
            return value
        if isinstance(
            value,
            ProductProjectiveSectionElement,
        ):
            value = self._cox_ring.include_section(value)
        return self.element_class(self, value)

    def _section_degree_of_exponent(self, exponent):
        actual_multidegree = (
            self._cox_ring.multidegree_on_basis(exponent)
        )
        candidate_degree = None
        for actual_degree, bundle_degree in zip(
            actual_multidegree,
            self._multidegree,
        ):
            if bundle_degree == 0:
                if actual_degree != 0:
                    return None
                continue
            if actual_degree % bundle_degree != 0:
                return None
            quotient = ZZ(actual_degree // bundle_degree)
            if candidate_degree is None:
                candidate_degree = quotient
            elif quotient != candidate_degree:
                return None

        if candidate_degree is None:
            candidate_degree = ZZ(0)
        if candidate_degree < 0:
            return None
        return candidate_degree

    def _coerce_and_validate(self, value):
        cox_element = self._cox_ring(value)
        for exponent in cox_element.monomial_coefficients():
            if self._section_degree_of_exponent(exponent) is None:
                raise ValueError(
                    f'{cox_element} does not belong to the '
                    f'Veronese subalgebra associated to '
                    f'{self._line_bundle}'
                )
        return cox_element

    def scheme(self):
        return self._scheme

    def line_bundle(self):
        return self._line_bundle

    def cox_ring(self):
        return self._cox_ring

    def inclusion(self, element):
        return self(element).cox_element()

    def from_polynomial(self, polynomial):
        return self(
            self._cox_ring.from_polynomial(polynomial)
        )

    def zero(self):
        return self(self._cox_ring.zero())

    def one(self):
        return self(self._cox_ring.one())

    def _an_element_(self):
        generators = self.gens()
        return self.one() if len(generators) == 0 else generators[0]

    def graded_piece(self, degree):
        degree = ZZ(degree)
        if degree < 0:
            raise ValueError('graded degree must be nonnegative')
        return (degree * self._line_bundle).H(0)

    def graded_basis(self, degree):
        return self.graded_piece(degree).basis()

    def gens(self):
        return tuple(
            self(section)
            for section in self._line_bundle.H(0).basis()
        )

    algebra_generators = gens

    def homogeneous_degree(self, element):
        element = self(element)
        degrees = {
            self._section_degree_of_exponent(exponent)
            for exponent
            in element.cox_element().monomial_coefficients()
        }
        if len(degrees) == 0:
            return ZZ(0)
        if len(degrees) == 1:
            return degrees.pop()
        return None

    def homogeneous_component(self, element, degree):
        element = self(element)
        degree = ZZ(degree)
        component = self._cox_ring._from_dict(
            {
                exponent: coefficient
                for exponent, coefficient
                in element.cox_element().monomial_coefficients().items()
                if self._section_degree_of_exponent(exponent) == degree
            },
            remove_zeros=True,
        )
        return self(component)


print('Loaded abstract Veronese section-ring parents.')

Loaded abstract Veronese section-ring parents.


## Installation on projective spaces

The next cell installs `Pic()`, `O(...)`, `canonical_bundle()`, `Picard_lattice()`, and `cox_ring()` on projective spaces and their products. The Picard computation is restricted to the ambient product itself; it does not infer the Picard group of an arbitrary closed subscheme.

In [17]:
def _picard_group(self):
    if self != self.ambient_space():
        raise NotImplementedError(
            'the Picard group of an arbitrary closed subscheme '
            'is not inferred from its ambient product'
        )
    attribute = '_projective_framework_picard_group'
    picard_group = getattr(self, attribute, None)
    if picard_group is None:
        picard_group = ProductProjectivePicardGroup(self)
        setattr(self, attribute, picard_group)
    return picard_group


def _structure_sheaf_twist(self, *degrees):
    return self.Pic()(*degrees)


def _canonical_bundle(self):
    return self.Pic().canonical_class()


def _picard_lattice(self):
    return self.Pic().lattice()


def _cox_ring(self):
    if self != self.ambient_space():
        raise NotImplementedError(
            'the Cox ring is currently implemented only for '
            'the ambient projective product'
        )
    attribute = '_projective_framework_abstract_cox_ring'
    cox_ring = getattr(self, attribute, None)
    if cox_ring is None:
        cox_ring = ProductProjectiveCoxRing(self)
        cox_ring._scheme = self
        cox_ring._polynomial_model = self.coordinate_ring()
        cox_ring._coordinate_blocks = _coordinate_blocks(
            self,
            self.coordinate_ring().gens(),
        )
        cox_ring._grading_cache = None
        cox_ring._polynomial_grading_cache = None
        cox_ring._polynomial_isomorphism_cache = None
        setattr(self, attribute, cox_ring)
    return cox_ring


def _cox_polynomial_model(self):
    return self.cox_ring().polynomial_model()


def _install_product_projective_picard_interface():
    for space_class in (
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
    ):
        space_class.Pic = _picard_group
        space_class.Picard_group = _picard_group
        space_class.picard_group = _picard_group
        space_class.O = _structure_sheaf_twist
        space_class.OO = _structure_sheaf_twist
        space_class.canonical_bundle = _canonical_bundle
        space_class.Picard_lattice = _picard_lattice
        space_class.picard_lattice = _picard_lattice
        space_class.cox_ring = _cox_ring
        space_class.cox_polynomial_model = (
            _cox_polynomial_model
        )


_install_product_projective_picard_interface()

print(
    'Installed Picard, line-bundle, abstract Cox-ring, '
    'graded-cohomology, and section-ring methods.'
)

Installed Picard, line-bundle, abstract Cox-ring, graded-cohomology, and section-ring methods.


## Complete linear systems and their associated morphisms

For a line bundle $L$ on $X$, the complete linear system is

$$
|L|=\mathbf P\!\left(H^0(X,L)^\vee\right).
$$

It is empty when $H^0(X,L)=0$. Its projective dimension is $h^0(X,L)-1$. The associated map is a rational map in general and is a morphism precisely when the complete linear system is basepoint free. On the supported projective products, global generation is certified from the multidegree before the morphism is constructed.

In [18]:
class LinearSystem(SageObject):
    def __init__(
        self,
        line_bundle,
        section_subspace,
        coordinate_names=None,
        projective_latex_name=None,
        morphism_latex_name=None,
        certified_basepoint_free=False,
    ):
        self._line_bundle = line_bundle
        self._ambient_section_space = line_bundle.H(0)
        self._section_subspace = section_subspace
        self._basis = tuple(section_subspace.basis())
        for section in self._basis:
            if section.parent() is not self._ambient_section_space:
                raise ValueError(
                    'every basis element of a linear system must lie in H^0(X,L)'
                )

        if len(self._basis) > 0:
            basis_matrix = matrix(
                self.scheme().base_ring(),
                [
                    tuple(section.to_vector())
                    for section in self._basis
                ],
            )
            if basis_matrix.rank() != len(self._basis):
                raise ValueError(
                    'the supplied linear-system basis is linearly dependent'
                )

        if coordinate_names is None:
            coordinate_names = tuple(
                f'ell_{index}'
                for index in range(len(self._basis))
            )
        else:
            coordinate_names = tuple(
                str(name)
                for name in coordinate_names
            )
        if len(coordinate_names) != len(self._basis):
            raise ValueError(
                'the number of projective coordinate names must equal the linear-system dimension plus one'
            )

        self._coordinate_names = coordinate_names
        self._projective_latex_name = projective_latex_name
        self._morphism_latex_name = morphism_latex_name
        self._certified_basepoint_free = bool(
            certified_basepoint_free
        )
        self._projective_space_cache = None
        self._morphism_cache = None
        self._base_locus_cache = None

    def _repr_(self):
        return (
            f'Linear system in {self._line_bundle} '
            f'with dimension {self.dimension()}'
        )

    def _latex_(self):
        if self.is_empty():
            return r'\varnothing'
        return (
            r'\mathbf P\!\left('
            + r'\left\langle '
            + ','.join(
                str(latex(section))
                for section in self._basis
            )
            + r'\right\rangle^{\vee}\right)'
        )

    def line_bundle(self):
        return self._line_bundle

    def scheme(self):
        return self._line_bundle.scheme()

    def ambient_section_space(self):
        return self._ambient_section_space

    def section_space(self):
        return self._section_subspace

    def basis(self):
        return self._basis

    def is_complete(self):
        return self._section_subspace is self._ambient_section_space

    def is_empty(self):
        return len(self._basis) == 0

    def dimension(self):
        if self.is_empty():
            return ZZ(-1)
        return ZZ(len(self._basis) - 1)

    projective_dimension = dimension

    def coordinate_names(self):
        return self._coordinate_names

    def projective_space(self):
        if self.is_empty():
            raise ValueError(
                'the linear system is empty'
            )
        if self._projective_space_cache is None:
            self._projective_space_cache = ProjectiveSpace(
                self.scheme().base_ring(),
                self.dimension(),
                names=self._coordinate_names,
            )
            if self._projective_latex_name is not None:
                self._projective_space_cache.set_latex_name(
                    self._projective_latex_name
                )
        return self._projective_space_cache

    parameter_space = projective_space

    def base_locus(self):
        if self._base_locus_cache is None:
            if self.is_empty():
                self._base_locus_cache = self.scheme()
            elif self._certified_basepoint_free:
                self._base_locus_cache = (
                    self.scheme().ambient_space().subscheme([1])
                )
            else:
                if not self.scheme().base_ring().is_exact():
                    raise NotImplementedError(
                        'the base-locus saturation backend requires an exact base field unless basepoint freeness is independently certified'
                    )
                ambient = self.scheme().ambient_space()
                coordinate_ring = ambient.coordinate_ring()
                equations = list(
                    _defining_equations(self.scheme())
                )
                equations.extend(
                    section.to_polynomial()
                    for section in self._basis
                )
                base_ideal = coordinate_ring.ideal(
                    equations
                )
                base_ideal = _saturate_projective_ideal(
                    self.scheme(),
                    base_ideal,
                    coordinate_ring.gens(),
                )
                self._base_locus_cache = ambient.subscheme(
                    base_ideal.gens()
                )
        return self._base_locus_cache

    def is_basepoint_free(self):
        return self.base_locus().defining_ideal().is_one()

    is_globally_generated = is_basepoint_free

    def morphism(self):
        if self.is_empty():
            raise ValueError(
                'an empty linear system defines no rational map'
            )
        if not self.is_basepoint_free():
            raise ValueError(
                'the linear system is not basepoint free; it defines only a rational map'
            )
        if self._morphism_cache is None:
            coordinates = tuple(
                section.to_polynomial()
                for section in self._basis
            )
            self._morphism_cache = self.scheme().hom(
                coordinates,
                self.projective_space(),
            )
            if self._morphism_latex_name is not None:
                self._morphism_cache.set_latex_name(
                    self._morphism_latex_name
                )
        return self._morphism_cache

    associated_morphism = morphism


class CompleteLinearSystem(LinearSystem):
    def __init__(self, line_bundle):
        super().__init__(
            line_bundle,
            line_bundle.H(0),
            projective_latex_name=(
                r'\lvert '
                + str(latex(line_bundle))
                + r'\rvert'
            ),
            morphism_latex_name=r'\varphi_{\lvert L\rvert}',
            certified_basepoint_free=(
                line_bundle.is_globally_generated()
            ),
        )

    def _repr_(self):
        return (
            f'Complete linear system of '
            f'{self._line_bundle}'
        )


def _line_bundle_complete_linear_system(self):
    attribute = '_projective_framework_complete_linear_system'
    linear_system = getattr(self, attribute, None)
    if linear_system is None:
        linear_system = CompleteLinearSystem(self)
        setattr(self, attribute, linear_system)
    return linear_system


def _line_bundle_linear_system(
    self,
    section_subspace,
    coordinate_names=None,
    projective_latex_name=None,
    morphism_latex_name=None,
):
    attribute = '_projective_framework_linear_system_cache'
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    coordinate_key = (
        None
        if coordinate_names is None
        else tuple(str(name) for name in coordinate_names)
    )
    key = (
        id(section_subspace),
        coordinate_key,
        projective_latex_name,
        morphism_latex_name,
    )
    if key not in cache:
        cache[key] = LinearSystem(
            self,
            section_subspace,
            coordinate_names=coordinate_names,
            projective_latex_name=projective_latex_name,
            morphism_latex_name=morphism_latex_name,
        )
    return cache[key]


ProductProjectiveLineBundle.complete_linear_system = (
    _line_bundle_complete_linear_system
)
ProductProjectiveLineBundle.linear_system = (
    _line_bundle_linear_system
)

print('Loaded complete and sub-linear systems and their associated morphisms.')

Loaded complete linear systems and associated morphisms.


## Universal divisor over the affine space of sections

For a linear system $V\subseteq H^0(X,L)$, the affine space $\mathbf A(V)$ parametrizes actual sections, while $\mathbf P(V)$ parametrizes their zero divisors modulo nonzero scalar. The method `system.affine_family()` constructs the categorical product

$$
X\times_S\mathbf A(V),
$$

pulls back $L$, and forms the universal section

$$
s_{\mathrm{univ}}=\sum_i a_i\,\operatorname{pr}_X^*s_i.
$$

Its zero scheme is the universal divisor over $\mathbf A(V)$. This construction uses only categorical products, pullback of line bundles and sections, and the chosen basis of $V$; it is independent of toric geometry.

In [59]:
from sage.rings.polynomial.multi_polynomial_ring_base import (
    MPolynomialRing_base,
)


def _flatten_nested_polynomial_ring(nested_ring):
    coefficient_ring = nested_ring.base_ring()
    if not isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    ):
        raise NotImplementedError(
            'the coefficient ring is not a multivariate polynomial ring'
        )
    ground_ring = coefficient_ring.base_ring()
    coefficient_names = tuple(
        str(name)
        for name in coefficient_ring.variable_names()
    )
    fiber_names = tuple(
        str(name)
        for name in nested_ring.variable_names()
    )
    flat_ring = PolynomialRing(
        ground_ring,
        names=coefficient_names + fiber_names,
    )
    coefficient_map = coefficient_ring.hom(
        tuple(
            flat_ring.gen(index)
            for index in range(len(coefficient_names))
        ),
        flat_ring,
    )
    to_flat = nested_ring.hom(
        tuple(
            flat_ring.gen(
                len(coefficient_names) + index
            )
            for index in range(len(fiber_names))
        ),
        flat_ring,
        base_map=coefficient_map,
    )
    from_flat = flat_ring.hom(
        tuple(
            nested_ring(coefficient_ring.gen(index))
            for index in range(len(coefficient_names))
        )
        + tuple(
            nested_ring.gen(index)
            for index in range(len(fiber_names))
        ),
        nested_ring,
    )
    return (
        flat_ring,
        to_flat,
        from_flat,
        len(coefficient_names),
    )


def _saturate_projective_ideal_over_polynomial_base(
    space,
    ideal,
):
    nested_ring = ideal.ring()
    nested_saturated_ideal = _saturate_projective_ideal(
        space,
        ideal,
        nested_ring.gens(),
    )
    (
        flat_ring,
        to_flat,
        from_flat,
        parameter_count,
    ) = _flatten_nested_polynomial_ring(nested_ring)
    saturated_flat_ideal = flat_ring.ideal(
        tuple(
            to_flat(generator)
            for generator in nested_saturated_ideal.gens()
        )
    )
    flat_fiber_coordinates = tuple(
        flat_ring.gen(parameter_count + index)
        for index in range(nested_ring.ngens())
    )
    return {
        'nested_ideal': nested_saturated_ideal,
        'flat_ideal': saturated_flat_ideal,
        'flat_ring': flat_ring,
        'to_flat': to_flat,
        'from_flat': from_flat,
        'parameter_count': ZZ(parameter_count),
        'flat_fiber_coordinates': flat_fiber_coordinates,
    }


import sage.rings.polynomial.multi_polynomial_ideal as _multi_polynomial_ideal_module


if (
    '_projective_framework_original_mpolynomial_ideal_saturation'
    not in globals()
):
    _projective_framework_original_mpolynomial_ideal_saturation = (
        _multi_polynomial_ideal_module
        .MPolynomialIdeal_singular_repr
        .saturation
    )


def _mpolynomial_ideal_saturation_with_polynomial_base(
    self,
    other,
):
    try:
        return (
            _projective_framework_original_mpolynomial_ideal_saturation(
                self,
                other,
            )
        )
    except TypeError as original_error:
        if self.ring() is not other.ring():
            raise
        nested_ring = self.ring()
        coefficient_ring = nested_ring.base_ring()
        if not all(
            hasattr(coefficient_ring, attribute)
            for attribute in (
                'variable_names',
                'gens',
                'base_ring',
                'hom',
            )
        ):
            raise original_error

        (
            flat_ring,
            to_flat,
            from_flat,
            parameter_count,
        ) = _flatten_nested_polynomial_ring(
            nested_ring
        )
        flat_self = flat_ring.ideal(
            tuple(
                to_flat(generator)
                for generator in self.gens()
            )
        )
        flat_other = flat_ring.ideal(
            tuple(
                to_flat(generator)
                for generator in other.gens()
            )
        )
        saturated_flat, exponent = (
            _projective_framework_original_mpolynomial_ideal_saturation(
                flat_self,
                flat_other,
            )
        )
        saturated_nested = nested_ring.ideal(
            tuple(
                from_flat(generator)
                for generator in saturated_flat.gens()
            )
        )
        saturated_nested._projective_framework_flattening_certificate = {
            'flat_ring': flat_ring,
            'to_flat': to_flat,
            'from_flat': from_flat,
            'parameter_count': ZZ(parameter_count),
        }
        return saturated_nested, exponent


_multi_polynomial_ideal_module.MPolynomialIdeal_singular_repr.saturation = (
    _mpolynomial_ideal_saturation_with_polynomial_base
)


import sage.rings.quotient_ring_element as _quotient_ring_element_module


if (
    '_projective_framework_original_quotient_element_reduce'
    not in globals()
):
    _projective_framework_original_quotient_element_reduce = (
        _quotient_ring_element_module
        .QuotientRingElement
        ._reduce_
    )


def _quotient_element_reduce_with_polynomial_base(self):
    parent = self.parent()
    cover_ring = parent.cover_ring()
    coefficient_ring = cover_ring.base_ring()
    polynomial_coefficient_presentation = isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    )
    ground_ring = (
        coefficient_ring.base_ring()
        if polynomial_coefficient_presentation
        else None
    )
    if not (
        polynomial_coefficient_presentation
        and hasattr(ground_ring, 'is_field')
        and ground_ring.is_field()
        and ground_ring.is_exact()
    ):
        return (
            _projective_framework_original_quotient_element_reduce(
                self
            )
        )

    cache = getattr(
        parent,
        '_projective_framework_flat_quotient_reduction_cache',
        None,
    )
    if cache is None:
        (
            flat_ring,
            to_flat,
            from_flat,
            parameter_count,
        ) = _flatten_nested_polynomial_ring(
            cover_ring
        )
        flat_ideal = flat_ring.ideal(
            tuple(
                to_flat(generator)
                for generator in parent.defining_ideal().gens()
            )
        )
        cache = {
            'flat_ring': flat_ring,
            'to_flat': to_flat,
            'from_flat': from_flat,
            'flat_ideal': flat_ideal,
            'parameter_count': ZZ(parameter_count),
        }
        parent._projective_framework_flat_quotient_reduction_cache = cache

    representative = getattr(
        self,
        '_QuotientRingElement__rep',
    )
    flat_representative = cache['to_flat'](
        representative
    )
    reduced_flat = cache['flat_ideal'].reduce(
        flat_representative
    )
    reduced_nested = cache['from_flat'](
        reduced_flat
    )
    setattr(
        self,
        '_QuotientRingElement__rep',
        reduced_nested,
    )
    return None


_quotient_ring_element_module.QuotientRingElement._reduce_ = (
    _quotient_element_reduce_with_polynomial_base
)


from sage.structure.richcmp import richcmp


if (
    '_projective_framework_original_quotient_element_richcmp'
    not in globals()
):
    _projective_framework_original_quotient_element_richcmp = (
        _quotient_ring_element_module
        .QuotientRingElement
        ._richcmp_
    )


def _quotient_element_richcmp_with_polynomial_base(
    self,
    other,
    operation,
):
    parent = self.parent()
    cover_ring = parent.cover_ring()
    coefficient_ring = cover_ring.base_ring()
    polynomial_coefficient_presentation = isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    )
    ground_ring = (
        coefficient_ring.base_ring()
        if polynomial_coefficient_presentation
        else None
    )
    if not (
        polynomial_coefficient_presentation
        and hasattr(ground_ring, 'is_field')
        and ground_ring.is_field()
        and ground_ring.is_exact()
    ):
        return (
            _projective_framework_original_quotient_element_richcmp(
                self,
                other,
                operation,
            )
        )

    self._reduce_()
    other._reduce_()
    left_representative = getattr(
        self,
        '_QuotientRingElement__rep',
    )
    right_representative = getattr(
        other,
        '_QuotientRingElement__rep',
    )
    return richcmp(
        left_representative,
        right_representative,
        operation,
    )


_quotient_ring_element_module.QuotientRingElement._richcmp_ = (
    _quotient_element_richcmp_with_polynomial_base
)


import sage.rings.quotient_ring as _quotient_ring_module


if (
    '_projective_framework_original_quotient_is_integral_domain'
    not in globals()
):
    _projective_framework_original_quotient_is_integral_domain = (
        _quotient_ring_module
        .QuotientRing_nc
        .is_integral_domain
    )


def _quotient_is_integral_domain_with_polynomial_base(
    self,
    proof=True,
):
    cover_ring = self.cover_ring()
    coefficient_ring = cover_ring.base_ring()
    polynomial_coefficient_presentation = isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    )
    ground_ring = (
        coefficient_ring.base_ring()
        if polynomial_coefficient_presentation
        else None
    )
    if not (
        polynomial_coefficient_presentation
        and hasattr(ground_ring, 'is_field')
        and ground_ring.is_field()
        and ground_ring.is_exact()
    ):
        return (
            _projective_framework_original_quotient_is_integral_domain(
                self,
                proof,
            )
        )

    cached = getattr(
        self,
        '_projective_framework_integral_domain_certificate',
        None,
    )
    if cached is None:
        reduction_cache = getattr(
            self,
            '_projective_framework_flat_quotient_reduction_cache',
            None,
        )
        if reduction_cache is None:
            (
                flat_ring,
                to_flat,
                from_flat,
                parameter_count,
            ) = _flatten_nested_polynomial_ring(
                cover_ring
            )
            flat_ideal = flat_ring.ideal(
                tuple(
                    to_flat(generator)
                    for generator in self.defining_ideal().gens()
                )
            )
            reduction_cache = {
                'flat_ring': flat_ring,
                'to_flat': to_flat,
                'from_flat': from_flat,
                'flat_ideal': flat_ideal,
                'parameter_count': ZZ(parameter_count),
            }
            self._projective_framework_flat_quotient_reduction_cache = (
                reduction_cache
            )
        cached = bool(
            reduction_cache['flat_ideal'].is_prime()
        )
        self._projective_framework_integral_domain_certificate = cached
    return cached


_quotient_ring_module.QuotientRing_nc.is_integral_domain = (
    _quotient_is_integral_domain_with_polynomial_base
)


def _quotient_krull_dimension_with_polynomial_base(self):
    cover_ring = self.cover_ring()
    coefficient_ring = cover_ring.base_ring()
    polynomial_coefficient_presentation = isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    )
    ground_ring = (
        coefficient_ring.base_ring()
        if polynomial_coefficient_presentation
        else None
    )
    if not (
        polynomial_coefficient_presentation
        and hasattr(ground_ring, 'is_field')
        and ground_ring.is_field()
        and ground_ring.is_exact()
    ):
        defining_ideal = self.defining_ideal()
        if hasattr(defining_ideal, 'dimension'):
            return ZZ(defining_ideal.dimension())
        raise NotImplementedError(
            'the quotient ring has no supported Krull-dimension backend'
        )

    cached = getattr(
        self,
        '_projective_framework_krull_dimension_certificate',
        None,
    )
    if cached is None:
        reduction_cache = getattr(
            self,
            '_projective_framework_flat_quotient_reduction_cache',
            None,
        )
        if reduction_cache is None:
            (
                flat_ring,
                to_flat,
                from_flat,
                parameter_count,
            ) = _flatten_nested_polynomial_ring(
                cover_ring
            )
            flat_ideal = flat_ring.ideal(
                tuple(
                    to_flat(generator)
                    for generator in self.defining_ideal().gens()
                )
            )
            reduction_cache = {
                'flat_ring': flat_ring,
                'to_flat': to_flat,
                'from_flat': from_flat,
                'flat_ideal': flat_ideal,
                'parameter_count': ZZ(parameter_count),
            }
            self._projective_framework_flat_quotient_reduction_cache = (
                reduction_cache
            )
        cached = ZZ(
            reduction_cache['flat_ideal'].dimension()
        )
        self._projective_framework_krull_dimension_certificate = cached
    return cached


_quotient_ring_module.QuotientRing_nc.krull_dimension = (
    _quotient_krull_dimension_with_polynomial_base
)


class AffineLinearSystemFamily(SageObject):
    def __init__(self, linear_system, parameter_names=None):
        self._linear_system = linear_system
        self._basis = tuple(linear_system.basis())
        if parameter_names is None:
            parameter_names = tuple(
                f'a_{index}'
                for index in range(len(self._basis))
            )
        else:
            parameter_names = tuple(
                str(name)
                for name in parameter_names
            )
        if len(parameter_names) != len(self._basis):
            raise ValueError(
                'the affine parameter space must have one coordinate for each basis section'
            )

        self._parameter_names = parameter_names
        self._parameter_space = AffineSpace(
            linear_system.scheme().base_ring(),
            len(self._basis),
            names=parameter_names,
        )
        self._product_diagram = linear_system.scheme().product(
            self._parameter_space
        )
        self._ambient_family = self._product_diagram.apex()
        self._projection_to_scheme = (
            self._product_diagram.left_projection()
        )
        self._projection_to_parameters = (
            self._product_diagram.right_projection()
        )
        self._universal_line_bundle = (
            linear_system.line_bundle().pullback(
                self._projection_to_scheme
            )
        )
        self._section_pullback = (
            linear_system.ambient_section_space().pullback(
                self._projection_to_scheme
            )
        )
        self._pulled_back_basis = tuple(
            self._section_pullback(section)
            for section in self._basis
        )
        target_sections = self._section_pullback.codomain()
        coefficient_ring = self._parameter_space.coordinate_ring()
        self._universal_section = target_sections.zero()
        for coefficient, section in zip(
            self._parameter_space.gens(),
            self._pulled_back_basis,
        ):
            self._universal_section += (
                coefficient_ring(coefficient) * section
            )

        self._basis_matrix = matrix(
            linear_system.scheme().base_ring(),
            [
                tuple(section.to_vector())
                for section in self._basis
            ],
        ) if len(self._basis) > 0 else matrix(
            linear_system.scheme().base_ring(),
            0,
            linear_system.ambient_section_space().dimension(),
        )
        self._universal_divisor_cache = None
        self._relative_singular_data_cache = None
        self._relative_singular_locus_cache = None
        self._discriminant_ideal_cache = None
        self._discriminant_subscheme_cache = None

    def _repr_(self):
        return (
            f'Universal divisor family of '
            f'{self._linear_system} over '
            f'{self._parameter_space}'
        )

    def _latex_(self):
        return (
            r'\mathcal D_{\mathrm{univ}}\subset '
            + str(latex(self.scheme()))
            + r'\times '
            + str(latex(self.parameter_space()))
        )

    def linear_system(self):
        return self._linear_system

    def scheme(self):
        return self._linear_system.scheme()

    def parameter_space(self):
        return self._parameter_space

    def coefficient_ring(self):
        return self._parameter_space.coordinate_ring()

    def parameter_coordinates(self):
        return tuple(self._parameter_space.gens())

    def coefficient_vector(self, section):
        if section.parent() is not (
            self._linear_system.ambient_section_space()
        ):
            raise ValueError(
                'the section must lie in the ambient H^0 space of the linear system'
            )
        section_vector = vector(
            self.scheme().base_ring(),
            tuple(section.to_vector()),
        )
        try:
            coefficients = (
                self._basis_matrix.transpose().solve_right(
                    section_vector
                )
            )
        except ValueError as error:
            raise ValueError(
                'the section does not lie in the chosen linear subsystem'
            ) from error
        if coefficients * self._basis_matrix != (
            section_vector
        ):
            raise ArithmeticError(
                'the computed subsystem coordinates do not reconstruct the section'
            )
        return coefficients

    def parameter_point(self, section):
        coefficients = self.coefficient_vector(section)
        return self._parameter_space(
            tuple(coefficients)
        )

    def specialize(self, parameter_point):
        if parameter_point.codomain() is not (
            self._parameter_space
        ):
            raise ValueError(
                'the specialization point does not lie in the affine section space'
            )
        coefficients = tuple(parameter_point)
        if len(coefficients) != len(self._basis):
            raise ArithmeticError(
                'the affine parameter point has the wrong coordinate count'
            )
        section = self._linear_system.ambient_section_space().zero()
        for coefficient, basis_section in zip(
            coefficients,
            self._basis,
        ):
            section += (
                self.scheme().base_ring()(coefficient)
                * basis_section
            )
        return section

    section_at = specialize

    def fiber_divisor(self, parameter_point):
        return self.specialize(
            parameter_point
        ).zero_subscheme()

    def product_diagram(self):
        return self._product_diagram

    def ambient_family(self):
        return self._ambient_family

    def projection_to_scheme(self):
        return self._projection_to_scheme

    def projection_to_parameters(self):
        return self._projection_to_parameters

    def universal_line_bundle(self):
        return self._universal_line_bundle

    def pulled_back_basis(self):
        return self._pulled_back_basis

    def universal_section(self):
        return self._universal_section

    def universal_divisor(self):
        if self._universal_divisor_cache is None:
            self._universal_divisor_cache = (
                self._universal_section.zero_subscheme()
            )
        return self._universal_divisor_cache

    def projection(self):
        return self.universal_divisor().base_morphism()

    def projectivized_parameter_space(self):
        return self._linear_system.projective_space()

    def _relative_singular_data(self):
        if self._relative_singular_data_cache is None:
            source_scheme = self.scheme()
            if source_scheme != source_scheme.ambient_space():
                raise NotImplementedError(
                    'the relative Jacobian backend currently requires the source scheme to equal its smooth supported projective ambient space'
                )
            if not self.coefficient_ring().is_exact():
                raise NotImplementedError(
                    'the relative Jacobian backend requires an exact coefficient ring'
                )
            ambient = self.ambient_family()
            coordinate_ring = ambient.coordinate_ring()
            equation = self.universal_section().to_polynomial()
            relative_jacobian_ideal = coordinate_ring.ideal(
                [equation]
                + [
                    equation.derivative(coordinate)
                    for coordinate in coordinate_ring.gens()
                ]
            )
            self._relative_singular_data_cache = (
                _saturate_projective_ideal_over_polynomial_base(
                    ambient,
                    relative_jacobian_ideal,
                )
            )
        return self._relative_singular_data_cache

    def relative_singular_ideal(self):
        return self._relative_singular_data()[
            'nested_ideal'
        ]

    def relative_singular_locus(self):
        if self._relative_singular_locus_cache is None:
            self._relative_singular_locus_cache = (
                self.ambient_family().subscheme(
                    self.relative_singular_ideal().gens()
                )
            )
        return self._relative_singular_locus_cache

    def discriminant_ideal(self):
        if self._discriminant_ideal_cache is None:
            data = self._relative_singular_data()
            flat_ideal = data['flat_ideal']
            flat_fiber_coordinates = data[
                'flat_fiber_coordinates'
            ]
            eliminated = flat_ideal.elimination_ideal(
                list(flat_fiber_coordinates)
            )
            parameter_count = data['parameter_count']
            parameter_ring = self.coefficient_ring()
            flat_ring = data['flat_ring']
            parameter_map = flat_ring.hom(
                tuple(
                    parameter_ring.gen(index)
                    for index in range(parameter_count)
                )
                + tuple(
                    parameter_ring.zero()
                    for _ in flat_fiber_coordinates
                ),
                parameter_ring,
            )
            self._discriminant_ideal_cache = (
                parameter_ring.ideal(
                    tuple(
                        parameter_map(generator)
                        for generator in eliminated.gens()
                    )
                )
            )
        return self._discriminant_ideal_cache

    def discriminant_subscheme(self):
        if self._discriminant_subscheme_cache is None:
            self._discriminant_subscheme_cache = (
                self.parameter_space().subscheme(
                    self.discriminant_ideal().gens()
                )
            )
        return self._discriminant_subscheme_cache

    def avoidance_polynomial(self, finite_embedding):
        restriction = (
            self._linear_system.ambient_section_space()
            .pullback(finite_embedding)
        )
        if not isinstance(
            restriction,
            SectionRestrictionMorphism,
        ):
            raise NotImplementedError(
                'avoidance polynomials are currently implemented for finite reduced rational subschemes'
            )
        matrix_rows = []
        for point_index in range(
            restriction.codomain().dimension()
        ):
            matrix_rows.append(
                tuple(
                    restriction(section).to_vector()[
                        point_index
                    ]
                    for section in self._basis
                )
            )
        coefficient_ring = self.coefficient_ring()
        parameters = self.parameter_coordinates()
        evaluation_forms = tuple(
            coefficient_ring(
                sum(
                    coefficient * parameter
                    for coefficient, parameter
                    in zip(row, parameters)
                )
            )
            for row in matrix_rows
        )
        return prod(evaluation_forms)


def _linear_system_affine_family(
    self,
    parameter_names=None,
):
    attribute = '_projective_framework_affine_family_cache'
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = (
        None
        if parameter_names is None
        else tuple(str(name) for name in parameter_names)
    )
    if key not in cache:
        cache[key] = AffineLinearSystemFamily(
            self,
            parameter_names=parameter_names,
        )
    return cache[key]


LinearSystem.affine_family = _linear_system_affine_family

print('Installed affine universal divisor families for linear systems.')

Installed affine universal divisor families for linear systems.


## Certified localization of polynomial-coefficient quotient domains

Sage's generic localization assumes factorization, numerator, and exact-division methods that multivariate quotient elements over polynomial coefficient rings do not provide. Once the flattened defining ideal certifies that such a quotient is an integral domain, `Q.localization(S)` now returns a mathematically exact localization parent whose elements are fractions

$$
\frac{r}{s_1^{e_1}\cdots s_m^{e_m}},
\qquad r\in Q,
$$

with equality decided by cross multiplication in $Q$. The public method remains `Q.localization(...)`; ordinary quotient rings continue to use Sage's original backend.

In [29]:
from sage.structure.parent import Parent
from sage.structure.element import CommutativeRingElement
from sage.categories.integral_domains import IntegralDomains


class CertifiedQuotientLocalizationElement(
    CommutativeRingElement
):
    def __init__(
        self,
        parent,
        numerator,
        denominator_exponents,
        numerator_unit_exponents=None,
    ):
        CommutativeRingElement.__init__(self, parent)
        self._numerator = parent.base_domain()(numerator)
        self._denominator_exponents = tuple(
            ZZ(exponent)
            for exponent in denominator_exponents
        )
        if any(
            exponent < 0
            for exponent in self._denominator_exponents
        ):
            raise ValueError(
                'denominator exponents must be nonnegative'
            )
        self._numerator_unit_exponents = (
            None
            if numerator_unit_exponents is None
            else tuple(
                ZZ(exponent)
                for exponent in numerator_unit_exponents
            )
        )

    def numerator(self):
        return self._numerator

    def denominator_exponents(self):
        return self._denominator_exponents

    def denominator(self):
        return self.parent().unit_monomial(
            self._denominator_exponents
        )

    def _repr_(self):
        denominator = self.denominator()
        if denominator == 1:
            return repr(self._numerator)
        return (
            f'({self._numerator})/'
            f'({denominator})'
        )

    def _latex_(self):
        denominator = self.denominator()
        if denominator == 1:
            return str(latex(self._numerator))
        return (
            r'\frac{'
            + str(latex(self._numerator))
            + r'}{'
            + str(latex(denominator))
            + r'}'
        )

    def _coerce_other(self, other):
        if (
            isinstance(
                other,
                CertifiedQuotientLocalizationElement,
            )
            and other.parent() is self.parent()
        ):
            return other
        return self.parent()(other)

    def __add__(self, other):
        other = self._coerce_other(other)
        common_exponents = tuple(
            max(left, right)
            for left, right in zip(
                self._denominator_exponents,
                other._denominator_exponents,
            )
        )
        left_multiplier = self.parent().unit_monomial(
            tuple(
                common - left
                for common, left in zip(
                    common_exponents,
                    self._denominator_exponents,
                )
            )
        )
        right_multiplier = self.parent().unit_monomial(
            tuple(
                common - right
                for common, right in zip(
                    common_exponents,
                    other._denominator_exponents,
                )
            )
        )
        return self.parent()._new_element(
            self._numerator * left_multiplier
            + other._numerator * right_multiplier,
            common_exponents,
            numerator_unit_exponents=None,
        )

    __radd__ = __add__

    def __neg__(self):
        return self.parent()._new_element(
            -self._numerator,
            self._denominator_exponents,
            numerator_unit_exponents=None,
        )

    def __sub__(self, other):
        return self.__add__(
            -self._coerce_other(other)
        )

    def __rsub__(self, other):
        return self.parent()(other).__sub__(self)

    def __mul__(self, other):
        other = self._coerce_other(other)
        numerator_units = None
        if (
            self._numerator_unit_exponents is not None
            and other._numerator_unit_exponents is not None
        ):
            numerator_units = tuple(
                left + right
                for left, right in zip(
                    self._numerator_unit_exponents,
                    other._numerator_unit_exponents,
                )
            )
        return self.parent()._new_element(
            self._numerator * other._numerator,
            tuple(
                left + right
                for left, right in zip(
                    self._denominator_exponents,
                    other._denominator_exponents,
                )
            ),
            numerator_unit_exponents=numerator_units,
        )

    __rmul__ = __mul__

    def __invert__(self):
        if self._numerator_unit_exponents is None:
            raise ZeroDivisionError(
                'the numerator is not certified to lie in the localized multiplicative set'
            )
        return self.parent()._new_element(
            self.denominator(),
            self._numerator_unit_exponents,
            numerator_unit_exponents=(
                self._denominator_exponents
            ),
        )

    def inverse(self):
        return ~self

    def __truediv__(self, other):
        return self * ~self._coerce_other(other)

    def __rtruediv__(self, other):
        return self.parent()(other) / self

    def __pow__(self, exponent):
        exponent = ZZ(exponent)
        if exponent < 0:
            return (~self)**(-exponent)
        result = self.parent().one()
        power = self
        while exponent:
            if exponent % 2:
                result = result * power
            power = power * power
            exponent //= 2
        return result

    def __eq__(self, other):
        try:
            other = self._coerce_other(other)
        except (TypeError, ValueError):
            return False
        return (
            self._numerator * other.denominator()
            == other._numerator * self.denominator()
        )

    def __ne__(self, other):
        return not self.__eq__(other)

    def is_zero(self):
        return self._numerator == 0

    def is_unit(self):
        return (
            self._numerator_unit_exponents
            is not None
        )


class CertifiedQuotientLocalization(Parent):
    Element = CertifiedQuotientLocalizationElement

    def __init__(
        self,
        base_domain,
        extra_units,
        names=None,
    ):
        if not base_domain.is_integral_domain():
            raise TypeError(
                'the localization base must be an integral domain'
            )
        self._base_domain = base_domain
        self._extra_units = tuple(
            base_domain(unit)
            for unit in (
                extra_units
                if isinstance(extra_units, (tuple, list))
                else (extra_units,)
            )
        )
        if len(self._extra_units) == 0:
            raise ValueError(
                'at least one localization generator is required'
            )
        self._names = names
        Parent.__init__(
            self,
            base=base_domain,
            category=IntegralDomains(),
        )

    def _repr_(self):
        return (
            f'{self._base_domain} localized at '
            f'{self._extra_units}'
        )

    def _latex_(self):
        return (
            str(latex(self._base_domain))
            + r'\left['
            + ','.join(
                str(latex(unit)) + r'^{-1}'
                for unit in self._extra_units
            )
            + r'\right]'
        )

    def base_domain(self):
        return self._base_domain

    def extra_units(self):
        return self._extra_units

    def ngens(self):
        return self._base_domain.ngens()

    def gens(self):
        return tuple(
            self(generator)
            for generator in self._base_domain.gens()
        )

    def gen(self, index=0):
        return self.gens()[ZZ(index)]

    def unit_monomial(self, exponents):
        return prod(
            unit**ZZ(exponent)
            for unit, exponent in zip(
                self._extra_units,
                exponents,
            )
        )

    def _unit_exponents_of(self, element):
        element = self._base_domain(element)
        if element == self._base_domain.one():
            return tuple(
                ZZ(0)
                for _ in self._extra_units
            )
        for index, unit in enumerate(
            self._extra_units
        ):
            if element == unit:
                return tuple(
                    ZZ(1)
                    if position == index
                    else ZZ(0)
                    for position in range(
                        len(self._extra_units)
                    )
                )
        return None

    def _new_element(
        self,
        numerator,
        denominator_exponents,
        numerator_unit_exponents=None,
    ):
        return self.element_class(
            self,
            numerator,
            denominator_exponents,
            numerator_unit_exponents=(
                numerator_unit_exponents
            ),
        )

    def _element_constructor_(self, value=0):
        if (
            isinstance(
                value,
                CertifiedQuotientLocalizationElement,
            )
            and value.parent() is self
        ):
            return value
        base_value = self._base_domain(value)
        return self._new_element(
            base_value,
            tuple(
                ZZ(0)
                for _ in self._extra_units
            ),
            numerator_unit_exponents=(
                self._unit_exponents_of(base_value)
            ),
        )

    def _coerce_map_from_(self, source):
        return (
            source is self._base_domain
            or self._base_domain.has_coerce_map_from(
                source
            )
        )

    def invert(self, unit):
        return ~self(unit)

    def is_integral_domain(self, proof=True):
        return True


def _quotient_localization_with_polynomial_base(
    self,
    additional_units,
    names=None,
    normalize=True,
    category=None,
    warning=True,
):
    cover_ring = self.cover_ring()
    coefficient_ring = cover_ring.base_ring()
    polynomial_coefficient_presentation = isinstance(
        coefficient_ring,
        MPolynomialRing_base,
    )
    ground_ring = (
        coefficient_ring.base_ring()
        if polynomial_coefficient_presentation
        else None
    )
    if (
        polynomial_coefficient_presentation
        and hasattr(ground_ring, 'is_field')
        and ground_ring.is_field()
        and ground_ring.is_exact()
    ):
        return CertifiedQuotientLocalization(
            self,
            additional_units,
            names=names,
        )
    from sage.rings.localization import Localization
    from sage.categories.integral_domains import IntegralDomains

    if not self.is_integral_domain():
        raise TypeError(
            'self must be an integral domain'
        )
    if self not in IntegralDomains():
        self._refine_category_(IntegralDomains())
    return Localization(
        self,
        additional_units,
        names=names,
        normalize=normalize,
        category=category,
        warning=warning,
    )


_quotient_ring_module.QuotientRing_nc.localization = (
    _quotient_localization_with_polynomial_base
)

print('Installed certified localization for polynomial-coefficient quotient domains.')

Installed certified localization for polynomial-coefficient quotient domains.


## Cyclic-cover data and $n$th-root validation

A cyclic cover of degree $n$ is determined by a line bundle $M$ and a section

$$
s\in H^0(X,M^{\otimes n}).
$$

The associated quasi-coherent algebra is

$$
\mathcal A=\bigoplus_{i=0}^{n-1}M^{-i},
$$

with multiplication across degree $n$ determined by $s$. The next cell implements and validates this datum, including the characteristic and $n$th-root conditions. It does not yet claim to construct the global finite morphism: that requires relative $\operatorname{Spec}_X$ for nonaffine bases and gluing, which remain separate prerequisites.

In [19]:
class CyclicCoverAlgebraDatum(SageObject):
    def __init__(self, root_line_bundle, branch_section, degree):
        self._root_line_bundle = root_line_bundle
        self._branch_section = branch_section
        self._degree = ZZ(degree)
        self._pieces = tuple(
            (-index) * root_line_bundle
            for index in range(self._degree)
        )

    def _repr_(self):
        return (
            f'Cyclic-cover algebra datum of degree {self._degree} '
            f'for {self._root_line_bundle}'
        )

    def degree(self):
        return self._degree

    def root_line_bundle(self):
        return self._root_line_bundle

    def branch_section(self):
        return self._branch_section

    def pieces(self):
        return self._pieces

    def product_degree(self, left_degree, right_degree):
        left_degree = ZZ(left_degree)
        right_degree = ZZ(right_degree)
        if not (
            0 <= left_degree < self._degree
            and 0 <= right_degree < self._degree
        ):
            raise ValueError('cyclic-cover degrees must lie between 0 and n-1')
        total = left_degree + right_degree
        return (
            total % self._degree,
            total >= self._degree,
        )


class CyclicCoverDatum(SageObject):
    def __init__(self, root_line_bundle, branch_section, degree):
        degree = ZZ(degree)
        if degree < 2:
            raise ValueError('a cyclic cover must have degree at least two')
        if not isinstance(
            branch_section.parent(),
            ProductProjectiveGlobalSections,
        ):
            raise TypeError('the branch datum must be a global section')
        if branch_section.parent().scheme() != root_line_bundle.scheme():
            raise ValueError('the root line bundle and branch section must live on the same scheme')
        branch_line_bundle = branch_section.parent().line_bundle()
        expected_line_bundle = degree * root_line_bundle
        if branch_line_bundle != expected_line_bundle:
            raise ValueError(
                f'the branch section lies in {branch_line_bundle}, '
                f'not in the required nth power {expected_line_bundle}'
            )
        characteristic = root_line_bundle.base_ring().characteristic()
        if characteristic != 0 and degree % characteristic == 0:
            raise NotImplementedError(
                'the current cyclic-cover datum excludes characteristic dividing the cover degree'
            )

        self._root_line_bundle = root_line_bundle
        self._branch_section = branch_section
        self._degree = degree
        self._algebra_datum = CyclicCoverAlgebraDatum(
            root_line_bundle,
            branch_section,
            degree,
        )

    def _repr_(self):
        return (
            f'Cyclic cover datum of degree {self._degree} '
            f'branched along {self._branch_section}'
        )

    def base_scheme(self):
        return self._root_line_bundle.scheme()

    def degree(self):
        return self._degree

    def root_line_bundle(self):
        return self._root_line_bundle

    def branch_section(self):
        return self._branch_section

    def branch_line_bundle(self):
        return self._branch_section.parent().line_bundle()

    def cover_algebra_datum(self):
        return self._algebra_datum

    def branch_subscheme(self):
        return self._branch_section.zero_subscheme()


def _section_zero_subscheme(self):
    section_space = self.parent()
    scheme = section_space.scheme()
    ambient = scheme.ambient_space()
    equations = list(_defining_equations(scheme))
    equations.append(self.to_polynomial())
    return ambient.subscheme(equations)


def _line_bundle_cyclic_cover_datum(self, branch_section, degree):
    return CyclicCoverDatum(self, branch_section, degree)


ProductProjectiveSectionElement.zero_subscheme = (
    _section_zero_subscheme
)
ProductProjectiveLineBundle.cyclic_cover_datum = (
    _line_bundle_cyclic_cover_datum
)

print('Loaded cyclic-cover data and nth-root validation.')

Loaded cyclic-cover data and nth-root validation.


## Cyclic-cover families over affine section spaces

Let $V\subseteq H^0(X,M^{\otimes n})$ and let $\mathcal D_{\mathrm{univ}}$ be the universal divisor over $\mathbf A(V)$. The object `system.cyclic_cover_family(M,n)` constructs the universal cyclic-cover algebra

$$
\bigoplus_{i=0}^{n-1}M^{-i}
$$

and its native monic quotient rings on the standard affine cover of $X\times\mathbf A(V)$. Pairwise overlaps are localizations, the root coordinate transforms by the transition function of $M^{-1}$, and the cocycle is verified on every triple intersection.

The resulting `CoveredScheme` is a genuine Sage scheme parent. Its global covering morphism to $X\times\mathbf A(V)$ and its family morphism to $\mathbf A(V)$ are `CoveredSchemeMorphism` objects whose chart maps are checked on every overlap. No toric hypothesis is used in this construction.

In [47]:
class AffineCyclicCoverChart(SageObject):
    def __init__(self, family, chart_morphism):
        self._family = family
        self._chart_morphism = chart_morphism
        self._label = chart_morphism.cover_label()

        chart_scheme = chart_morphism.domain()
        parameter_space = family.parameter_space()
        self._base_product = (
            _mixed_affine_base_change_product(
                chart_scheme,
                parameter_space,
                chart_scheme.base_scheme(),
                affine_on_right=True,
            )
        )
        self._base_chart_family = (
            self._base_product.apex()
        )
        self._projection_to_chart = (
            self._base_product.left_projection()
        )
        self._projection_to_parameters = (
            self._base_product.right_projection()
        )
        chart_to_source = (
            chart_morphism * self._projection_to_chart
        )
        self._chart_to_source = chart_to_source

        universal_ambient_ring = (
            family.divisor_family()
            .ambient_family()
            .coordinate_ring()
        )
        chart_family_ring = (
            self._base_chart_family.coordinate_ring()
        )
        parameter_ring = family.parameter_space().coordinate_ring()
        parameter_map = chart_family_ring.coerce_map_from(
            parameter_ring
        )
        if parameter_map is None:
            raise ValueError(
                'the chart-family coordinate ring does not contain the parameter ring canonically'
            )
        chart_pullback = universal_ambient_ring.hom(
            tuple(chart_to_source.defining_polynomials()),
            chart_family_ring,
            base_map=parameter_map,
        )
        self._branch_function = chart_pullback(
            family.universal_section().to_polynomial()
        )

        self._fiber_coordinates = tuple(
            self._base_chart_family.gens()[
                :chart_scheme.ambient_space().ngens()
            ]
        )
        if isinstance(self._label, tuple):
            label_suffix = '_'.join(
                str(index)
                for index in self._label
            )
        else:
            label_suffix = str(self._label)
        self._label_suffix = label_suffix
        self._cover_polynomial_ring = PolynomialRing(
            chart_family_ring,
            names=(f'z_cover_{label_suffix}',),
        )
        self._cover_polynomial_coordinate = (
            self._cover_polynomial_ring.gen(0)
        )
        self._cover_equation = (
            self._cover_polynomial_coordinate**family.degree()
            - self._cover_polynomial_ring(
                self._branch_function
            )
        )
        self._cover_coordinate_ring = (
            self._cover_polynomial_ring.quotient(
                self._cover_equation,
                names=(f'zbar_cover_{label_suffix}',),
            )
        )
        self._cover_coordinate = (
            self._cover_coordinate_ring.gen(0)
        )
        self._cover_scheme = Spec(
            self._cover_coordinate_ring
        )
        base_inclusion = chart_family_ring.hom(
            self._cover_coordinate_ring
        )
        self._cover_projection = self._cover_scheme.hom(
            base_inclusion,
            self._base_chart_family,
        )
        self._parameter_ring_map = (
            self._cover_coordinate_ring.coerce_map_from(
                parameter_ring
            )
        )
        if self._parameter_ring_map is None:
            raise ValueError(
                'the cyclic-cover chart ring does not contain the parameter ring canonically'
            )
        self._parameter_projection = self._cover_scheme.hom(
            self._parameter_ring_map,
            parameter_space,
        )
        self._base_chart_inclusion = (
            family.divisor_family()
            .product_diagram()
            .universal_morphism(
                self._chart_to_source,
                self._projection_to_parameters,
            )
        )
        ambient_family = (
            family.divisor_family().ambient_family()
        )
        ambient_family_ring = ambient_family.coordinate_ring()
        ambient_family_ring_map = ambient_family_ring.hom(
            tuple(
                base_inclusion(polynomial)
                for polynomial
                in self._chart_to_source.defining_polynomials()
            ),
            self._cover_coordinate_ring,
            base_map=self._parameter_ring_map,
        )
        self._cover_to_ambient_family = (
            self._cover_scheme.hom(
                ambient_family_ring_map,
                ambient_family,
            )
        )
        self._branch_subscheme_cache = None
        self._ramification_subscheme_cache = None
        self._relative_branch_singular_locus_cache = None
        self._relative_cover_singular_locus_cache = None

    def _repr_(self):
        return (
            f'Affine cyclic-cover family chart '
            f'{self._label} of {self._family}'
        )

    def _latex_(self):
        return (
            r'V\!\left('
            + str(latex(self._cover_equation))
            + r'\right)\longrightarrow '
            + str(latex(self._base_chart_family))
        )

    def family(self):
        return self._family

    def label(self):
        return self._label

    def chart_morphism(self):
        return self._chart_morphism

    def chart_to_source(self):
        return self._chart_to_source

    def base_product(self):
        return self._base_product

    def base_chart_family(self):
        return self._base_chart_family

    def branch_function(self):
        return self._branch_function

    def fiber_coordinates(self):
        return self._fiber_coordinates

    def cover_polynomial_ring(self):
        return self._cover_polynomial_ring

    def cover_coordinate_ring(self):
        return self._cover_coordinate_ring

    def cover_coordinate(self):
        return self._cover_coordinate

    def cover_equation(self):
        return self._cover_equation

    def cover_scheme(self):
        return self._cover_scheme

    def cover_projection(self):
        return self._cover_projection

    def parameter_ring_map(self):
        return self._parameter_ring_map

    def parameter_projection(self):
        return self._parameter_projection

    def base_chart_inclusion(self):
        return self._base_chart_inclusion

    def cover_to_ambient_family(self):
        return self._cover_to_ambient_family

    def branch_subscheme(self):
        if self._branch_subscheme_cache is None:
            self._branch_subscheme_cache = (
                self._base_chart_family.subscheme([
                    self._branch_function
                ])
            )
        return self._branch_subscheme_cache

    def ramification_subscheme(self):
        if self._ramification_subscheme_cache is None:
            base_ring = self._base_chart_family.coordinate_ring()
            ramification_ring = base_ring.quotient(
                base_ring.ideal([
                    self._branch_function
                ])
            )
            self._ramification_subscheme_cache = Spec(
                ramification_ring
            )
        return self._ramification_subscheme_cache

    def relative_branch_singular_locus(self):
        if self._relative_branch_singular_locus_cache is None:
            ring = self._base_chart_family.coordinate_ring()
            ideal = ring.ideal(
                [self._branch_function]
                + [
                    self._branch_function.derivative(coordinate)
                    for coordinate in self._fiber_coordinates
                ]
            )
            self._relative_branch_singular_locus_cache = (
                self._base_chart_family.subscheme(
                    ideal.gens()
                )
            )
        return self._relative_branch_singular_locus_cache

    def relative_cover_singular_locus(self):
        if self._relative_cover_singular_locus_cache is None:
            base_ring = self._base_chart_family.coordinate_ring()
            branch_singular_ideal = base_ring.ideal(
                [self._branch_function]
                + [
                    self._branch_function.derivative(
                        coordinate
                    )
                    for coordinate in self._fiber_coordinates
                ]
            )
            singular_base_ring = base_ring.quotient(
                branch_singular_ideal
            )
            if self.family().degree() == 2:
                singular_ring = singular_base_ring
            else:
                singular_polynomial_ring = PolynomialRing(
                    singular_base_ring,
                    names=(
                        f'z_singular_{self._label_suffix}',
                    ),
                )
                singular_coordinate = (
                    singular_polynomial_ring.gen(0)
                )
                singular_ring = (
                    singular_polynomial_ring.quotient(
                        singular_coordinate**(
                            self.family().degree() - 1
                        ),
                        names=(
                            f'zbar_singular_{self._label_suffix}',
                        ),
                    )
                )
            self._relative_cover_singular_locus_cache = Spec(
                singular_ring
            )
        return self._relative_cover_singular_locus_cache


class AffineCyclicCoverFamily(SageObject):
    def __init__(
        self,
        linear_system,
        root_line_bundle,
        degree,
        parameter_names=None,
    ):
        degree = ZZ(degree)
        if degree < 2:
            raise ValueError(
                'a cyclic cover family must have degree at least two'
            )
        if root_line_bundle.scheme() != linear_system.scheme():
            raise ValueError(
                'the root line bundle and linear system must lie on the same scheme'
            )
        if degree * root_line_bundle != linear_system.line_bundle():
            raise ValueError(
                'the linear-system line bundle is not the required nth tensor power of the root line bundle'
            )

        self._linear_system = linear_system
        self._root_line_bundle = root_line_bundle
        self._degree = degree
        self._divisor_family = linear_system.affine_family(
            parameter_names=parameter_names
        )
        self._root_line_bundle_family = (
            root_line_bundle.pullback(
                self._divisor_family.projection_to_scheme()
            )
        )
        self._cyclic_cover_datum = CyclicCoverDatum(
            self._root_line_bundle_family,
            self._divisor_family.universal_section(),
            degree,
        )
        self._chart_cache = {}
        self._overlap_cache = {}
        self._covered_scheme_cache = None
        self._covering_morphism_cache = None
        self._family_morphism_cache = None

    def _repr_(self):
        return (
            f'Affine degree-{self._degree} cyclic-cover '
            f'family of {self._linear_system}'
        )

    def _latex_(self):
        return (
            r'\mathcal X_{\mathrm{cyc}}\dashrightarrow '
            + str(latex(self.parameter_space()))
        )

    def linear_system(self):
        return self._linear_system

    def root_line_bundle(self):
        return self._root_line_bundle

    def degree(self):
        return self._degree

    def divisor_family(self):
        return self._divisor_family

    def parameter_space(self):
        return self._divisor_family.parameter_space()

    def parameter_coordinates(self):
        return self._divisor_family.parameter_coordinates()

    def universal_section(self):
        return self._divisor_family.universal_section()

    def parameter_point(self, branch_section):
        return self._divisor_family.parameter_point(
            branch_section
        )

    def specialize_branch_section(self, parameter_point):
        return self._divisor_family.specialize(
            parameter_point
        )

    def specialize(self, parameter_point):
        return self._root_line_bundle.cyclic_cover(
            self.specialize_branch_section(
                parameter_point
            ),
            self._degree,
        )

    fiber_morphism = specialize

    def fiber_scheme(self, parameter_point):
        return self.specialize(
            parameter_point
        ).domain()

    def universal_branch_divisor(self):
        return self._divisor_family.universal_divisor()

    def root_line_bundle_family(self):
        return self._root_line_bundle_family

    def cyclic_cover_datum(self):
        return self._cyclic_cover_datum

    def cover_algebra_datum(self):
        return self._cyclic_cover_datum.cover_algebra_datum()

    def labels(self):
        return self._linear_system.scheme().affine_cover().labels()

    def chart(self, label):
        if label not in self._chart_cache:
            chart_morphism = (
                self._linear_system.scheme()
                .affine_cover()
                .chart(label)
            )
            self._chart_cache[label] = (
                AffineCyclicCoverChart(
                    self,
                    chart_morphism,
                )
            )
        return self._chart_cache[label]

    def charts(self):
        return tuple(
            self.chart(label)
            for label in self.labels()
        )

    def overlap(self, left_label, right_label):
        if left_label == right_label:
            chart = self.chart(left_label)
            identity = chart.cover_scheme().identity_morphism()
            return CoveredSchemeOverlap(
                left_label,
                right_label,
                chart.cover_scheme(),
                identity,
                identity,
                certificate='identity overlap',
            )
        key = (left_label, right_label)
        if key in self._overlap_cache:
            return self._overlap_cache[key]
        reverse_key = (right_label, left_label)
        if reverse_key in self._overlap_cache:
            return self._overlap_cache[
                reverse_key
            ].reversed()
        overlap = _cyclic_cover_overlap(
            self,
            left_label,
            right_label,
        )
        self._overlap_cache[key] = overlap
        return overlap

    def overlaps(self):
        return tuple(
            self.overlap(left_label, right_label)
            for left_label, right_label in _chart_combinations(
                self.labels(),
                2,
            )
        )

    def cover_scheme(self):
        if self._covered_scheme_cache is None:
            overlaps = {
                (
                    overlap.left_label(),
                    overlap.right_label(),
                ): overlap
                for overlap in self.overlaps()
            }
            cocycle_verified = (
                _cyclic_cover_cocycle_verified(self)
            )
            if not cocycle_verified:
                raise ArithmeticError(
                    'the cyclic-cover transition maps do not satisfy the cocycle condition'
                )
            self._covered_scheme_cache = CoveredScheme(
                {
                    label: self.chart(label).cover_scheme()
                    for label in self.labels()
                },
                overlaps,
                self.parameter_space(),
                cocycle_verified=True,
                label=(
                    r'\mathcal X_{\mathrm{cyc},\mathbf A(V)}'
                ),
            )
        return self._covered_scheme_cache

    def covering_morphism(self):
        if self._covering_morphism_cache is None:
            self._covering_morphism_cache = (
                self.cover_scheme().morphism(
                    self.divisor_family().ambient_family(),
                    {
                        label: self.chart(label)
                        .cover_to_ambient_family()
                        for label in self.labels()
                    },
                    latex_name=r'\pi_{\mathrm{univ}}',
                )
            )
        return self._covering_morphism_cache

    def family_morphism(self):
        if self._family_morphism_cache is None:
            self._family_morphism_cache = (
                self.cover_scheme().morphism(
                    self.parameter_space(),
                    {
                        label: self.chart(label)
                        .parameter_projection()
                        for label in self.labels()
                    },
                    latex_name=r'p_{\mathrm{univ}}',
                )
            )
            self.cover_scheme()._base_morphism_cache = (
                self._family_morphism_cache
            )
        return self._family_morphism_cache

    def global_morphism(self):
        return self.covering_morphism()

    def projection(self):
        return self.family_morphism()

    def relative_branch_singular_locus(self):
        return self._divisor_family.relative_singular_locus()

    def discriminant_ideal(self):
        return self._divisor_family.discriminant_ideal()

    def discriminant_subscheme(self):
        return self._divisor_family.discriminant_subscheme()

    def avoidance_polynomial(self, finite_embedding):
        return self._divisor_family.avoidance_polynomial(
            finite_embedding
        )



def _linear_system_cyclic_cover_family(
    self,
    root_line_bundle,
    degree,
    parameter_names=None,
):
    attribute = (
        '_projective_framework_affine_cyclic_cover_family_cache'
    )
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = (
        id(root_line_bundle),
        ZZ(degree),
        None
        if parameter_names is None
        else tuple(str(name) for name in parameter_names),
    )
    if key not in cache:
        cache[key] = AffineCyclicCoverFamily(
            self,
            root_line_bundle,
            degree,
            parameter_names=parameter_names,
        )
    return cache[key]


LinearSystem.cyclic_cover_family = (
    _linear_system_cyclic_cover_family
)

print('Installed affine cyclic-cover family charts over linear-system coefficient spaces.')

Installed affine cyclic-cover family charts over linear-system coefficient spaces.


## Covered schemes and compatible global morphisms

A covered scheme is specified by affine charts $U_i$, affine overlap schemes $U_{ij}$, and open-immersion maps

$$
U_{ij}\longrightarrow U_i,
\qquad
U_{ij}\longrightarrow U_j,
$$

satisfying the cocycle condition. `CoveredScheme` is a Sage `Scheme` parent storing this atlas. A `CoveredSchemeMorphism` is a genuine scheme morphism represented by compatible chart morphisms; compatibility is checked after pullback to every overlap by comparing the induced ring maps.

The cyclic-cover family uses localizations of the native monic quotient rings. Transition maps are derived from homogeneous-coordinate ratios and the transition functions of the root line bundle, and the cocycle is verified on all triple intersections.

In [54]:
from sage.schemes.generic.scheme import Scheme
from sage.schemes.generic.morphism import SchemeMorphism


def _ring_map_test_elements(ring):
    elements = list(ring.gens())
    coefficient_ring = ring.base_ring()
    visited = set()
    while (
        id(coefficient_ring) not in visited
        and hasattr(coefficient_ring, 'gens')
        and coefficient_ring is not ring
    ):
        visited.add(id(coefficient_ring))
        try:
            elements.extend(
                ring(generator)
                for generator in coefficient_ring.gens()
            )
        except (TypeError, ValueError):
            pass
        next_ring = (
            coefficient_ring.base_ring()
            if hasattr(coefficient_ring, 'base_ring')
            else coefficient_ring
        )
        if next_ring is coefficient_ring:
            break
        coefficient_ring = next_ring
    return tuple(elements)


def _ring_maps_equal(left, right):
    if (
        left.domain() != right.domain()
        or left.codomain() != right.codomain()
    ):
        return False
    try:
        if left == right:
            return True
    except (TypeError, ValueError, NotImplementedError):
        pass
    return all(
        left(element) == right(element)
        for element in _ring_map_test_elements(
            left.domain()
        )
    )


def _projective_ring_maps_define_same_morphism(
    target,
    left_map,
    right_map,
):
    target_ring = target.coordinate_ring()
    left_images = tuple(
        left_map(generator)
        for generator in target_ring.gens()
    )
    right_images = tuple(
        right_map(generator)
        for generator in target_ring.gens()
    )
    left_blocks = _coordinate_blocks(
        target,
        left_images,
    )
    right_blocks = _coordinate_blocks(
        target,
        right_images,
    )
    for left_block, right_block in zip(
        left_blocks,
        right_blocks,
    ):
        for first in range(len(left_block)):
            for second in range(first + 1, len(left_block)):
                if (
                    left_block[first] * right_block[second]
                    - left_block[second] * right_block[first]
                    != 0
                ):
                    return False

    coefficient_ring = target_ring.base_ring()
    if hasattr(coefficient_ring, 'gens'):
        for coefficient in coefficient_ring.gens():
            target_coefficient = target_ring(coefficient)
            if (
                left_map(target_coefficient)
                != right_map(target_coefficient)
            ):
                return False
    return True


class CoveredSchemeOverlap(SageObject):
    def __init__(
        self,
        left_label,
        right_label,
        overlap_scheme,
        map_to_left,
        map_to_right,
        certificate=None,
    ):
        if map_to_left.domain() != overlap_scheme:
            raise ValueError(
                'the left overlap map has the wrong domain'
            )
        if map_to_right.domain() != overlap_scheme:
            raise ValueError(
                'the right overlap map has the wrong domain'
            )
        self._left_label = left_label
        self._right_label = right_label
        self._scheme = overlap_scheme
        self._map_to_left = map_to_left
        self._map_to_right = map_to_right
        self._certificate = certificate

    def _repr_(self):
        return (
            f'Overlap of charts {self._left_label} '
            f'and {self._right_label}'
        )

    def _latex_(self):
        return (
            r'U_{'
            + str(self._left_label)
            + str(self._right_label)
            + r'}'
        )

    def left_label(self):
        return self._left_label

    def right_label(self):
        return self._right_label

    def scheme(self):
        return self._scheme

    def map_to_left(self):
        return self._map_to_left

    def map_to_right(self):
        return self._map_to_right

    left_map = map_to_left
    right_map = map_to_right

    def certificate(self):
        return self._certificate

    def reversed(self):
        return CoveredSchemeOverlap(
            self._right_label,
            self._left_label,
            self._scheme,
            self._map_to_right,
            self._map_to_left,
            certificate=self._certificate,
        )


class CoveredSchemeMorphism(SchemeMorphism):
    def __init__(
        self,
        parent,
        local_morphisms,
        check=True,
        latex_name=None,
    ):
        SchemeMorphism.__init__(self, parent)
        self._local_morphisms = dict(local_morphisms)
        self._latex_name = latex_name
        domain = self.domain()
        if not isinstance(domain, CoveredScheme):
            raise TypeError(
                'a covered-scheme morphism must have a CoveredScheme domain'
            )
        if set(self._local_morphisms) != set(
            domain.labels()
        ):
            raise ValueError(
                'a covered-scheme morphism needs one local morphism on every chart'
            )
        for label, local_morphism in (
            self._local_morphisms.items()
        ):
            if local_morphism.domain() != domain.chart(label):
                raise ValueError(
                    f'the local morphism on chart {label} has the wrong domain'
                )
            if local_morphism.codomain() != self.codomain():
                raise ValueError(
                    f'the local morphism on chart {label} has the wrong codomain'
                )
        if check and not self.is_compatible():
            raise ArithmeticError(
                'the local morphisms do not agree on chart overlaps'
            )

    def _repr_(self):
        return (
            f'Covered-scheme morphism from '
            f'{self.domain()} to {self.codomain()}'
        )

    def _latex_(self):
        symbol = (
            self._latex_name
            if self._latex_name is not None
            else r'f'
        )
        return (
            symbol
            + r':\;'
            + str(latex(self.domain()))
            + r'\longrightarrow '
            + str(latex(self.codomain()))
        )

    def local_morphisms(self):
        return dict(self._local_morphisms)

    def local_morphism(self, label):
        return self._local_morphisms[label]

    def compatible_on_overlap(self, overlap):
        left_local = self.local_morphism(
            overlap.left_label()
        )
        right_local = self.local_morphism(
            overlap.right_label()
        )
        if not all(
            hasattr(morphism, 'ring_homomorphism')
            for morphism in (
                left_local,
                right_local,
                overlap.map_to_left(),
                overlap.map_to_right(),
            )
        ):
            raise NotImplementedError(
                'overlap compatibility currently requires explicit affine ring maps'
            )
        left_pullback = (
            overlap.map_to_left().ring_homomorphism()
            * left_local.ring_homomorphism()
        )
        right_pullback = (
            overlap.map_to_right().ring_homomorphism()
            * right_local.ring_homomorphism()
        )
        try:
            _require_supported_projective_ambient(
                self.codomain(),
                'covered-scheme morphism compatibility',
            )
        except NotImplementedError:
            return _ring_maps_equal(
                left_pullback,
                right_pullback,
            )
        return _projective_ring_maps_define_same_morphism(
            self.codomain(),
            left_pullback,
            right_pullback,
        )

    def is_compatible(self):
        return all(
            self.compatible_on_overlap(overlap)
            for overlap in self.domain().overlaps()
        )


class CoveredScheme(Scheme):
    def __init__(
        self,
        charts,
        overlaps,
        base_scheme,
        cocycle_verified=False,
        label=None,
    ):
        Scheme.__init__(self, base_scheme)
        self._charts = dict(charts)
        self._overlaps = dict(overlaps)
        self._cocycle_verified = bool(
            cocycle_verified
        )
        self._label = label
        self._base_morphism_cache = None
        if len(self._charts) == 0:
            raise ValueError(
                'a covered scheme needs at least one affine chart'
            )
        chart_dimensions = {
            ZZ(chart.dimension())
            for chart in self._charts.values()
        }
        if len(chart_dimensions) != 1:
            raise ValueError(
                'all affine charts of a covered scheme must have the same dimension'
            )
        self._dimension = chart_dimensions.pop()
        for (left_label, right_label), overlap in (
            self._overlaps.items()
        ):
            if left_label == right_label:
                raise ValueError(
                    'nontrivial overlap data must use distinct chart labels'
                )
            if overlap.left_label() != left_label:
                raise ValueError(
                    'the overlap key and left label disagree'
                )
            if overlap.right_label() != right_label:
                raise ValueError(
                    'the overlap key and right label disagree'
                )
            if overlap.map_to_left().codomain() != self.chart(
                left_label
            ):
                raise ValueError(
                    'the left overlap map has the wrong codomain'
                )
            if overlap.map_to_right().codomain() != self.chart(
                right_label
            ):
                raise ValueError(
                    'the right overlap map has the wrong codomain'
                )

    def _repr_(self):
        if self._label is not None:
            return str(self._label)
        return (
            f'Covered scheme with {len(self._charts)} '
            f'affine charts over {self.base_scheme()}'
        )

    def _latex_(self):
        if self._label is not None:
            return str(self._label)
        return r'\mathcal X'

    def labels(self):
        return tuple(self._charts)

    def chart(self, label):
        return self._charts[label]

    def charts(self):
        return tuple(
            self._charts[label]
            for label in self.labels()
        )

    def overlap(self, left_label, right_label):
        if left_label == right_label:
            chart = self.chart(left_label)
            identity = chart.identity_morphism()
            return CoveredSchemeOverlap(
                left_label,
                right_label,
                chart,
                identity,
                identity,
                certificate='identity overlap',
            )
        key = (left_label, right_label)
        if key in self._overlaps:
            return self._overlaps[key]
        reverse_key = (right_label, left_label)
        if reverse_key in self._overlaps:
            return self._overlaps[reverse_key].reversed()
        raise KeyError(
            f'no overlap is stored for {left_label} and {right_label}'
        )

    def overlaps(self):
        return tuple(self._overlaps.values())

    def dimension(self):
        return self._dimension

    def gluing_verified(self):
        return self._cocycle_verified

    def is_separated(self):
        return self._cocycle_verified

    def morphism(
        self,
        target,
        local_morphisms,
        check=True,
        latex_name=None,
    ):
        return CoveredSchemeMorphism(
            self.Hom(target),
            local_morphisms,
            check=check,
            latex_name=latex_name,
        )

    def base_morphism(self):
        if self._base_morphism_cache is None:
            local_maps = {
                label: self.chart(label).structure_morphism()
                if self.chart(label).base_scheme()
                == self.base_scheme()
                else None
                for label in self.labels()
            }
            if any(
                morphism is None
                for morphism in local_maps.values()
            ):
                raise NotImplementedError(
                    'the affine charts do not carry the covered scheme base as their native base scheme'
                )
            self._base_morphism_cache = self.morphism(
                self.base_scheme(),
                local_maps,
                latex_name=r'p',
            )
        return self._base_morphism_cache


print('Loaded covered schemes, overlaps, and compatible global morphisms.')

Loaded covered schemes, overlaps, and compatible global morphisms.


In [52]:
from itertools import combinations as _chart_combinations
from itertools import product as _chart_product


def _chart_pivot_indices(chart_morphism):
    blocks = _coordinate_blocks(
        chart_morphism.codomain(),
        tuple(chart_morphism.defining_polynomials()),
    )
    pivots = []
    for block in blocks:
        candidates = tuple(
            index
            for index, polynomial in enumerate(block)
            if polynomial == 1
        )
        if len(candidates) != 1:
            raise ArithmeticError(
                'a standard affine chart must have exactly one unit homogeneous coordinate in each projective factor'
            )
        pivots.append(ZZ(candidates[0]))
    return tuple(pivots)


def _chart_transition_images(
    left_chart,
    right_chart,
    target_ring,
):
    source_scheme = left_chart.family().linear_system().scheme()
    left_blocks = _coordinate_blocks(
        source_scheme,
        tuple(
            left_chart.chart_to_source().defining_polynomials()
        ),
    )
    right_pivots = _chart_pivot_indices(
        right_chart.chart_morphism()
    )
    target_images = []
    denominators = []
    coefficient_ring = target_ring.base_ring()
    for block, right_pivot in zip(
        left_blocks,
        right_pivots,
    ):
        denominator = coefficient_ring(
            block[right_pivot]
        )
        denominators.append(denominator)
        inverse_denominator = ~denominator
        for coordinate_index, coordinate in enumerate(block):
            if coordinate_index == right_pivot:
                continue
            target_images.append(
                target_ring(
                    coefficient_ring(coordinate)
                    * inverse_denominator
                )
            )
    return (
        tuple(target_images),
        tuple(denominators),
    )


def _cyclic_cover_overlap(
    family,
    left_label,
    right_label,
):
    left_chart = family.chart(left_label)
    right_chart = family.chart(right_label)
    left_base_ring = (
        left_chart.base_chart_family().coordinate_ring()
    )
    right_base_ring = (
        right_chart.base_chart_family().coordinate_ring()
    )
    left_blocks = _coordinate_blocks(
        family.linear_system().scheme(),
        tuple(
            left_chart.chart_to_source().defining_polynomials()
        ),
    )
    right_pivots = _chart_pivot_indices(
        right_chart.chart_morphism()
    )
    overlap_denominators = tuple(
        left_blocks[factor_index][right_pivot]
        for factor_index, right_pivot in enumerate(
            right_pivots
        )
        if left_blocks[factor_index][right_pivot] != 1
    )
    if any(
        denominator == 0
        for denominator in overlap_denominators
    ):
        raise ArithmeticError(
            'a chart overlap denominator is zero'
        )
    if len(overlap_denominators) == 0:
        localized_base_ring = left_base_ring
    else:
        localized_base_ring = left_base_ring.localization(
            overlap_denominators
        )
    label_suffix = (
        '_'.join(str(index) for index in left_label)
        + '__'
        + '_'.join(str(index) for index in right_label)
    )
    overlap_polynomial_ring = PolynomialRing(
        localized_base_ring,
        names=(f'z_overlap_{label_suffix}',),
    )
    overlap_polynomial_coordinate = (
        overlap_polynomial_ring.gen(0)
    )
    overlap_equation = (
        overlap_polynomial_coordinate**family.degree()
        - overlap_polynomial_ring(
            left_chart.branch_function()
        )
    )
    overlap_coordinate_ring = (
        overlap_polynomial_ring.quotient(
            overlap_equation,
            names=(f'zbar_overlap_{label_suffix}',),
        )
    )
    overlap_coordinate = overlap_coordinate_ring.gen(0)
    overlap_scheme = Spec(overlap_coordinate_ring)

    left_base_map = left_base_ring.hom(
        overlap_coordinate_ring
    )
    left_ring_map = (
        left_chart.cover_coordinate_ring().hom(
            [overlap_coordinate],
            overlap_coordinate_ring,
            base_map=left_base_map,
        )
    )
    map_to_left = overlap_scheme.hom(
        left_ring_map,
        left_chart.cover_scheme(),
    )

    (
        right_coordinate_images,
        transition_denominators,
    ) = _chart_transition_images(
        left_chart,
        right_chart,
        overlap_coordinate_ring,
    )
    right_base_map = right_base_ring.hom(
        right_coordinate_images,
        overlap_coordinate_ring,
    )
    root_degrees = (
        family.root_line_bundle().multidegree()
    )
    root_transition = prod(
        denominator**degree
        for denominator, degree in zip(
            transition_denominators,
            root_degrees,
        )
    )
    right_cover_coordinate = (
        overlap_coordinate
        * overlap_coordinate_ring(
            ~root_transition
        )
    )
    right_ring_map = (
        right_chart.cover_coordinate_ring().hom(
            [right_cover_coordinate],
            overlap_coordinate_ring,
            base_map=right_base_map,
        )
    )
    map_to_right = overlap_scheme.hom(
        right_ring_map,
        right_chart.cover_scheme(),
    )

    if (
        left_base_map(left_chart.branch_function())
        != overlap_coordinate**family.degree()
    ):
        raise ArithmeticError(
            'the left cover equation does not hold on the overlap'
        )
    if (
        right_base_map(right_chart.branch_function())
        != right_cover_coordinate**family.degree()
    ):
        raise ArithmeticError(
            'the right cover equation does not hold on the overlap'
        )
    parameter_ring = (
        family.parameter_space().coordinate_ring()
    )
    if not all(
        left_ring_map(
            left_chart.parameter_ring_map()(parameter)
        )
        == right_ring_map(
            right_chart.parameter_ring_map()(parameter)
        )
        for parameter in parameter_ring.gens()
    ):
        raise ArithmeticError(
            'the parameter projections disagree on the overlap'
        )

    return CoveredSchemeOverlap(
        left_label,
        right_label,
        overlap_scheme,
        map_to_left,
        map_to_right,
        certificate={
            'localized_base_ring': localized_base_ring,
            'overlap_denominators': overlap_denominators,
            'root_transition': root_transition,
            'right_coordinate_images': (
                right_coordinate_images
            ),
        },
    )


def _cyclic_cover_cocycle_verified(family):
    source_scheme = family.linear_system().scheme()
    root_degrees = (
        family.root_line_bundle().multidegree()
    )
    labels = family.labels()
    for left_label, middle_label, right_label in (
        _chart_product(labels, repeat=3)
    ):
        left_chart = family.chart(left_label)
        left_base_ring = (
            left_chart.base_chart_family().coordinate_ring()
        )
        (
            flat_ring,
            to_flat,
            from_flat,
            parameter_count,
        ) = _flatten_nested_polynomial_ring(
            left_base_ring
        )
        fraction_field = flat_ring.fraction_field()
        left_blocks = tuple(
            tuple(
                fraction_field(to_flat(coordinate))
                for coordinate in block
            )
            for block in _coordinate_blocks(
                source_scheme,
                tuple(
                    left_chart.chart_to_source()
                    .defining_polynomials()
                ),
            )
        )
        middle_pivots = _chart_pivot_indices(
            family.chart(middle_label).chart_morphism()
        )
        right_pivots = _chart_pivot_indices(
            family.chart(right_label).chart_morphism()
        )

        for block, middle_pivot, right_pivot in zip(
            left_blocks,
            middle_pivots,
            right_pivots,
        ):
            for coordinate_index in range(len(block)):
                if coordinate_index == right_pivot:
                    continue
                direct = (
                    block[coordinate_index]
                    / block[right_pivot]
                )
                via_middle = (
                    (
                        block[coordinate_index]
                        / block[middle_pivot]
                    )
                    / (
                        block[right_pivot]
                        / block[middle_pivot]
                    )
                )
                if direct != via_middle:
                    return False

        root_left_middle = prod(
            block[middle_pivot]**degree
            for block, middle_pivot, degree in zip(
                left_blocks,
                middle_pivots,
                root_degrees,
            )
        )
        root_left_right = prod(
            block[right_pivot]**degree
            for block, right_pivot, degree in zip(
                left_blocks,
                right_pivots,
                root_degrees,
            )
        )
        root_middle_right_in_left = prod(
            (
                block[right_pivot]
                / block[middle_pivot]
            )**degree
            for (
                block,
                middle_pivot,
                right_pivot,
                degree,
            ) in zip(
                left_blocks,
                middle_pivots,
                right_pivots,
                root_degrees,
            )
        )
        if (
            (1 / root_left_middle)
            * (1 / root_middle_right_in_left)
            != (1 / root_left_right)
        ):
            return False
    return True


print('Loaded cyclic-cover overlap construction and cocycle verification.')

Loaded cyclic-cover overlap construction and cocycle verification.


## Covered diagonal-sign involutions and invariant quotient families

Assume $X=\mathbf P^1\times\mathbf P^1$, the base involution acts by simultaneous sign change on every standard affine chart, the universal branch subsystem is invariant, and the root line bundle has even multidegree. The lift acting by $z\mapsto-z$ is a chart-preserving automorphism of the covered cyclic family.

Its quotient is constructed chartwise from the invariant generators

$$
A=u^2,
\quad B=uv,
\quad C=v^2,
\quad P=uz,
\quad Q=vz,
\quad R=z^2,
$$

with the rank-one symmetric-matrix relations and $R=\bar f(A,B,C)$. Quotient overlaps are principal-open affine schemes obtained by adjoining inverses of $A$ and/or $C$. A global covered quotient scheme, its parameter morphism, and the global quotient morphism are returned only after all overlap factorizations and the transition cocycle are verified.

In [51]:
def _even_invariant_lift(
    polynomial,
    A,
    B,
    C,
    target_ring,
):
    lifted = target_ring.zero()
    for exponent, coefficient in polynomial.dict().items():
        if len(exponent) != 2:
            raise ValueError(
                'the invariant-lift backend requires two affine fiber coordinates'
            )
        u_exponent, v_exponent = exponent
        if (
            u_exponent % 2 == 0
            and v_exponent % 2 == 0
        ):
            monomial = (
                A**(u_exponent // 2)
                * C**(v_exponent // 2)
            )
        elif (
            u_exponent % 2 == 1
            and v_exponent % 2 == 1
        ):
            monomial = (
                B
                * A**((u_exponent - 1) // 2)
                * C**((v_exponent - 1) // 2)
            )
        else:
            raise ValueError(
                'the branch function is not invariant under simultaneous sign change'
            )
        lifted += target_ring(coefficient) * monomial
    return lifted


def _unique_polynomials(polynomials):
    result = []
    for polynomial in polynomials:
        if polynomial not in result:
            result.append(polynomial)
    return tuple(result)


class DiagonalSignInvariantQuotientChart(SageObject):
    def __init__(self, automorphism, label):
        self._automorphism = automorphism
        self._family = automorphism.family()
        self._label = label
        self._cover_chart = self._family.chart(label)
        parameter_ring = (
            self._family.parameter_space().coordinate_ring()
        )
        label_suffix = '_'.join(
            str(index)
            for index in label
        )
        self._polynomial_ring = PolynomialRing(
            parameter_ring,
            names=(
                f'A_{label_suffix}',
                f'B_{label_suffix}',
                f'C_{label_suffix}',
                f'P_{label_suffix}',
                f'Q_{label_suffix}',
                f'R_{label_suffix}',
            ),
        )
        A, B, C, P, Q, R = (
            self._polynomial_ring.gens()
        )
        self._coordinates = (A, B, C, P, Q, R)
        self._branch_invariant_lift = (
            _even_invariant_lift(
                self._cover_chart.branch_function(),
                A,
                B,
                C,
                self._polynomial_ring,
            )
        )
        symmetric_matrix = matrix(
            self._polynomial_ring,
            [
                [A, B, P],
                [B, C, Q],
                [P, Q, R],
            ],
        )
        relations = _unique_polynomials(
            tuple(symmetric_matrix.minors(2))
            + (R - self._branch_invariant_lift,)
        )
        self._defining_ideal = (
            self._polynomial_ring.ideal(relations)
        )
        self._coordinate_ring = (
            self._polynomial_ring.quotient(
                self._defining_ideal,
                names=tuple(
                    f'{name}bar_{label_suffix}'
                    for name in ('A', 'B', 'C', 'P', 'Q', 'R')
                ),
            )
        )
        self._scheme = Spec(self._coordinate_ring)

        cover_base_ring = (
            self._cover_chart.base_chart_family()
            .coordinate_ring()
        )
        u, v = cover_base_ring.gens()
        cover_ring = (
            self._cover_chart.cover_coordinate_ring()
        )
        z = self._cover_chart.cover_coordinate()
        invariant_images = (
            cover_ring(u**2),
            cover_ring(u * v),
            cover_ring(v**2),
            cover_ring(u) * z,
            cover_ring(v) * z,
            z**2,
        )
        self._quotient_ring_map = (
            self._coordinate_ring.hom(
                invariant_images,
                cover_ring,
            )
        )
        self._quotient_morphism = (
            self._cover_chart.cover_scheme().hom(
                self._quotient_ring_map,
                self._scheme,
            )
        )
        parameter_map = parameter_ring.hom(
            self._coordinate_ring
        )
        self._parameter_morphism = self._scheme.hom(
            parameter_map,
            self._family.parameter_space(),
        )

    def _repr_(self):
        return (
            f'Invariant quotient chart {self._label} '
            f'of {self._family}'
        )

    def _latex_(self):
        return (
            r'\mathcal W_{'
            + ''.join(str(index) for index in self._label)
            + r'}'
        )

    def automorphism(self):
        return self._automorphism

    def family(self):
        return self._family

    def label(self):
        return self._label

    def cover_chart(self):
        return self._cover_chart

    def polynomial_ring(self):
        return self._polynomial_ring

    def defining_ideal(self):
        return self._defining_ideal

    def coordinate_ring(self):
        return self._coordinate_ring

    def coordinates(self):
        return self._coordinates

    def branch_invariant_lift(self):
        return self._branch_invariant_lift

    def scheme(self):
        return self._scheme

    def quotient_ring_map(self):
        return self._quotient_ring_map

    def quotient_morphism(self):
        return self._quotient_morphism

    def parameter_morphism(self):
        return self._parameter_morphism


class CoveredDiagonalSignAutomorphism(SchemeMorphism):
    def __init__(
        self,
        family,
        base_automorphism,
        fiber_scalar=-1,
    ):
        if family.degree() != 2:
            raise NotImplementedError(
                'the diagonal-sign quotient backend currently requires a double cover'
            )
        if family.linear_system().scheme().dimension() != 2:
            raise NotImplementedError(
                'the diagonal-sign quotient backend currently requires a surface base'
            )
        factors = _projective_factors(
            family.linear_system().scheme()
        )
        if not (
            len(factors) == 2
            and all(
                factor.dimension_relative() == 1
                for factor in factors
            )
        ):
            raise NotImplementedError(
                'the invariant generators A,B,C,P,Q,R require two P1 factors'
            )
        root_degrees = (
            family.root_line_bundle().multidegree()
        )
        if any(degree % 2 for degree in root_degrees):
            raise NotImplementedError(
                'the chart-preserving diagonal-sign lift requires even root multidegrees'
            )
        fiber_scalar = family.parameter_space().base_ring()(
            fiber_scalar
        )
        if fiber_scalar != -1:
            raise ValueError(
                'the Enriques lift uses fiber scalar -1'
            )

        self._family = family
        self._base_automorphism = base_automorphism
        self._fiber_scalar = fiber_scalar
        self._local_morphisms = {}
        self._quotient_cache = None
        cover_scheme = family.cover_scheme()
        SchemeMorphism.__init__(
            self,
            cover_scheme.Hom(cover_scheme),
        )

        ambient_sections = (
            family.linear_system().ambient_section_space()
        )
        pullback = ambient_sections.pullback(
            base_automorphism
        )
        if not all(
            pullback(section) == section
            for section in family.linear_system().basis()
        ):
            raise ValueError(
                'the base automorphism does not fix the chosen branch subsystem pointwise'
            )

        source_affine_cover = (
            family.linear_system().scheme().affine_cover()
        )
        for label in family.labels():
            source_chart_morphism = (
                source_affine_cover.chart(label)
            )
            source_chart_scheme = (
                source_chart_morphism.domain()
            )
            source_chart_coordinates = tuple(
                source_chart_scheme.gens()
            )
            local_sign_morphism = source_chart_scheme.hom(
                tuple(
                    -coordinate
                    for coordinate in source_chart_coordinates
                ),
                source_chart_scheme,
            )
            if not _projective_morphisms_equal_on_domain(
                source_chart_morphism * local_sign_morphism,
                base_automorphism * source_chart_morphism,
            ):
                raise ValueError(
                    f'the base automorphism is not simultaneous sign change on chart {label}'
                )

            chart = family.chart(label)
            chart_scheme = chart.cover_scheme()
            base_ring = (
                chart.base_chart_family().coordinate_ring()
            )
            cover_ring = chart.cover_coordinate_ring()
            fiber_coordinates = tuple(base_ring.gens())
            if len(fiber_coordinates) != 2:
                raise ArithmeticError(
                    'each standard affine chart must have two fiber coordinates'
                )
            coefficient_map = cover_ring.coerce_map_from(
                base_ring.base_ring()
            )
            base_map = base_ring.hom(
                tuple(
                    -cover_ring(coordinate)
                    for coordinate in fiber_coordinates
                ),
                cover_ring,
                base_map=coefficient_map,
            )
            ring_map = cover_ring.hom(
                [
                    fiber_scalar
                    * chart.cover_coordinate()
                ],
                cover_ring,
                base_map=base_map,
            )
            local_morphism = chart_scheme.hom(
                ring_map,
                chart_scheme,
            )
            if not _ring_maps_equal(
                ring_map * ring_map,
                cover_ring.hom(cover_ring),
            ):
                raise ArithmeticError(
                    f'the local lift on chart {label} is not an involution'
                )
            self._local_morphisms[label] = local_morphism

        if not self.is_compatible():
            raise ArithmeticError(
                'the diagonal-sign local lifts are incompatible with the cover cocycle'
            )

    def _repr_(self):
        return (
            f'Diagonal-sign covered automorphism of '
            f'{self.domain()}'
        )

    def _latex_(self):
        return r'\iota_{\mathrm{En},\mathrm{univ}}'

    def family(self):
        return self._family

    def base_automorphism(self):
        return self._base_automorphism

    def fiber_scalar(self):
        return self._fiber_scalar

    def local_morphism(self, label):
        return self._local_morphisms[label]

    def local_morphisms(self):
        return dict(self._local_morphisms)

    def is_compatible(self):
        return (
            self._family.cover_scheme().gluing_verified()
            and _cyclic_cover_cocycle_verified(
                self._family
            )
            and all(
                degree % 2 == 0
                for degree in self._family.root_line_bundle().multidegree()
            )
        )

    def is_automorphism(self):
        return True

    def inverse(self):
        return self

    def quotient(self):
        if self._quotient_cache is None:
            self._quotient_cache = (
                DiagonalSignQuotientFamily(self)
            )
        return self._quotient_cache


print('Loaded covered diagonal-sign automorphisms and native invariant quotient charts.')

Loaded covered diagonal-sign automorphisms and native invariant quotient charts.


In [42]:
def _invariant_laurent_monomial_from_principal_open(
    overlap_ring,
    invariant_generators,
    inverse_A,
    inverse_C,
    u_exponent,
    v_exponent,
    z_exponent,
):
    A, B, C, P, Q, R = invariant_generators
    u_exponent = ZZ(u_exponent)
    v_exponent = ZZ(v_exponent)
    z_exponent = ZZ(z_exponent)
    if z_exponent < 0:
        raise NotImplementedError(
            'negative cover-coordinate exponents are unsupported'
        )

    def power_A(exponent):
        exponent = ZZ(exponent)
        if exponent >= 0:
            return A**exponent
        if inverse_A is None:
            raise ValueError(
                'the overlap does not invert A'
            )
        return inverse_A**(-exponent)

    def power_C(exponent):
        exponent = ZZ(exponent)
        if exponent >= 0:
            return C**exponent
        if inverse_C is None:
            raise ValueError(
                'the overlap does not invert C'
            )
        return inverse_C**(-exponent)

    result = R**(z_exponent // 2)
    if z_exponent % 2 == 0:
        if (
            u_exponent % 2 == 0
            and v_exponent % 2 == 0
        ):
            return (
                result
                * power_A(u_exponent // 2)
                * power_C(v_exponent // 2)
            )
        if (
            u_exponent % 2 == 1
            and v_exponent % 2 == 1
        ):
            return (
                result
                * B
                * power_A((u_exponent - 1) // 2)
                * power_C((v_exponent - 1) // 2)
            )
        raise ValueError(
            'the Laurent monomial is not sign invariant'
        )

    if (
        u_exponent % 2 == 1
        and v_exponent % 2 == 0
    ):
        return (
            result
            * P
            * power_A((u_exponent - 1) // 2)
            * power_C(v_exponent // 2)
        )
    if (
        u_exponent % 2 == 0
        and v_exponent % 2 == 1
    ):
        return (
            result
            * Q
            * power_A(u_exponent // 2)
            * power_C((v_exponent - 1) // 2)
        )
    raise ValueError(
        'the odd-cover-coordinate Laurent monomial is not sign invariant'
    )


def _diagonal_sign_quotient_overlap(
    quotient_family,
    left_label,
    right_label,
):
    family = quotient_family.cover_family()
    left_chart = quotient_family.chart(left_label)
    right_chart = quotient_family.chart(right_label)
    left_ring = left_chart.coordinate_ring()
    right_ring = right_chart.coordinate_ring()
    left_polynomial_ring = left_chart.polynomial_ring()
    left_ideal = left_chart.defining_ideal()
    parameter_ring = family.parameter_space().coordinate_ring()

    change_u = left_label[0] != right_label[0]
    change_v = left_label[1] != right_label[1]
    inverse_names = []
    label_suffix = (
        '_'.join(str(index) for index in left_label)
        + '__'
        + '_'.join(str(index) for index in right_label)
    )
    if change_u:
        inverse_names.append(
            f'invA_{label_suffix}'
        )
    if change_v:
        inverse_names.append(
            f'invC_{label_suffix}'
        )

    overlap_polynomial_ring = PolynomialRing(
        parameter_ring,
        names=(
            tuple(
                str(name)
                for name in left_polynomial_ring.variable_names()
            )
            + tuple(inverse_names)
        ),
    )
    quotient_variables = tuple(
        overlap_polynomial_ring.gens()[:6]
    )
    A, B, C, P, Q, R = quotient_variables
    left_polynomial_map = left_polynomial_ring.hom(
        quotient_variables,
        overlap_polynomial_ring,
    )
    relations = [
        left_polynomial_map(generator)
        for generator in left_ideal.gens()
    ]
    next_index = 6
    inverse_A_polynomial = None
    inverse_C_polynomial = None
    if change_u:
        inverse_A_polynomial = (
            overlap_polynomial_ring.gen(next_index)
        )
        next_index += 1
        relations.append(
            A * inverse_A_polynomial - 1
        )
    if change_v:
        inverse_C_polynomial = (
            overlap_polynomial_ring.gen(next_index)
        )
        relations.append(
            C * inverse_C_polynomial - 1
        )

    overlap_coordinate_ring = (
        overlap_polynomial_ring.quotient(
            overlap_polynomial_ring.ideal(relations),
            names=tuple(
                f'{name}bar'
                for name in overlap_polynomial_ring.variable_names()
            ),
        )
    )
    overlap_scheme = Spec(overlap_coordinate_ring)
    invariant_generators = tuple(
        overlap_coordinate_ring.gen(index)
        for index in range(6)
    )
    inverse_A = (
        overlap_coordinate_ring.gen(6)
        if change_u
        else None
    )
    inverse_C = (
        overlap_coordinate_ring.gen(
            6 + int(change_u)
        )
        if change_v
        else None
    )

    left_ring_map = left_ring.hom(
        invariant_generators,
        overlap_coordinate_ring,
    )
    epsilon_u = ZZ(-1 if change_u else 1)
    epsilon_v = ZZ(-1 if change_v else 1)
    root_u, root_v = (
        family.root_line_bundle().multidegree()
    )
    if root_u % 2 or root_v % 2:
        raise NotImplementedError(
            'the quotient overlap requires even root multidegrees'
        )
    z_u = -root_u if change_u else ZZ(0)
    z_v = -root_v if change_v else ZZ(0)
    right_images = (
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            2 * epsilon_u,
            0,
            0,
        ),
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            epsilon_u,
            epsilon_v,
            0,
        ),
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            0,
            2 * epsilon_v,
            0,
        ),
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            epsilon_u + z_u,
            z_v,
            1,
        ),
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            z_u,
            epsilon_v + z_v,
            1,
        ),
        _invariant_laurent_monomial_from_principal_open(
            overlap_coordinate_ring,
            invariant_generators,
            inverse_A,
            inverse_C,
            2 * z_u,
            2 * z_v,
            2,
        ),
    )
    right_ring_map = right_ring.hom(
        right_images,
        overlap_coordinate_ring,
    )
    map_to_left = overlap_scheme.hom(
        left_ring_map,
        left_chart.scheme(),
    )
    map_to_right = overlap_scheme.hom(
        right_ring_map,
        right_chart.scheme(),
    )

    cover_overlap = family.overlap(
        left_label,
        right_label,
    )
    cover_overlap_ring = (
        cover_overlap.scheme().coordinate_ring()
    )
    left_cover_factor = (
        cover_overlap.map_to_left().ring_homomorphism()
        * left_chart.quotient_ring_map()
    )
    factor_images = [
        left_cover_factor(generator)
        for generator in left_ring.gens()
    ]
    localized_cover_base = cover_overlap.certificate()[
        'localized_base_ring'
    ]
    left_cover_base = (
        family.chart(left_label)
        .base_chart_family()
        .coordinate_ring()
    )
    if change_u:
        inverse_u = ~localized_cover_base(
            left_cover_base.gen(0)
        )
        factor_images.append(
            cover_overlap_ring(inverse_u**2)
        )
    if change_v:
        inverse_v = ~localized_cover_base(
            left_cover_base.gen(1)
        )
        factor_images.append(
            cover_overlap_ring(inverse_v**2)
        )
    cover_factor_ring_map = (
        overlap_coordinate_ring.hom(
            tuple(factor_images),
            cover_overlap_ring,
        )
    )
    cover_factor_morphism = (
        cover_overlap.scheme().hom(
            cover_factor_ring_map,
            overlap_scheme,
        )
    )

    left_via_quotient_overlap = (
        cover_factor_ring_map * left_ring_map
    )
    left_via_cover_overlap = (
        cover_overlap.map_to_left().ring_homomorphism()
        * left_chart.quotient_ring_map()
    )
    right_via_quotient_overlap = (
        cover_factor_ring_map * right_ring_map
    )
    right_via_cover_overlap = (
        cover_overlap.map_to_right().ring_homomorphism()
        * right_chart.quotient_ring_map()
    )
    if not _ring_maps_equal(
        left_via_quotient_overlap,
        left_via_cover_overlap,
    ):
        raise ArithmeticError(
            'the quotient morphism does not factor through the left quotient overlap'
        )
    if not _ring_maps_equal(
        right_via_quotient_overlap,
        right_via_cover_overlap,
    ):
        raise ArithmeticError(
            'the quotient morphism does not factor through the right quotient overlap'
        )

    return CoveredSchemeOverlap(
        left_label,
        right_label,
        overlap_scheme,
        map_to_left,
        map_to_right,
        certificate={
            'right_images': right_images,
            'principal_open_relations': tuple(relations),
            'cover_overlap': cover_overlap,
            'cover_factor_morphism': (
                cover_factor_morphism
            ),
        },
    )


def _diagonal_sign_transition_matrix(
    family,
    left_label,
    right_label,
):
    change_u = left_label[0] != right_label[0]
    change_v = left_label[1] != right_label[1]
    epsilon_u = ZZ(-1 if change_u else 1)
    epsilon_v = ZZ(-1 if change_v else 1)
    root_u, root_v = (
        family.root_line_bundle().multidegree()
    )
    return matrix(
        ZZ,
        [
            [
                epsilon_u,
                0,
                -root_u if change_u else 0,
            ],
            [
                0,
                epsilon_v,
                -root_v if change_v else 0,
            ],
            [0, 0, 1],
        ],
    )


def _diagonal_sign_quotient_cocycle_verified(
    family,
):
    labels = family.labels()
    for left, middle, right in _chart_product(
        labels,
        repeat=3,
    ):
        if (
            _diagonal_sign_transition_matrix(
                family,
                left,
                middle,
            )
            * _diagonal_sign_transition_matrix(
                family,
                middle,
                right,
            )
            != _diagonal_sign_transition_matrix(
                family,
                left,
                right,
            )
        ):
            return False
    return True


class CoveredSchemeChartwiseMorphism(SchemeMorphism):
    def __init__(
        self,
        parent,
        local_morphisms,
        overlap_morphisms,
        check=True,
        latex_name=None,
    ):
        SchemeMorphism.__init__(self, parent)
        self._local_morphisms = dict(
            local_morphisms
        )
        self._overlap_morphisms = dict(
            overlap_morphisms
        )
        self._latex_name = latex_name
        if not (
            isinstance(self.domain(), CoveredScheme)
            and isinstance(self.codomain(), CoveredScheme)
        ):
            raise TypeError(
                'a chartwise covered morphism requires covered domain and codomain'
            )
        if set(self._local_morphisms) != set(
            self.domain().labels()
        ):
            raise ValueError(
                'one local morphism is required on every domain chart'
            )
        if check and not self.is_compatible():
            raise ArithmeticError(
                'the chartwise covered morphism is incompatible on overlaps'
            )

    def _repr_(self):
        return (
            f'Chartwise covered morphism from '
            f'{self.domain()} to {self.codomain()}'
        )

    def _latex_(self):
        symbol = (
            self._latex_name
            if self._latex_name is not None
            else r'f'
        )
        return (
            symbol
            + r':\;'
            + str(latex(self.domain()))
            + r'\longrightarrow '
            + str(latex(self.codomain()))
        )

    def local_morphism(self, label):
        return self._local_morphisms[label]

    def overlap_morphism(
        self,
        left_label,
        right_label,
    ):
        key = (left_label, right_label)
        if key in self._overlap_morphisms:
            return self._overlap_morphisms[key]
        reverse = (right_label, left_label)
        if reverse in self._overlap_morphisms:
            return self._overlap_morphisms[reverse]
        raise KeyError(key)

    def compatible_on_overlap(self, overlap):
        left_label = overlap.left_label()
        right_label = overlap.right_label()
        target_overlap = self.codomain().overlap(
            left_label,
            right_label,
        )
        overlap_morphism = self.overlap_morphism(
            left_label,
            right_label,
        )
        left_direct = (
            overlap.map_to_left().ring_homomorphism()
            * self.local_morphism(
                left_label
            ).ring_homomorphism()
        )
        left_factored = (
            overlap_morphism.ring_homomorphism()
            * target_overlap.map_to_left().ring_homomorphism()
        )
        right_direct = (
            overlap.map_to_right().ring_homomorphism()
            * self.local_morphism(
                right_label
            ).ring_homomorphism()
        )
        right_factored = (
            overlap_morphism.ring_homomorphism()
            * target_overlap.map_to_right().ring_homomorphism()
        )
        return (
            _ring_maps_equal(
                left_direct,
                left_factored,
            )
            and _ring_maps_equal(
                right_direct,
                right_factored,
            )
        )

    def is_compatible(self):
        return all(
            self.compatible_on_overlap(overlap)
            for overlap in self.domain().overlaps()
        )


print('Loaded invariant quotient overlaps and chartwise covered morphisms.')

Loaded invariant quotient overlaps and chartwise covered morphisms.


In [43]:
class DiagonalSignQuotientFamily(SageObject):
    def __init__(self, automorphism):
        if not isinstance(
            automorphism,
            CoveredDiagonalSignAutomorphism,
        ):
            raise TypeError(
                'the quotient requires a covered diagonal-sign automorphism'
            )
        self._automorphism = automorphism
        self._cover_family = automorphism.family()
        self._chart_cache = {}
        self._overlap_cache = {}
        self._covered_scheme_cache = None
        self._family_morphism_cache = None
        self._quotient_morphism_cache = None

    def _repr_(self):
        return (
            f'Invariant quotient family of '
            f'{self._cover_family}'
        )

    def _latex_(self):
        return r'\mathcal W_{\mathrm{univ}}'

    def automorphism(self):
        return self._automorphism

    def cover_family(self):
        return self._cover_family

    def parameter_space(self):
        return self._cover_family.parameter_space()

    def labels(self):
        return self._cover_family.labels()

    def chart(self, label):
        if label not in self._chart_cache:
            self._chart_cache[label] = (
                DiagonalSignInvariantQuotientChart(
                    self._automorphism,
                    label,
                )
            )
        return self._chart_cache[label]

    def charts(self):
        return tuple(
            self.chart(label)
            for label in self.labels()
        )

    def overlap(self, left_label, right_label):
        if left_label == right_label:
            chart = self.chart(left_label)
            identity = chart.scheme().identity_morphism()
            return CoveredSchemeOverlap(
                left_label,
                right_label,
                chart.scheme(),
                identity,
                identity,
                certificate='identity overlap',
            )
        key = (left_label, right_label)
        if key in self._overlap_cache:
            return self._overlap_cache[key]
        reverse = (right_label, left_label)
        if reverse in self._overlap_cache:
            return self._overlap_cache[reverse].reversed()
        overlap = _diagonal_sign_quotient_overlap(
            self,
            left_label,
            right_label,
        )
        self._overlap_cache[key] = overlap
        return overlap

    def overlaps(self):
        return tuple(
            self.overlap(left, right)
            for left, right in _chart_combinations(
                self.labels(),
                2,
            )
        )

    def covered_scheme(self):
        if self._covered_scheme_cache is None:
            cocycle_verified = (
                _diagonal_sign_quotient_cocycle_verified(
                    self._cover_family
                )
            )
            if not cocycle_verified:
                raise ArithmeticError(
                    'the invariant quotient transition maps do not satisfy the cocycle condition'
                )
            overlaps = {
                (
                    overlap.left_label(),
                    overlap.right_label(),
                ): overlap
                for overlap in self.overlaps()
            }
            self._covered_scheme_cache = CoveredScheme(
                {
                    label: self.chart(label).scheme()
                    for label in self.labels()
                },
                overlaps,
                self.parameter_space(),
                cocycle_verified=True,
                label=r'\mathcal W_{\mathrm{univ}}',
            )
        return self._covered_scheme_cache

    quotient_scheme = covered_scheme

    def family_morphism(self):
        if self._family_morphism_cache is None:
            self._family_morphism_cache = (
                self.covered_scheme().morphism(
                    self.parameter_space(),
                    {
                        label: self.chart(label)
                        .parameter_morphism()
                        for label in self.labels()
                    },
                    latex_name=r'p_{\mathcal W}',
                )
            )
            self.covered_scheme()._base_morphism_cache = (
                self._family_morphism_cache
            )
        return self._family_morphism_cache

    projection = family_morphism

    def quotient_morphism(self):
        if self._quotient_morphism_cache is None:
            overlap_morphisms = {
                (
                    overlap.left_label(),
                    overlap.right_label(),
                ): overlap.certificate()[
                    'cover_factor_morphism'
                ]
                for overlap in self.overlaps()
            }
            self._quotient_morphism_cache = (
                CoveredSchemeChartwiseMorphism(
                    self._cover_family.cover_scheme().Hom(
                        self.covered_scheme()
                    ),
                    {
                        label: self.chart(label)
                        .quotient_morphism()
                        for label in self.labels()
                    },
                    overlap_morphisms,
                    latex_name=r'q_{\mathrm{univ}}',
                )
            )
        return self._quotient_morphism_cache

    global_quotient_morphism = quotient_morphism


def _affine_cyclic_family_diagonal_sign_automorphism(
    self,
    base_automorphism,
    fiber_scalar=-1,
):
    attribute = (
        '_projective_framework_diagonal_sign_automorphism_cache'
    )
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = (
        id(base_automorphism),
        self.parameter_space().base_ring()(fiber_scalar),
    )
    if key not in cache:
        cache[key] = CoveredDiagonalSignAutomorphism(
            self,
            base_automorphism,
            fiber_scalar=fiber_scalar,
        )
    return cache[key]


def _affine_cyclic_family_diagonal_sign_quotient(
    self,
    base_automorphism,
    fiber_scalar=-1,
):
    return self.diagonal_sign_automorphism(
        base_automorphism,
        fiber_scalar=fiber_scalar,
    ).quotient()


AffineCyclicCoverFamily.diagonal_sign_automorphism = (
    _affine_cyclic_family_diagonal_sign_automorphism
)
AffineCyclicCoverFamily.diagonal_sign_quotient = (
    _affine_cyclic_family_diagonal_sign_quotient
)
AffineCyclicCoverFamily.enriques_lift = (
    _affine_cyclic_family_diagonal_sign_automorphism
)
AffineCyclicCoverFamily.enriques_quotient = (
    _affine_cyclic_family_diagonal_sign_quotient
)

print('Installed global covered diagonal-sign quotient families and quotient morphisms.')

Installed global covered diagonal-sign quotient families and quotient morphisms.


## Global cyclic covers on products of projective spaces

For the supported product-projective bases, the split projective bundle

$$
\mathbf P_X\!\left(\mathcal O_X\oplus M^{-1}\right)
$$

is constructed as a smooth complete toric variety from its Cox grading. If $s\in H^0(X,M^{\otimes n})$, the homogeneous hypersurface

$$
v^n-\Phi(s)u^n=0
$$

is contained in the chart $u\neq0$ and is canonically the relative spectrum of $\bigoplus_{i=0}^{n-1}M^{-i}$. The resulting native Sage morphism to $X$ carries its cyclic-cover datum, branch and ramification subschemes, and—when the base field contains a primitive $n$th root of unity—a deck transformation.

In [27]:
from itertools import product as _fan_product
import sage.schemes.toric.morphism as _toric_morphism_module


def _fresh_cover_coordinate_names(base_names):
    used = set(str(name) for name in base_names)

    def fresh(stem):
        candidate = stem
        index = 0
        while candidate in used:
            index += 1
            candidate = f'{stem}_{index}'
        used.add(candidate)
        return candidate

    return fresh('u_cover'), fresh('v_cover')


class SplitRankTwoProjectiveBundle(SageObject):
    def __init__(self, twisting_line_bundle):
        base_scheme = twisting_line_bundle.scheme()
        if base_scheme != base_scheme.ambient_space():
            raise NotImplementedError(
                'the toric projective-bundle backend currently requires an ambient projective product'
            )
        if not base_scheme.base_ring().is_field():
            raise NotImplementedError(
                'the toric projective-bundle backend requires a field base'
            )

        self._base_scheme = base_scheme
        self._twisting_line_bundle = twisting_line_bundle
        self._dimensions = tuple(
            ZZ(factor.dimension_relative())
            for factor in _projective_factors(base_scheme)
        )
        self._degrees = twisting_line_bundle.multidegree()
        self._base_coordinate_count = sum(
            dimension + 1
            for dimension in self._dimensions
        )
        total_coordinate_count = self._base_coordinate_count + 2
        grading_rank = len(self._dimensions) + 1
        grading_matrix = zero_matrix(
            ZZ,
            grading_rank,
            total_coordinate_count,
        )

        coordinate_blocks = []
        start = 0
        for row, dimension in enumerate(self._dimensions):
            block = tuple(
                range(start, start + dimension + 1)
            )
            coordinate_blocks.append(block)
            for column in block:
                grading_matrix[row, column] = 1
            start += dimension + 1

        self._u_index = self._base_coordinate_count
        self._v_index = self._base_coordinate_count + 1
        for row, degree in enumerate(self._degrees):
            grading_matrix[row, self._v_index] = ZZ(degree)
        grading_matrix[-1, self._u_index] = 1
        grading_matrix[-1, self._v_index] = 1
        self._grading_matrix = grading_matrix

        kernel_matrix = grading_matrix.right_kernel_matrix()
        rays = []
        for column in range(kernel_matrix.ncols()):
            raw_ray = tuple(
                ZZ(entry)
                for entry in kernel_matrix.column(column)
            )
            common_divisor = gcd(
                [abs(entry) for entry in raw_ray if entry != 0]
            ) if any(raw_ray) else ZZ(1)
            rays.append(
                vector(
                    ZZ,
                    tuple(
                        entry // common_divisor
                        for entry in raw_ray
                    ),
                )
            )

        base_cone_choices = []
        for block in coordinate_blocks:
            base_cone_choices.append(
                tuple(
                    tuple(
                        coordinate
                        for coordinate in block
                        if coordinate != omitted
                    )
                    for omitted in block
                )
            )

        maximal_cones = []
        for choice in _fan_product(
            *base_cone_choices,
            (self._u_index, self._v_index),
        ):
            cone = []
            for base_part in choice[:-1]:
                cone.extend(base_part)
            cone.append(choice[-1])
            maximal_cones.append(cone)

        self._fan = Fan(
            maximal_cones,
            rays=rays,
            check=True,
            is_complete=True,
        )
        if not self._fan.is_smooth():
            raise ArithmeticError(
                'the constructed split projective-bundle fan is not smooth'
            )

        base_names = tuple(
            str(name)
            for name in base_scheme.coordinate_ring().variable_names()
        )
        u_name, v_name = _fresh_cover_coordinate_names(base_names)
        self._scheme = ToricVariety(
            self._fan,
            names=base_names + (u_name, v_name),
            base_ring=base_scheme.base_ring(),
        )
        self._base_coordinates = tuple(
            self._scheme.gens()[:self._base_coordinate_count]
        )
        self._u = self._scheme.gens()[self._u_index]
        self._v = self._scheme.gens()[self._v_index]

    def _repr_(self):
        return (
            f'Projective bundle P(O plus {self._twisting_line_bundle.dual()}) '
            f'over {self._base_scheme}'
        )

    def base_scheme(self):
        return self._base_scheme

    def twisting_line_bundle(self):
        return self._twisting_line_bundle

    def scheme(self):
        return self._scheme

    def fan(self):
        return self._fan

    def grading_matrix(self):
        return self._grading_matrix

    def base_coordinates(self):
        return self._base_coordinates

    def fiber_coordinates(self):
        return self._u, self._v


class CyclicCoverMorphism(
    _toric_morphism_module.SchemeMorphism_polynomial_toric_variety
):
    def __init__(
        self,
        parent,
        polynomials,
        datum,
        projective_bundle,
        cover_equation,
    ):
        self._cyclic_cover_datum = datum
        self._projective_bundle = projective_bundle
        self._cover_equation = cover_equation
        self._ramification_cache = None
        self._deck_transformation_cache = None
        super().__init__(parent, polynomials, check=True)

    def cyclic_cover_datum(self):
        return self._cyclic_cover_datum

    def cover_degree(self):
        return self._cyclic_cover_datum.degree()

    def is_finite(self):
        return True

    def projective_bundle(self):
        return self._projective_bundle

    def cover_equation(self):
        return self._cover_equation

    def branch_subscheme(self):
        return self._cyclic_cover_datum.branch_subscheme()

    def ramification_subscheme(self):
        if self._ramification_cache is None:
            _, v = self._projective_bundle.fiber_coordinates()
            ambient = self.domain().ambient_space()
            equations = list(self.domain().defining_polynomials())
            equations.append(v)
            self._ramification_cache = ambient.subscheme(equations)
        return self._ramification_cache

    def deck_transformation(self):
        if self._deck_transformation_cache is None:
            degree = self.cover_degree()
            base_field = self.domain().base_ring()
            root_ring = PolynomialRing(base_field, names=('T_deck',))
            T_deck = root_ring.gen()
            roots = tuple(
                root
                for root in (T_deck**degree - 1).roots(
                    base_field,
                    multiplicities=False,
                )
                if root.multiplicative_order() == degree
            )
            if len(roots) == 0:
                raise NotImplementedError(
                    'the base field does not contain a primitive root of unity of the cover degree'
                )
            primitive_root = roots[0]
            coordinates = list(
                self.projective_bundle().scheme().gens()
            )
            coordinates[-1] = primitive_root * coordinates[-1]
            self._deck_transformation_cache = self.domain().hom(
                coordinates,
                self.domain(),
            )
        return self._deck_transformation_cache


class CyclicCoverLiftMorphism(
    _toric_morphism_module.SchemeMorphism_polynomial_toric_variety
):
    def __init__(
        self,
        parent,
        polynomials,
        cover_projection,
        base_automorphism,
        fiber_scalar,
    ):
        self._cyclic_cover_projection = cover_projection
        self._cyclic_cover_base_automorphism = (
            base_automorphism
        )
        self._cyclic_cover_fiber_scalar = fiber_scalar
        self._fixed_subscheme_cache = None
        super().__init__(parent, polynomials, check=True)

    def cover_projection(self):
        return self._cyclic_cover_projection

    def base_automorphism(self):
        return self._cyclic_cover_base_automorphism

    def fiber_scalar(self):
        return self._cyclic_cover_fiber_scalar

    def is_automorphism(self):
        return True

    def inverse(self):
        base_automorphism = self.base_automorphism()
        base_identity = self.cover_projection().codomain().identity_morphism()
        if _morphisms_have_identical_coordinates(
            base_automorphism,
            base_identity,
        ):
            base_inverse = base_identity
        else:
            base_inverse = base_automorphism.inverse()
        return self.cover_projection().lift_automorphism(
            base_inverse,
            self.fiber_scalar()**(-1),
        )

    def fixed_subscheme(self):
        if self._fixed_subscheme_cache is None:
            projection = self.cover_projection()
            projective_bundle = projection.projective_bundle()
            ambient = projective_bundle.scheme()
            if not ambient.base_ring().is_exact():
                raise NotImplementedError(
                    'the toric fixed-subscheme backend requires an exact base field'
                )
            ambient_ring = ambient.coordinate_ring()
            base_ring = projection.codomain().coordinate_ring()
            base_embedding = base_ring.hom(
                projective_bundle.base_coordinates(),
                ambient_ring,
            )
            equations = list(
                projection.domain().defining_polynomials()
            )

            base_automorphism = self.base_automorphism()
            base_identity = (
                projection.codomain().identity_morphism()
            )
            if not _morphisms_have_identical_coordinates(
                base_automorphism,
                base_identity,
            ):
                base_fixed_subscheme = (
                    base_automorphism.fixed_subscheme()
                )
                equations.extend(
                    base_embedding(equation)
                    for equation
                    in base_fixed_subscheme.defining_polynomials()
                )

            if self.fiber_scalar() != 1:
                equations.append(
                    projective_bundle.fiber_coordinates()[1]
                )

            fixed_ideal = ambient_ring.ideal(equations)
            fixed_ideal = fixed_ideal.saturation(
                _toric_irrelevant_ideal(ambient)
            )[0]
            self._fixed_subscheme_cache = ambient.subscheme(
                fixed_ideal.gens()
            )
        return self._fixed_subscheme_cache



def _cyclic_datum_projective_bundle(self):
    projective_bundle = getattr(
        self,
        '_projective_framework_projective_bundle',
        None,
    )
    if projective_bundle is None:
        projective_bundle = SplitRankTwoProjectiveBundle(
            self.root_line_bundle()
        )
        self._projective_framework_projective_bundle = (
            projective_bundle
        )
    return projective_bundle


def _cyclic_datum_cover_equation(self):
    equation = getattr(
        self,
        '_projective_framework_cover_equation',
        None,
    )
    if equation is None:
        projective_bundle = self.projective_bundle()
        ambient = projective_bundle.scheme()
        ambient_ring = ambient.coordinate_ring()
        base_ring = self.base_scheme().coordinate_ring()
        base_embedding = base_ring.hom(
            projective_bundle.base_coordinates(),
            ambient_ring,
        )
        branch_polynomial = base_embedding(
            self.branch_section().to_polynomial()
        )
        u, v = projective_bundle.fiber_coordinates()
        equation = (
            v**self.degree()
            - branch_polynomial * u**self.degree()
        )
        if not ambient.is_homogeneous(equation):
            raise ArithmeticError(
                'the cyclic-cover equation is not homogeneous in the projective-bundle Cox grading'
            )
        self._projective_framework_cover_equation = equation
    return equation


def _cyclic_datum_cover_scheme(self):
    cover_scheme = getattr(
        self,
        '_projective_framework_cover_scheme',
        None,
    )
    if cover_scheme is None:
        cover_scheme = self.projective_bundle().scheme().subscheme([
            self.cover_equation()
        ])
        self._projective_framework_cover_scheme = cover_scheme
    return cover_scheme


def _cyclic_datum_morphism(self):
    morphism = getattr(
        self,
        '_projective_framework_cover_morphism',
        None,
    )
    if morphism is None:
        cover_scheme = self.cover_scheme()
        base_scheme = self.base_scheme()
        morphism = CyclicCoverMorphism(
            cover_scheme.Hom(base_scheme),
            self.projective_bundle().base_coordinates(),
            self,
            self.projective_bundle(),
            self.cover_equation(),
        )
        self._projective_framework_cover_morphism = morphism
    return morphism


def _line_bundle_cyclic_cover(self, branch_section, degree):
    return self.cyclic_cover_datum(
        branch_section,
        degree,
    ).morphism()


def _morphisms_have_identical_coordinates(left, right):
    return (
        left.domain() == right.domain()
        and left.codomain() == right.codomain()
        and tuple(left.defining_polynomials())
        == tuple(right.defining_polynomials())
    )


def _section_proportionality_scalar(section, image):
    if section.parent() is not image.parent():
        raise ValueError(
            'the two sections must lie in the same section space'
        )
    section_vector = tuple(section.to_vector())
    image_vector = tuple(image.to_vector())
    scalar = None
    for source_coefficient, image_coefficient in zip(
        section_vector,
        image_vector,
    ):
        if source_coefficient != 0:
            candidate = image_coefficient / source_coefficient
            if scalar is None:
                scalar = candidate
            elif candidate != scalar:
                raise ValueError(
                    'the pulled-back branch section is not a scalar multiple of the original section'
                )
        elif image_coefficient != 0:
            raise ValueError(
                'the pulled-back branch section is not a scalar multiple of the original section'
            )
    if scalar is None:
        raise ValueError(
            'lifting an automorphism for the zero branch section is not currently implemented'
        )
    return scalar


def _cyclic_cover_branch_scaling(self, base_automorphism):
    base_scheme = self.codomain()
    if (
        base_automorphism.domain() != base_scheme
        or base_automorphism.codomain() != base_scheme
    ):
        raise ValueError(
            'the supplied morphism must be an endomorphism of the cover base'
        )
    identity_morphism = base_scheme.identity_morphism()
    is_identity = base_automorphism == identity_morphism
    is_automorphism = (
        is_identity
        or (
            hasattr(base_automorphism, 'is_automorphism')
            and base_automorphism.is_automorphism()
        )
    )
    if not is_automorphism:
        raise ValueError(
            'lifting currently requires an automorphism of the cover base'
        )

    root_line_bundle = (
        self.cyclic_cover_datum().root_line_bundle()
    )
    if root_line_bundle.pullback(base_automorphism) != root_line_bundle:
        raise ValueError(
            'the base automorphism does not preserve the cyclic-cover root line bundle'
        )

    branch_section = self.cyclic_cover_datum().branch_section()
    pulled_branch_section = branch_section.parent().pullback(
        base_automorphism
    )(branch_section)
    return _section_proportionality_scalar(
        branch_section,
        pulled_branch_section,
    )


def _cyclic_cover_lift_automorphism(
    self,
    base_automorphism,
    fiber_scalar,
):
    branch_scaling = self.branch_scaling(base_automorphism)
    fiber_scalar = self.domain().base_ring()(fiber_scalar)
    if fiber_scalar**self.cover_degree() != branch_scaling:
        raise ValueError(
            'the fiber scalar must be an nth root of the branch-section scaling'
        )

    attribute = '_projective_framework_lift_cache'
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = (id(base_automorphism), fiber_scalar)
    if key not in cache:
        projective_bundle = self.projective_bundle()
        ambient_ring = projective_bundle.scheme().coordinate_ring()
        base_ring = self.codomain().coordinate_ring()
        base_embedding = base_ring.hom(
            projective_bundle.base_coordinates(),
            ambient_ring,
        )
        lifted_base_coordinates = tuple(
            base_embedding(polynomial)
            for polynomial in base_automorphism.defining_polynomials()
        )
        u, v = projective_bundle.fiber_coordinates()
        lift = CyclicCoverLiftMorphism(
            self.domain().Hom(self.domain()),
            lifted_base_coordinates
            + (u, fiber_scalar * v),
            self,
            base_automorphism,
            fiber_scalar,
        )
        lifted_projection = self * lift
        projected_base_automorphism = (
            base_automorphism * self
        )
        if tuple(
            lifted_projection.defining_polynomials()
        ) != tuple(
            projected_base_automorphism.defining_polynomials()
        ):
            raise ArithmeticError(
                'the constructed lift does not commute with the cyclic-cover projection'
            )
        lift._cyclic_cover_projection = self
        lift._cyclic_cover_base_automorphism = base_automorphism
        lift._cyclic_cover_fiber_scalar = fiber_scalar
        cache[key] = lift
    return cache[key]


def _cyclic_cover_lift_automorphisms(
    self,
    base_automorphism,
):
    branch_scaling = self.branch_scaling(base_automorphism)
    base_field = self.domain().base_ring()
    root_ring = PolynomialRing(
        base_field,
        names=('T_lift',),
    )
    T_lift = root_ring.gen()
    roots = tuple(
        (T_lift**self.cover_degree() - branch_scaling).roots(
            base_field,
            multiplicities=False,
        )
    )
    if len(roots) == 0:
        raise NotImplementedError(
            'the base field contains no required nth root of the branch-section scaling'
        )
    return tuple(
        self.lift_automorphism(
            base_automorphism,
            root,
        )
        for root in roots
    )


def _cyclic_cover_deck_transformations(self):
    return self.lift_automorphisms(
        self.codomain().identity_morphism()
    )


def _cyclic_cover_deck_transformation(self):
    if self._deck_transformation_cache is None:
        degree = self.cover_degree()
        transformations = self.deck_transformations()
        primitive = tuple(
            transformation
            for transformation in transformations
            if transformation._cyclic_cover_fiber_scalar.multiplicative_order()
            == degree
        )
        if len(primitive) == 0:
            raise NotImplementedError(
                'the base field does not contain a primitive root of unity of the cover degree'
            )
        self._deck_transformation_cache = primitive[0]
    return self._deck_transformation_cache


CyclicCoverMorphism.branch_scaling = (
    _cyclic_cover_branch_scaling
)
CyclicCoverMorphism.lift_automorphism = (
    _cyclic_cover_lift_automorphism
)
CyclicCoverMorphism.lift_automorphisms = (
    _cyclic_cover_lift_automorphisms
)
CyclicCoverMorphism.deck_transformations = (
    _cyclic_cover_deck_transformations
)
CyclicCoverMorphism.deck_transformation = (
    _cyclic_cover_deck_transformation
)


CyclicCoverDatum.projective_bundle = (
    _cyclic_datum_projective_bundle
)
CyclicCoverDatum.cover_equation = _cyclic_datum_cover_equation
CyclicCoverDatum.cover_scheme = _cyclic_datum_cover_scheme
CyclicCoverDatum.morphism = _cyclic_datum_morphism
ProductProjectiveLineBundle.cyclic_cover = (
    _line_bundle_cyclic_cover
)

print('Installed global toric cyclic-cover morphisms on projective products.')

Installed global toric cyclic-cover morphisms on projective products.


## Constructor aliases, standard affine covers, and native scheme points

The aliases `PP`, `AA`, and `OO` return existing Sage spaces and line bundles rather than wrapper types. Supported constructor forms are

$$
\texttt{PP(R,n)},\qquad \texttt{PP(R)^n},\qquad \texttt{(PP^n)(R)},
$$

and similarly for `AA`. Unparenthesized `PP^n(R)` cannot be supported because Python parses it as `PP ** (n(R))`. In a non-preparsed Python context, use `**` in place of `^`.

Sage already uses `AA` for the algebraic real field. This framework intentionally shadows that alias, so it must be executed after `from sage.all import *`; a later wildcard import would restore Sage's original `AA`.

The alias `OO(X,d_1,\ldots,d_r)` constructs an element of `X.Pic()`, `OO(X)` constructs the structure-sheaf class, and `OO(L)` returns the line bundle $L$ after recovering its scheme from its Picard parent.

An $R$-valued point remains Sage's native morphism $\operatorname{Spec}R\to X$, constructed by `X(R)` and `X((\ldots))`. No aliases for point sets or coordinate tuples are added.

The method `X.affine_cover()` returns a cached finite sequence of actual Sage morphisms

$$
j_\alpha:U_\alpha\longrightarrow X,
$$

where each $U_\alpha$ is an affine scheme and each $j_\alpha$ is a certified standard open immersion. The cover records every chart containing a point. Its `canonical_chart(p)` is not random: it is the first containing chart in the standard lexicographic order on homogeneous-coordinate indices. The actual affine coordinates of $p$ in a chart are the coordinates of `chart.preimage_point(p)` in the affine scheme `chart.domain()`.

In [22]:
from sage.schemes.affine.affine_space import AffineSpace_generic
from sage.schemes.affine.affine_subscheme import AlgebraicScheme_subscheme_affine
from sage.schemes.projective.projective_subscheme import AlgebraicScheme_subscheme_projective
from sage.schemes.product_projective.subscheme import AlgebraicScheme_subscheme_product_projective
import sage.schemes.affine.affine_morphism as _affine_morphism_module
import sage.schemes.affine.affine_point as _affine_point_module
import sage.schemes.projective.projective_point as _projective_point_module
import sage.schemes.product_projective.point as _product_point_module
from sage.structure.sage_object import SageObject


class _FixedDimensionSpaceConstructor:
    def __init__(self, constructor, symbol, dimension):
        self._constructor = constructor
        self._symbol = symbol
        self._dimension = ZZ(dimension)

    def _repr_(self):
        return f'{self._symbol}^{self._dimension}'

    __repr__ = _repr_

    def __call__(self, base_ring, *args, **kwds):
        return self._constructor(
            base_ring,
            self._dimension,
            *args,
            **kwds,
        )


class _BaseRingSpaceConstructor:
    def __init__(self, constructor, symbol, base_ring, args, kwds):
        self._constructor = constructor
        self._symbol = symbol
        self._base_ring = base_ring
        self._args = tuple(args)
        self._kwds = dict(kwds)

    def _repr_(self):
        return f'{self._symbol}({self._base_ring})'

    __repr__ = _repr_

    def __pow__(self, dimension):
        return self._constructor(
            self._base_ring,
            ZZ(dimension),
            *self._args,
            **self._kwds,
        )


class _SpaceConstructorAlias:
    def __init__(self, constructor, symbol):
        self._constructor = constructor
        self._symbol = symbol

    def _repr_(self):
        return self._symbol

    __repr__ = _repr_

    def __call__(self, base_ring, dimension=None, *args, **kwds):
        if dimension is None:
            return _BaseRingSpaceConstructor(
                self._constructor,
                self._symbol,
                base_ring,
                args,
                kwds,
            )
        return self._constructor(
            base_ring,
            ZZ(dimension),
            *args,
            **kwds,
        )

    def __pow__(self, dimension):
        return _FixedDimensionSpaceConstructor(
            self._constructor,
            self._symbol,
            dimension,
        )


PP = _SpaceConstructorAlias(ProjectiveSpace, 'PP')
AA = _SpaceConstructorAlias(AffineSpace, 'AA')


def OO(*arguments):
    if len(arguments) == 0:
        raise TypeError('OO requires a scheme or a line bundle')

    first = arguments[0]
    if isinstance(first, ProductProjectiveLineBundle):
        if len(arguments) != 1:
            raise TypeError('OO(L) accepts no additional arguments')
        return first

    scheme = first
    if not hasattr(scheme, 'Pic'):
        raise TypeError('the first argument to OO must be a supported scheme or line bundle')

    if len(arguments) == 1:
        return scheme.Pic().zero()

    twisting_data = arguments[1:]
    if len(twisting_data) == 1:
        datum = twisting_data[0]
        if isinstance(datum, ProductProjectiveLineBundle):
            if datum.scheme() != scheme:
                raise ValueError('OO(X,L) requires L to lie in Pic(X)')
            return datum
        if isinstance(datum, (tuple, list)):
            twisting_data = tuple(datum)

    return scheme.O(*twisting_data)


def _affine_space_function_field(self):
    coordinate_ring = self.coordinate_ring()
    if not coordinate_ring.is_integral_domain():
        raise ValueError('the affine space has no generic point over a non-domain base')
    return coordinate_ring.fraction_field()


def _affine_space_generic_point(self):
    function_field = self.function_field()
    return self(function_field)(
        tuple(function_field(generator) for generator in self.coordinate_ring().gens())
    )


def _standard_affine_chart_labels(scheme):
    ambient = scheme.ambient_space()
    if isinstance(ambient, AffineSpace_generic):
        return (tuple(),)
    factors = _projective_factors(scheme)
    labels = tuple(
        tuple(indices)
        for indices in _cartesian_product(*(
            range(ZZ(factor.dimension_relative()) + 1)
            for factor in factors
        ))
    )
    if len(factors) == 1:
        return tuple(label[0] for label in labels)
    return labels


def _standard_affine_chart_coordinates(scheme, affine_patch, label):
    factors = _projective_factors(scheme)
    label_tuple = (ZZ(label),) if len(factors) == 1 else tuple(ZZ(i) for i in label)
    affine_coordinates = tuple(affine_patch.ambient_space().gens())
    projective_coordinates = []
    start = 0
    for factor, distinguished_index in zip(factors, label_tuple):
        dimension = ZZ(factor.dimension_relative())
        affine_block = affine_coordinates[start:start + dimension]
        start += dimension
        affine_position = 0
        for coordinate_index in range(dimension + 1):
            if coordinate_index == distinguished_index:
                projective_coordinates.append(
                    affine_patch.ambient_space().coordinate_ring().one()
                )
            else:
                projective_coordinates.append(affine_block[affine_position])
                affine_position += 1
    return tuple(projective_coordinates)


def _mark_standard_affine_chart(chart, scheme, label):
    chart._standard_affine_cover_scheme = scheme
    chart._standard_affine_chart_label = label
    return chart


def _standard_affine_chart_contains_point(chart, point):
    scheme = chart.codomain()
    if point.codomain() != scheme:
        return False
    label = chart._standard_affine_chart_label
    if label == tuple():
        return True

    ambient = scheme.ambient_space()
    if isinstance(ambient, ProductProjectiveSpaces_ring):
        return all(
            tuple(point[factor_index])[coordinate_index] != 0
            for factor_index, coordinate_index in enumerate(label)
        )
    return tuple(point)[ZZ(label)] != 0


def _standard_affine_chart_preimage_point(chart, point):
    if not _standard_affine_chart_contains_point(chart, point):
        raise ValueError('the point is not contained in this affine chart')
    label = chart._standard_affine_chart_label
    if label == tuple():
        preimage = point
    elif isinstance(chart.codomain().ambient_space(), ProductProjectiveSpaces_ring):
        preimage = point.dehomogenize([ZZ(index) for index in label])
    else:
        preimage = point.dehomogenize(ZZ(label))
    assert chart(preimage) == point
    return preimage


def _affine_chart_is_standard(self):
    return hasattr(self, '_standard_affine_chart_label')


def _affine_chart_label(self):
    if not self.is_standard_affine_chart():
        raise ValueError('this morphism is not a standard affine chart')
    return self._standard_affine_chart_label


def _affine_chart_is_open_immersion(self):
    if self.is_standard_affine_chart():
        return True
    raise NotImplementedError(
        'open-immersion recognition is currently certified only for standard affine charts'
    )


def _affine_chart_contains_point_method(self, point):
    if not self.is_standard_affine_chart():
        raise ValueError('this morphism is not a standard affine chart')
    return _standard_affine_chart_contains_point(self, point)


def _affine_chart_preimage_point_method(self, point):
    if not self.is_standard_affine_chart():
        raise ValueError('this morphism is not a standard affine chart')
    return _standard_affine_chart_preimage_point(self, point)


class StandardAffineCover(SageObject):
    def __init__(self, scheme, charts, labels):
        self._scheme = scheme
        self._charts = tuple(charts)
        self._labels = tuple(labels)
        self._chart_by_label = dict(zip(self._labels, self._charts))
        self._label_by_chart_id = {
            id(chart): label
            for label, chart in zip(self._labels, self._charts)
        }

    def _repr_(self):
        return f'Standard affine cover of {self._scheme} by {len(self)} charts'

    def scheme(self):
        return self._scheme

    def __len__(self):
        return len(self._charts)

    def __iter__(self):
        return iter(self._charts)

    def __getitem__(self, key):
        if key in self._chart_by_label:
            return self._chart_by_label[key]
        return self._charts[key]

    def labels(self):
        return self._labels

    def charts(self):
        return self._charts

    def chart(self, label):
        return self._chart_by_label[label]

    def label(self, chart):
        try:
            return self._label_by_chart_id[id(chart)]
        except KeyError as error:
            raise ValueError('the morphism is not a chart of this cover') from error

    def __contains__(self, chart):
        return id(chart) in self._label_by_chart_id

    def charts_containing(self, point):
        if point.codomain() != self._scheme:
            raise ValueError('the point does not lie on the scheme covered by this cover')
        return tuple(
            chart
            for chart in self._charts
            if _standard_affine_chart_contains_point(chart, point)
        )

    def labels_containing(self, point):
        return tuple(
            self.label(chart)
            for chart in self.charts_containing(point)
        )

    def canonical_chart(self, point):
        containing = self.charts_containing(point)
        if len(containing) == 0:
            raise ValueError('the point is not contained in any chart of the cover')
        return containing[0]

    def canonical_label(self, point):
        return self.label(self.canonical_chart(point))

    def preimage(self, point, chart=None):
        if chart is None:
            chart = self.canonical_chart(point)
        elif chart not in self:
            chart = self.chart(chart)
        return _standard_affine_chart_preimage_point(chart, point)


def _standard_affine_cover(self):
    attribute = '_projective_framework_standard_affine_cover'
    cover = getattr(self, attribute, None)
    if cover is not None:
        return cover

    labels = _standard_affine_chart_labels(self)
    ambient = self.ambient_space()
    charts = []
    if isinstance(ambient, AffineSpace_generic):
        chart = self.identity_morphism()
        _mark_standard_affine_chart(chart, self, tuple())
        charts.append(chart)
    else:
        for label in labels:
            affine_patch = self.affine_patch(label)
            coordinates = _standard_affine_chart_coordinates(
                self,
                affine_patch,
                label,
            )
            chart = affine_patch.hom(coordinates, self)
            _mark_standard_affine_chart(chart, self, label)
            charts.append(chart)

    cover = StandardAffineCover(self, charts, labels)
    setattr(self, attribute, cover)
    return cover


def _product_point_as_subscheme(self):
    scheme = self.codomain()
    ambient = scheme.ambient_space()
    coordinate_ring = ambient.coordinate_ring()
    coordinate_blocks = _coordinate_blocks(
        scheme,
        coordinate_ring.gens(),
    )
    equations = list(_defining_equations(scheme))
    for factor_index, coordinate_block in enumerate(coordinate_blocks):
        point_block = tuple(self[factor_index])
        comparison = matrix(
            coordinate_ring,
            [
                coordinate_block,
                tuple(coordinate_ring(value) for value in point_block),
            ],
        )
        equations.extend(comparison.minors(2))

    point_ideal = coordinate_ring.ideal(equations)
    point_ideal = _saturate_projective_ideal(
        scheme,
        point_ideal,
        coordinate_ring.gens(),
    )
    return ambient.subscheme(point_ideal.gens())


def _install_constructor_cover_and_point_interface():
    AffineSpace_generic.function_field = _affine_space_function_field
    AffineSpace_generic.generic_point = _affine_space_generic_point

    scheme_classes = (
        AffineSpace_generic,
        AlgebraicScheme_subscheme_affine,
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
        AlgebraicScheme_subscheme_projective,
        AlgebraicScheme_subscheme_product_projective,
    )
    for scheme_class in scheme_classes:
        for obsolete_name in ('points_over', 'point_from'):
            if obsolete_name in scheme_class.__dict__:
                delattr(scheme_class, obsolete_name)
        scheme_class.affine_cover = _standard_affine_cover

    point_classes = (
        _affine_point_module.SchemeMorphism_point_affine,
        _affine_point_module.SchemeMorphism_point_affine_field,
        _affine_point_module.SchemeMorphism_point_affine_finite_field,
        _projective_point_module.SchemeMorphism_point_projective_ring,
        _projective_point_module.SchemeMorphism_point_projective_field,
        _projective_point_module.SchemeMorphism_point_projective_finite_field,
        _product_point_module.ProductProjectiveSpaces_point_ring,
        _product_point_module.ProductProjectiveSpaces_point_field,
        _product_point_module.ProductProjectiveSpaces_point_finite_field,
    )
    for point_class in point_classes:
        for obsolete_name in (
            'coordinates',
            'factor_coordinates',
            'affine_patch_indices',
            'affine_patch',
            'affine_expression',
            'local_equations',
            'local_ideal',
        ):
            if obsolete_name in point_class.__dict__:
                delattr(point_class, obsolete_name)

    for point_class in (
        _product_point_module.ProductProjectiveSpaces_point_ring,
        _product_point_module.ProductProjectiveSpaces_point_field,
        _product_point_module.ProductProjectiveSpaces_point_finite_field,
    ):
        point_class.as_subscheme = _product_point_as_subscheme

    affine_morphism_classes = (
        _affine_morphism_module.SchemeMorphism_polynomial_affine_space,
        _affine_morphism_module.SchemeMorphism_polynomial_affine_space_field,
        _affine_morphism_module.SchemeMorphism_polynomial_affine_space_finite_field,
        _affine_morphism_module.SchemeMorphism_polynomial_affine_subscheme_field,
    )
    for morphism_class in affine_morphism_classes:
        morphism_class.is_standard_affine_chart = _affine_chart_is_standard
        morphism_class.cover_label = _affine_chart_label
        morphism_class.is_open_immersion = _affine_chart_is_open_immersion
        morphism_class.contains_point = _affine_chart_contains_point_method
        morphism_class.preimage_point = _affine_chart_preimage_point_method


_install_constructor_cover_and_point_interface()

print('Installed PP, AA, OO, standard affine covers, and chart morphism operations.')

Installed PP, AA, OO, standard affine covers, and chart morphism operations.


## Relative spectra of finite free quasi-coherent modules

Let $S=\operatorname{Spec}A$ and let $E$ be a finite free $A$-module, viewed through the equivalence

$$
A\text{-}\operatorname{Mod}\simeq\operatorname{QCoh}(S).
$$

We use the convention

$$
\mathbf V_S(E)=\underline{\operatorname{Spec}}_S\operatorname{Sym}_A(E).
$$

Thus, for the structure morphism $p:X\to\operatorname{Spec}k$ and a line bundle $L$ on $X$, the affine scheme of vectors in $H^0(X,L)$ is

$$
\mathbf V_{\operatorname{Spec}k}\!\left((p_*L)^\vee\right)
=
\operatorname{Spec}\operatorname{Sym}_k\!\left(H^0(X,L)^\vee\right).
$$

The next cell shadows Sage's absolute `Spec(R)` constructor so that a commutative $k$-algebra is inferred as `Spec(R,k)`; an explicit second argument still specifies the base. It then implements finite free quasi-coherent modules as Sage modules, their duals and symmetric algebras, `VV(E)`, generic points of integral affine schemes, and `p.pushforward(L)` for the supported structure morphisms. For `VV(E)`, the total affine scheme retains its ground-base scheme and an explicit projection to the affine scheme carrying $E$.

In [29]:
from sage.combinat.free_module import CombinatorialFreeModule
from sage.schemes.generic.scheme import AffineScheme
from sage.schemes.generic.morphism import SchemeMorphism_structure_map
from sage.categories.commutative_rings import CommutativeRings


if '_projective_framework_original_Spec' not in globals():
    _projective_framework_original_Spec = Spec
    _projective_framework_original_affine_base_ring = AffineScheme.base_ring
    _projective_framework_original_affine_base_scheme = AffineScheme.base_scheme
    _projective_framework_original_affine_base_morphism = AffineScheme.base_morphism


def _canonical_ring_map(source_ring, target_ring):
    if source_ring == target_ring:
        return source_ring.identity_morphism()
    coercion = target_ring.coerce_map_from(source_ring)
    if coercion is not None:
        return coercion
    generators = tuple(source_ring.gens())
    if len(generators) == 0:
        return source_ring.hom(target_ring)
    return source_ring.hom(
        [target_ring(generator) for generator in generators],
        target_ring,
    )


def _affine_scheme_base_ring(self):
    if hasattr(self, '_relative_base_ring'):
        return self._relative_base_ring
    return _projective_framework_original_affine_base_ring(self)


def _affine_scheme_base_scheme(self):
    if hasattr(self, '_relative_base_scheme'):
        return self._relative_base_scheme
    return _projective_framework_original_affine_base_scheme(self)


def _affine_scheme_base_morphism(self):
    if hasattr(self, '_relative_base_morphism'):
        return self._relative_base_morphism
    return _projective_framework_original_affine_base_morphism(self)


def Spec(R, S=None):
    if R not in CommutativeRings():
        return _projective_framework_original_Spec(R, S)

    if S is None:
        try:
            inferred_base = R.base_ring()
        except (AttributeError, TypeError, ValueError):
            inferred_base = R
        if inferred_base == R:
            return _projective_framework_original_Spec(R)
        S = inferred_base

    if isinstance(S, AffineScheme):
        base_scheme = S
        base_ring = S.coordinate_ring()
    else:
        base_ring = S
        base_scheme = Spec(base_ring)

    relative_spec = _projective_framework_original_Spec(R, base_ring)
    structure_ring_map = _canonical_ring_map(base_ring, R)
    structure_morphism = relative_spec.hom(
        structure_ring_map,
        base_scheme,
    )
    relative_spec._relative_base_ring = base_ring
    relative_spec._relative_base_scheme = base_scheme
    relative_spec._relative_base_morphism = structure_morphism
    return relative_spec


AffineScheme.base_ring = _affine_scheme_base_ring
AffineScheme.base_scheme = _affine_scheme_base_scheme
AffineScheme.base_morphism = _affine_scheme_base_morphism
AffineScheme.structure_morphism = _affine_scheme_base_morphism

import sage.all as _sage_all_module
import sage.schemes.generic.spec as _spec_module
_sage_all_module.Spec = Spec
_spec_module.Spec = Spec


class AffineQuasiCoherentFreeModule(CombinatorialFreeModule):
    def __init__(
        self,
        base_scheme,
        basis_names,
        dual_basis_names=None,
        source_sections=None,
        ground_ring=None,
    ):
        if not isinstance(base_scheme, AffineScheme):
            raise TypeError('the base of an affine quasi-coherent module must be affine')
        self._base_scheme = base_scheme
        self._coordinate_ring = base_scheme.coordinate_ring()
        if ground_ring is None:
            if self._coordinate_ring.is_field():
                ground_ring = self._coordinate_ring
            else:
                ground_ring = base_scheme.base_ring()
        self._ground_ring = ground_ring
        self._basis_names = tuple(str(name) for name in basis_names)
        if dual_basis_names is None:
            dual_basis_names = tuple(f'{name}_dual' for name in self._basis_names)
        self._dual_basis_names = tuple(str(name) for name in dual_basis_names)
        if len(self._dual_basis_names) != len(self._basis_names):
            raise ValueError('the basis and dual-basis name lists must have equal length')
        self._source_sections = source_sections
        self._dual_cache = None
        self._symmetric_algebra_cache = {}
        self._relative_spec_cache = {}
        CombinatorialFreeModule.__init__(
            self,
            self._coordinate_ring,
            tuple(range(len(self._basis_names))),
            prefix='',
        )

    def _repr_(self):
        return (
            f'Finite free quasi-coherent module of rank {self.rank()} '
            f'on {self._base_scheme}'
        )

    def _repr_term(self, index):
        return self._basis_names[index]

    def base_scheme(self):
        return self._base_scheme

    scheme = base_scheme

    def ground_ring(self):
        return self._ground_ring

    def rank(self):
        return ZZ(len(self._basis_names))

    dimension = rank

    def basis_names(self):
        return self._basis_names

    def source_sections(self):
        return self._source_sections

    def dual(self):
        if self._dual_cache is None:
            dual_module = AffineQuasiCoherentFreeModule(
                self._base_scheme,
                self._dual_basis_names,
                dual_basis_names=self._basis_names,
                ground_ring=self._ground_ring,
            )
            dual_module._dual_of = self
            dual_module._dual_cache = self
            self._dual_cache = dual_module
        return self._dual_cache

    def symmetric_algebra(self, names=None):
        if names is None:
            names = self._basis_names
        names = tuple(str(name) for name in names)
        if len(names) != self.rank():
            raise ValueError(
                f'expected {self.rank()} symmetric-algebra generators, '
                f'received {len(names)}'
            )
        if names not in self._symmetric_algebra_cache:
            self._symmetric_algebra_cache[names] = PolynomialRing(
                self._coordinate_ring,
                names=names,
            )
        return self._symmetric_algebra_cache[names]

    def relative_spec(self, names=None):
        if names is None:
            names = self._basis_names
        names = tuple(str(name) for name in names)
        if names not in self._relative_spec_cache:
            symmetric_algebra = self.symmetric_algebra(names=names)
            relative_spec = Spec(
                symmetric_algebra,
                self._base_scheme,
            )
            relative_spec._vv_module = self
            relative_spec._vv_generator_names = names
            projection = relative_spec.base_morphism()
            projection._projective_framework_morphism_role = (
                'vector_bundle_projection'
            )
            self._relative_spec_cache[names] = relative_spec
        return self._relative_spec_cache[names]


def VV(module, names=None):
    if not isinstance(module, AffineQuasiCoherentFreeModule):
        raise TypeError(
            'VV currently accepts finite free quasi-coherent modules on affine schemes'
        )
    return module.relative_spec(names=names)


def _affine_scheme_function_field(self):
    coordinate_ring = self.coordinate_ring()
    if not coordinate_ring.is_integral_domain():
        raise ValueError('an affine scheme has a generic point only when integral')
    return coordinate_ring.fraction_field()


def _affine_scheme_generic_point(self):
    coordinate_ring = self.coordinate_ring()
    function_field = self.function_field()
    inclusion = coordinate_ring.hom(
        [function_field(generator) for generator in coordinate_ring.gens()],
        function_field,
    )
    generic_domain = Spec(function_field, self.base_scheme())
    generic_domain._projective_framework_generic_point_of = self
    generic_domain._projective_framework_latex_name = (
        r'\operatorname{Spec}K'
    )
    generic_point = generic_domain.hom(inclusion, self)
    generic_point._projective_framework_morphism_role = (
        'generic_point'
    )
    generic_domain.base_morphism()._projective_framework_morphism_role = (
        'generic_base_change'
    )
    return generic_point


def _affine_scheme_vv_module(self):
    try:
        return self._vv_module
    except AttributeError as error:
        raise ValueError('this affine scheme was not constructed by VV') from error


def _affine_scheme_vv_projection(self):
    self.vv_module()
    return self.base_morphism()


def _global_sections_as_quasicoherent_module(self, base_scheme=None):
    if base_scheme is None:
        base_scheme = self.scheme().base_scheme()
    if base_scheme.coordinate_ring() != self.base_ring():
        raise ValueError(
            'the affine base scheme must have coordinate ring equal to the section-space base ring'
        )
    attribute = '_projective_framework_qcoherent_module_cache'
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = id(base_scheme)
    if key not in cache:
        basis_names = tuple(f's_{index}' for index in range(self.dimension()))
        dual_basis_names = tuple(f'c_{index}' for index in range(self.dimension()))
        cache[key] = AffineQuasiCoherentFreeModule(
            base_scheme,
            basis_names,
            dual_basis_names=dual_basis_names,
            source_sections=self,
            ground_ring=self.base_ring(),
        )
    return cache[key]


def _global_sections_affine_space_via_vv(self, names=None, prefix='c'):
    pushforward_module = self.as_quasicoherent_module()
    dual_module = pushforward_module.dual()
    if names is None:
        names = tuple(f'{prefix}_{index}' for index in range(self.dimension()))
    return VV(dual_module, names=names)


def _structure_map_pushforward(self, sheaf):
    if not isinstance(sheaf, ProductProjectiveLineBundle):
        raise NotImplementedError(
            'pushforward is currently implemented only for supported line bundles'
        )
    if sheaf.scheme() != self.domain():
        raise ValueError('the line bundle must live on the domain of the morphism')
    if not isinstance(self.codomain(), AffineScheme):
        raise NotImplementedError('the current pushforward target must be affine')
    if self.codomain().coordinate_ring() != sheaf.scheme().base_ring():
        raise NotImplementedError(
            'the current structure-map pushforward is implemented over an affine point'
        )
    return sheaf.H(0).as_quasicoherent_module(self.codomain())


def _install_relative_spec_interface():
    AffineScheme.function_field = _affine_scheme_function_field
    AffineScheme.generic_point = _affine_scheme_generic_point
    AffineScheme.vv_module = _affine_scheme_vv_module
    AffineScheme.projection = _affine_scheme_vv_projection
    if 'relative_base_scheme' in AffineScheme.__dict__:
        delattr(AffineScheme, 'relative_base_scheme')
    ProductProjectiveGlobalSections.as_quasicoherent_module = (
        _global_sections_as_quasicoherent_module
    )
    ProductProjectiveGlobalSections.affine_space = (
        _global_sections_affine_space_via_vv
    )
    SchemeMorphism_structure_map.pushforward = _structure_map_pushforward


_install_relative_spec_interface()

print('Installed affine quasi-coherent modules, VV, generic points, and structure-map pushforward.')

Installed affine quasi-coherent modules, VV, generic points, and structure-map pushforward.


## Base change along an explicit morphism

For an $S$-scheme $X$ and a morphism $q:T\to S$, the base change is the fiber product

$$
X_T=X\times_S T.
$$

The public method `X.base_change(q)` therefore requires the scheme morphism $q$; it does not accept a bare ring or rely on a selected natural coercion. On the supported affine and projective coordinate models, Sage's ring-change backend is applied only after verifying the induced ring homomorphism

$$
\mathcal O(S)\longrightarrow\mathcal O(T).
$$

The returned scheme records $T$ as its actual base scheme. Line bundles and global-section spaces derive their base-change operations from this scheme-level construction.

In [5]:
def _validate_base_change_morphism(scheme, base_morphism):
    if not (
        hasattr(base_morphism, 'domain')
        and hasattr(base_morphism, 'codomain')
    ):
        raise TypeError('base_change requires a morphism of affine schemes')
    if base_morphism.codomain() != scheme.base_scheme():
        raise ValueError(
            'the codomain of the base-change morphism must equal the base scheme'
        )
    if not isinstance(base_morphism.domain(), AffineScheme):
        raise NotImplementedError(
            'the current coordinate backend requires an affine base-change source'
        )

    if hasattr(base_morphism, 'ring_homomorphism'):
        ring_map = base_morphism.ring_homomorphism()
    else:
        from sage.schemes.generic.morphism import (
            SchemeMorphism_structure_map,
        )
        if not isinstance(
            base_morphism,
            SchemeMorphism_structure_map,
        ):
            raise TypeError(
                'a non-structure base-change morphism must expose its ring homomorphism'
            )
        source_ring = base_morphism.codomain().coordinate_ring()
        target_ring = base_morphism.domain().coordinate_ring()
        ring_map = target_ring.coerce_map_from(source_ring)
        if ring_map is None:
            raise ValueError(
                'the affine structure map does not induce a canonical scalar homomorphism'
            )

    if ring_map.domain() != scheme.base_ring():
        raise ValueError(
            'the ring homomorphism of the base-change morphism has the wrong source'
        )
    if ring_map.codomain() != base_morphism.domain().coordinate_ring():
        raise ValueError(
            'the ring homomorphism of the base-change morphism has the wrong target'
        )
    return ring_map


def _scheme_base_change(self, base_morphism):
    ring_map = _validate_base_change_morphism(self, base_morphism)
    attribute = '_projective_framework_base_change_cache'
    cache = getattr(self, attribute, None)
    if cache is None:
        cache = {}
        setattr(self, attribute, cache)
    key = id(base_morphism)
    if key not in cache:
        changed_scheme = self.change_ring(ring_map)
        changed_scheme._base_ring = (
            base_morphism.domain().coordinate_ring()
        )
        changed_scheme._base_scheme = base_morphism.domain()
        if hasattr(changed_scheme, '_base_morphism'):
            delattr(changed_scheme, '_base_morphism')
        assert (
            changed_scheme.base_morphism().codomain()
            == base_morphism.domain()
        )
        cache[key] = changed_scheme
    return cache[key]


def _install_explicit_base_change_interface():
    scheme_classes = (
        AffineSpace_generic,
        AlgebraicScheme_subscheme_affine,
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
        AlgebraicScheme_subscheme_projective,
        AlgebraicScheme_subscheme_product_projective,
    )
    for scheme_class in scheme_classes:
        scheme_class.base_change = _scheme_base_change


_install_explicit_base_change_interface()

print('Installed morphism-based base change on supported affine and projective schemes.')

Installed morphism-based base change on supported affine and projective schemes.


## Functorial pullback on Picard groups and global sections

For a morphism $f:X\to Y$, pullback of invertible sheaves gives a homomorphism

$$
f^*:\operatorname{Pic}(Y)\longrightarrow\operatorname{Pic}(X).
$$

For $L\in\operatorname{Pic}(Y)$, functoriality of global sections gives the induced linear map

$$
H^0(Y,L)\longrightarrow H^0(X,f^*L).
$$

The next cell constructs these maps on their ambient parents. Thus `Y.Pic().pullback(f)` is a group homomorphism, `L.pullback(f)` is its value at $L$, and `L.H(0).pullback(f)` is a linear map of section spaces. No element-level `f.pullback_section(s)` method is installed.

Pullback on Weil divisor class groups is not asserted for an arbitrary morphism; such functoriality requires additional hypotheses. The implementation here concerns the unconditional Picard/invertible-sheaf pullback on the supported projective products.

In [31]:
def _multidegree_of_product_homogeneous_polynomial(polynomial, source_scheme):
    coordinate_ring = source_scheme.coordinate_ring()
    polynomial = coordinate_ring(polynomial)
    if polynomial == 0:
        return None

    coordinate_blocks = _coordinate_blocks(
        source_scheme,
        coordinate_ring.gens(),
    )
    block_sizes = tuple(len(block) for block in coordinate_blocks)
    degrees = set()
    for exponent, coefficient in polynomial.dict().items():
        if coefficient == 0:
            continue
        degree = []
        start = 0
        for block_size in block_sizes:
            stop = start + block_size
            degree.append(sum(exponent[start:stop]))
            start = stop
        degrees.add(tuple(ZZ(value) for value in degree))

    if len(degrees) != 1:
        raise ValueError(
            f'{polynomial} is not multihomogeneous on {source_scheme}'
        )
    return degrees.pop()


def _picard_generator_pullback_multidegrees(morphism):
    source = morphism.domain()
    target = morphism.codomain()
    if source != source.ambient_space() or target != target.ambient_space():
        raise NotImplementedError(
            'Picard pullback is currently implemented only for ambient projective products'
        )
    _require_supported_projective_ambient(source, 'Picard pullback')
    _require_supported_projective_ambient(target, 'Picard pullback')

    target_coordinate_blocks = _coordinate_blocks(
        target,
        tuple(morphism.defining_polynomials()),
    )
    generator_images = []
    for target_block in target_coordinate_blocks:
        nonzero_degrees = {
            _multidegree_of_product_homogeneous_polynomial(
                coordinate,
                source,
            )
            for coordinate in target_block
            if coordinate != 0
        }
        if len(nonzero_degrees) != 1:
            raise ValueError(
                'the coordinate block of a projective factor must have one common multidegree'
            )
        generator_images.append(nonzero_degrees.pop())
    return tuple(generator_images)


def _picard_pullback(self, morphism):
    if morphism.codomain() != self.scheme():
        raise ValueError(
            'the codomain of the morphism must equal the scheme of the Picard group'
        )
    source_picard_group = morphism.domain().Pic()
    generator_multidegrees = _picard_generator_pullback_multidegrees(
        morphism
    )
    generator_images = tuple(
        source_picard_group(multidegree)
        for multidegree in generator_multidegrees
    )

    def pullback_class(divisor_class):
        divisor_class = self(divisor_class)
        result = source_picard_group.zero()
        for coefficient, generator_image in zip(
            divisor_class.multidegree(),
            generator_images,
        ):
            result += coefficient * generator_image
        return result

    return self.module_morphism(
        function=pullback_class,
        codomain=source_picard_group,
    )


def _line_bundle_pullback(self, morphism):
    return self.parent().pullback(morphism)(self)


def _global_sections_pullback(self, morphism):
    line_bundle = self.line_bundle()
    if morphism.codomain() != line_bundle.scheme():
        raise ValueError(
            'the codomain of the morphism must equal the scheme carrying the line bundle'
        )

    pulled_line_bundle = line_bundle.pullback(morphism)
    target_sections = pulled_line_bundle.H(0)
    source_coordinate_ring = morphism.domain().coordinate_ring()
    target_coordinate_ring = morphism.codomain().coordinate_ring()
    coordinate_pullback = target_coordinate_ring.hom(
        tuple(morphism.defining_polynomials()),
        source_coordinate_ring,
    )

    def pullback_on_basis(basis_index):
        section = self.basis()[basis_index]
        pulled_polynomial = coordinate_pullback(
            section.to_polynomial()
        )
        return target_sections.from_polynomial(pulled_polynomial)

    return self.module_morphism(
        on_basis=pullback_on_basis,
        codomain=target_sections,
    )


def _install_functorial_pullback_interface():
    ProductProjectivePicardGroup.pullback = _picard_pullback
    ProductProjectiveLineBundle.pullback = _line_bundle_pullback
    ProductProjectiveGlobalSections.pullback = _global_sections_pullback

    for morphism_class in _projective_morphism_classes:
        for obsolete_name in (
            'pullback_line_bundle',
            'pullback_section',
        ):
            if obsolete_name in morphism_class.__dict__:
                delattr(morphism_class, obsolete_name)


_install_functorial_pullback_interface()

print('Installed functorial pullback on Picard groups, line bundles, and global sections.')

NameError: name '_projective_morphism_classes' is not defined

## Restriction of line bundles and sections to finite reduced subschemes

Let $i:Z\to X$ be a named morphism from a finite reduced $k$-scheme whose geometric points are all $k$-rational. For a line bundle $L$ on $X$, pullback gives $i^*L$ on $Z$ and a linear map

$$
i^*:H^0(X,L)\longrightarrow H^0(Z,i^*L).
$$

The target is represented as the direct sum of the one-dimensional fibers over the rational points of $Z$. Each fiber basis records the standard affine chart used to trivialize $L$; therefore the matrix is an explicit coordinate realization of the intrinsic restriction map, not the primary object.

In [ ]:
def _finite_reduced_rational_subscheme_data(scheme):
    if not hasattr(scheme, 'dimension') or scheme.dimension() != 0:
        raise ValueError(
            'the current restriction backend requires a zero-dimensional scheme'
        )
    if not scheme.base_ring().is_field():
        raise NotImplementedError(
            'the current restriction backend requires a field base'
        )
    if not scheme.base_ring().is_exact():
        raise NotImplementedError(
            'the current restriction backend requires an exact base field'
        )
    if not hasattr(scheme, 'defining_ideal'):
        raise NotImplementedError(
            'the finite scheme must have an explicit defining ideal'
        )

    defining_ideal = scheme.defining_ideal()
    if defining_ideal.radical() != defining_ideal:
        raise NotImplementedError(
            'the current restriction backend requires a reduced finite scheme'
        )

    points = tuple(scheme.rational_points())
    if ZZ(scheme.degree()) != len(points):
        raise NotImplementedError(
            'the current restriction backend requires every geometric point to be rational'
        )
    return points


class FiniteReducedSubschemeLineBundle(SageObject):
    def __init__(self, source_line_bundle, base_morphism):
        if base_morphism.codomain() != source_line_bundle.scheme():
            raise ValueError(
                'the morphism codomain must carry the source line bundle'
            )

        self._source_line_bundle = source_line_bundle
        self._base_morphism = base_morphism
        self._scheme = base_morphism.domain()
        self._points = _finite_reduced_rational_subscheme_data(
            self._scheme
        )
        self._ambient_points = tuple(
            base_morphism(point)
            for point in self._points
        )
        self._cover = source_line_bundle.scheme().affine_cover()
        self._charts = tuple(
            self._cover.canonical_chart(point)
            for point in self._ambient_points
        )
        self._chart_points = tuple(
            chart.preimage_point(point)
            for chart, point in zip(
                self._charts,
                self._ambient_points,
            )
        )
        self._H0_cache = None

    def _repr_(self):
        return (
            f'Pullback of {self._source_line_bundle} '
            f'along {self._base_morphism}'
        )

    def _cache_key(self):
        return (
            self._source_line_bundle,
            self._base_morphism,
        )

    def __hash__(self):
        return hash(self._cache_key())

    def scheme(self):
        return self._scheme

    def base_ring(self):
        return self._scheme.base_ring()

    def source_line_bundle(self):
        return self._source_line_bundle

    def base_morphism(self):
        return self._base_morphism

    def points(self):
        return self._points

    def ambient_points(self):
        return self._ambient_points

    def trivializing_cover(self):
        return self._cover

    def trivializing_charts(self):
        return self._charts

    def chart_points(self):
        return self._chart_points

    def trivialization_labels(self):
        return tuple(
            chart.cover_label()
            for chart in self._charts
        )

    def H(self, degree):
        degree = ZZ(degree)
        if degree != 0:
            raise NotImplementedError(
                'the current finite restriction backend constructs only H^0'
            )
        if self._H0_cache is None:
            self._H0_cache = FiniteReducedSubschemeGlobalSections(
                self
            )
        return self._H0_cache


class FiniteReducedSubschemeGlobalSections(CombinatorialFreeModule):
    def __init__(self, restriction):
        self._restriction = restriction
        self._points = restriction.points()
        CombinatorialFreeModule.__init__(
            self,
            restriction.base_ring(),
            tuple(range(len(self._points))),
            prefix='e',
        )

    def _repr_(self):
        return (
            f'H^0({self._restriction.scheme()}, '
            f'{self._restriction})'
        )

    def restriction(self):
        return self._restriction

    def scheme(self):
        return self._restriction.scheme()

    def points(self):
        return self._points

    def fiber_basis(self):
        return self.basis()

    def trivialization_labels(self):
        return self._restriction.trivialization_labels()


class SectionRestrictionMorphism(SageObject):
    def __init__(self, underlying_morphism, restriction):
        self._underlying_morphism = underlying_morphism
        self._restriction = restriction
        self._matrix_cache = None
        self._kernel_cache = None
        self._image_cache = None
        self._cokernel_cache = None

    def _repr_(self):
        return (
            f'Restriction morphism from {self.domain()} '
            f'to {self.codomain()}'
        )

    def domain(self):
        return self._underlying_morphism.domain()

    def codomain(self):
        return self._underlying_morphism.codomain()

    def __call__(self, section):
        return self._underlying_morphism(section)

    def underlying_morphism(self):
        return self._underlying_morphism

    def restriction(self):
        return self._restriction

    def base_morphism(self):
        return self._restriction.base_morphism()

    def matrix(self):
        if self._matrix_cache is None:
            self._matrix_cache = self._underlying_morphism.matrix()
        return self._matrix_cache

    def rank(self):
        return ZZ(self.matrix().rank())

    def kernel(self):
        if self._kernel_cache is None:
            generators = tuple(
                self.domain().from_vector(vector)
                for vector in self.matrix().right_kernel().basis()
            )
            self._kernel_cache = self.domain().submodule(
                generators
            )
        return self._kernel_cache

    def image(self):
        if self._image_cache is None:
            generators = tuple(
                self.codomain().from_vector(vector)
                for vector in self.matrix().column_space().basis()
            )
            self._image_cache = self.codomain().submodule(
                generators
            )
        return self._image_cache

    def cokernel(self):
        if self._cokernel_cache is None:
            self._cokernel_cache = self.codomain().quotient(
                self.image()
            )
        return self._cokernel_cache

    def trivialization_labels(self):
        return self._restriction.trivialization_labels()


def _finite_section_restriction_morphism(section_space, base_morphism):
    restriction = section_space.line_bundle().pullback(
        base_morphism
    )
    if not isinstance(
        restriction,
        FiniteReducedSubschemeLineBundle,
    ):
        raise TypeError(
            'a finite restriction morphism requires a finite restricted line bundle'
        )
    target_sections = restriction.H(0)
    source_coordinate_ring = section_space.scheme().coordinate_ring()

    evaluation_data = []
    for chart, chart_point in zip(
        restriction.trivializing_charts(),
        restriction.chart_points(),
    ):
        chart_ring = chart.domain().coordinate_ring()
        coordinate_pullback = source_coordinate_ring.hom(
            tuple(chart.defining_polynomials()),
            chart_ring,
        )
        evaluation_data.append(
            (coordinate_pullback, tuple(chart_point))
        )

    def pullback_on_basis(basis_index):
        polynomial = section_space.basis()[
            basis_index
        ].to_polynomial()
        coefficients = {}
        for point_index, (
            coordinate_pullback,
            chart_coordinates,
        ) in enumerate(evaluation_data):
            value = coordinate_pullback(polynomial)(
                *chart_coordinates
            )
            if value != 0:
                coefficients[point_index] = value
        return target_sections._from_dict(
            coefficients,
            remove_zeros=True,
        )

    underlying = section_space.module_morphism(
        on_basis=pullback_on_basis,
        codomain=target_sections,
    )
    return SectionRestrictionMorphism(
        underlying,
        restriction,
    )


_ambient_line_bundle_pullback = ProductProjectiveLineBundle.pullback
_ambient_global_sections_pullback = ProductProjectiveGlobalSections.pullback


def _line_bundle_pullback_with_finite_restriction(self, morphism):
    if morphism.codomain() != self.scheme():
        raise ValueError(
            'the morphism codomain must equal the scheme carrying the line bundle'
        )
    if hasattr(morphism.domain(), 'dimension') and morphism.domain().dimension() == 0:
        attribute = '_projective_framework_finite_restriction_cache'
        cache = getattr(self, attribute, None)
        if cache is None:
            cache = {}
            setattr(self, attribute, cache)
        key = id(morphism)
        if key not in cache:
            cache[key] = FiniteReducedSubschemeLineBundle(
                self,
                morphism,
            )
        return cache[key]
    return _ambient_line_bundle_pullback(self, morphism)


def _global_sections_pullback_with_finite_restriction(self, morphism):
    if morphism.codomain() != self.scheme():
        raise ValueError(
            'the morphism codomain must equal the scheme carrying the line bundle'
        )
    if hasattr(morphism.domain(), 'dimension') and morphism.domain().dimension() == 0:
        attribute = '_projective_framework_finite_restriction_morphism_cache'
        cache = getattr(self, attribute, None)
        if cache is None:
            cache = {}
            setattr(self, attribute, cache)
        key = id(morphism)
        if key not in cache:
            cache[key] = _finite_section_restriction_morphism(
                self,
                morphism,
            )
        return cache[key]
    return _ambient_global_sections_pullback(self, morphism)


ProductProjectiveLineBundle.pullback = (
    _line_bundle_pullback_with_finite_restriction
)
ProductProjectiveGlobalSections.pullback = (
    _global_sections_pullback_with_finite_restriction
)

print('Installed finite reduced restriction of line bundles and sections.')

## Local germs and isolated hypersurface singularities at points

For a rational point $p\in X$, `p.local_germ()` chooses a standard affine chart containing $p$, pulls back the defining ideal of $X$, and translates $p$ to the origin. The resulting object records an affine presentation of the local germ, its maximal ideal, tangent space, embedding dimension, multiplicity, and tangent-cone equation.

For a hypersurface germ, the point methods `local_equation()`, `Milnor_algebra()`, `Tjurina_algebra()`, and their dimensions are obtained from the primary components supported at the chosen point. In characteristic zero, isolated plane-curve germs are classified when the computed invariants certify one of the simple types $A_n$, $D_n$, $E_6$, $E_7$, or $E_8$. The certificate uses Hessian rank, equality $\mu=\tau$, multiplicity, the cubic tangent cone, and the Milnor number; unsupported higher-modality cases raise `NotImplementedError`.

In [ ]:
import sage.schemes.generic.morphism as _generic_point_morphism_module


class LocalSchemeGerm(SageObject):
    def __init__(self, point):
        self._point = point
        self._scheme = point.codomain()
        self._ambient = self._scheme.ambient_space()
        if not _supported_projective_ambient(self._scheme):
            raise NotImplementedError(
                'local germs are currently implemented only for supported projective schemes'
            )
        if not self._scheme.base_ring().is_field():
            raise NotImplementedError(
                'the current local-germ backend requires a field base'
            )

        if self._scheme == self._ambient:
            self._ambient_point = point
        else:
            ambient_factors = _projective_factors(
                self._ambient
            )
            if len(ambient_factors) == 1:
                ambient_coordinates = tuple(point)
            else:
                ambient_coordinates = tuple(
                    coordinate
                    for factor_index in range(
                        len(ambient_factors)
                    )
                    for coordinate in tuple(
                        point[factor_index]
                    )
                )
            self._ambient_point = self._ambient(
                ambient_coordinates
            )

        self._cover = self._ambient.affine_cover()
        self._chart = self._cover.canonical_chart(
            self._ambient_point
        )
        self._chart_point = self._chart.preimage_point(
            self._ambient_point
        )
        self._chart_ring = self._chart.domain().coordinate_ring()

        local_names = tuple(
            f't_{index}'
            for index in range(self._chart_ring.ngens())
        )
        self._presentation_ring = PolynomialRing(
            self._scheme.base_ring(),
            names=local_names,
        )
        translated_coordinates = tuple(
            self._presentation_ring.gen(index)
            + self._scheme.base_ring()(self._chart_point[index])
            for index in range(self._chart_ring.ngens())
        )
        translation = self._chart_ring.hom(
            translated_coordinates,
            self._presentation_ring,
        )
        ambient_coordinate_ring = self._ambient.coordinate_ring()
        chart_pullback = ambient_coordinate_ring.hom(
            tuple(self._chart.defining_polynomials()),
            self._chart_ring,
        )
        self._equations = tuple(
            translation(chart_pullback(equation))
            for equation in _defining_equations(self._scheme)
        )
        self._defining_ideal = self._presentation_ring.ideal(
            self._equations
        )
        self._maximal_ideal = self._presentation_ring.ideal(
            self._presentation_ring.gens()
        )
        self._hypersurface_presentation_cache = None
        self._local_equation_cache = None
        self._multiplicity_cache = None
        self._tangent_cone_cache = None
        self._milnor_ideal_cache = None
        self._tjurina_ideal_cache = None
        self._local_milnor_ideal_cache = None
        self._local_tjurina_ideal_cache = None

    def _repr_(self):
        return f'Local germ of {self._scheme} at {self._point}'

    def point(self):
        return self._point

    def scheme(self):
        return self._scheme

    def ambient_space(self):
        return self._ambient

    def chart(self):
        return self._chart

    def chart_point(self):
        return self._chart_point

    def presentation_ring(self):
        return self._presentation_ring

    def equations(self):
        return self._equations

    def defining_ideal(self):
        return self._defining_ideal

    def maximal_ideal(self):
        return self._maximal_ideal

    def jacobian_matrix_at_origin(self):
        variables = self._presentation_ring.gens()
        zero = {variable: 0 for variable in variables}
        if len(self._equations) == 0:
            return matrix(
                self._scheme.base_ring(),
                0,
                len(variables),
            )
        return matrix(
            self._scheme.base_ring(),
            [
                [
                    self._scheme.base_ring()(
                        equation.derivative(variable).subs(zero)
                    )
                    for variable in variables
                ]
                for equation in self._equations
            ],
        )

    def tangent_space(self):
        return self.jacobian_matrix_at_origin().right_kernel()

    def embedding_dimension(self):
        return ZZ(self.tangent_space().dimension())

    def local_dimension(self):
        return ZZ(self._scheme.dimension())

    def is_singular(self):
        return self.embedding_dimension() > self.local_dimension()

    def hypersurface_presentation(self):
        return _germ_hypersurface_presentation(self)

    def local_equation(self):
        if self._local_equation_cache is None:
            self._local_equation_cache = (
                self.hypersurface_presentation().equation()
            )
        return self._local_equation_cache

    def multiplicity(self):
        if self._multiplicity_cache is None:
            equation = self.local_equation()
            if equation == 0:
                raise NotImplementedError(
                    'multiplicity of the zero local equation is undefined'
                )
            self._multiplicity_cache = ZZ(
                min(
                    sum(monomial)
                    for monomial in equation.dict()
                )
            )
        return self._multiplicity_cache

    def tangent_cone_equation(self):
        if self._tangent_cone_cache is None:
            equation = self.local_equation()
            multiplicity = self.multiplicity()
            variables = self._presentation_ring.gens()
            self._tangent_cone_cache = self._presentation_ring(
                sum(
                    coefficient
                    * prod(
                        variable**exponent
                        for variable, exponent
                        in zip(variables, monomial)
                    )
                    for monomial, coefficient
                    in equation.dict().items()
                    if sum(monomial) == multiplicity
                )
            )
        return self._tangent_cone_cache

    tangent_cone = tangent_cone_equation

    def hessian_at_origin(self):
        equation = self.local_equation()
        variables = equation.parent().gens()
        zero = {variable: 0 for variable in variables}
        return matrix(
            self._scheme.base_ring(),
            [
                [
                    self._scheme.base_ring()(
                        equation.derivative(left).derivative(right).subs(zero)
                    )
                    for right in variables
                ]
                for left in variables
            ],
        )

    def milnor_ideal(self):
        if self._milnor_ideal_cache is None:
            equation = self.local_equation()
            hypersurface_ring = equation.parent()
            self._milnor_ideal_cache = hypersurface_ring.ideal(
                [
                    equation.derivative(variable)
                    for variable in hypersurface_ring.gens()
                ]
            )
        return self._milnor_ideal_cache

    def tjurina_ideal(self):
        if self._tjurina_ideal_cache is None:
            equation = self.local_equation()
            hypersurface_ring = equation.parent()
            self._tjurina_ideal_cache = hypersurface_ring.ideal(
                [equation]
                + [
                    equation.derivative(variable)
                    for variable in hypersurface_ring.gens()
                ]
            )
        return self._tjurina_ideal_cache

    def _local_primary_component(self, ideal):
        if ideal.dimension() != 0:
            raise NotImplementedError(
                'the singularity is not isolated in the current affine presentation'
            )
        local_maximal_ideal = ideal.ring().ideal(
            ideal.ring().gens()
        )
        components = tuple(
            component
            for component in ideal.primary_decomposition()
            if component.radical() == local_maximal_ideal
        )
        if len(components) != 1:
            raise NotImplementedError(
                'could not isolate a unique primary component supported at the point'
            )
        return components[0]

    def local_milnor_ideal(self):
        if self._local_milnor_ideal_cache is None:
            self._local_milnor_ideal_cache = (
                self._local_primary_component(
                    self.milnor_ideal()
                )
            )
        return self._local_milnor_ideal_cache

    def local_tjurina_ideal(self):
        if self._local_tjurina_ideal_cache is None:
            self._local_tjurina_ideal_cache = (
                self._local_primary_component(
                    self.tjurina_ideal()
                )
            )
        return self._local_tjurina_ideal_cache

    def is_isolated_hypersurface_singularity(self):
        try:
            self.local_milnor_ideal()
        except NotImplementedError:
            return False
        return self.is_singular()

    def milnor_algebra(self):
        local_ideal = self.local_milnor_ideal()
        return local_ideal.ring().quotient(
            local_ideal
        )

    def tjurina_algebra(self):
        local_ideal = self.local_tjurina_ideal()
        return local_ideal.ring().quotient(
            local_ideal
        )

    def milnor_number(self):
        return ZZ(
            self.local_milnor_ideal().vector_space_dimension()
        )

    def tjurina_number(self):
        return ZZ(
            self.local_tjurina_ideal().vector_space_dimension()
        )

    def ADE_type(self):
        if not self.is_singular():
            return None
        if self._scheme.base_ring().characteristic() != 0:
            raise NotImplementedError(
                'ADE classification is currently implemented only in characteristic zero'
            )
        hypersurface_ring = self.local_equation().parent()
        variable_count = hypersurface_ring.ngens()
        if variable_count not in (2, 3):
            raise NotImplementedError(
                'the current ADE classifier supports plane curves and hypersurface surfaces'
            )
        if not self.is_isolated_hypersurface_singularity():
            raise NotImplementedError(
                'the current ADE classifier requires an isolated hypersurface singularity'
            )

        hessian_rank = self.hessian_at_origin().rank()
        milnor_number = self.milnor_number()
        tjurina_number = self.tjurina_number()
        if milnor_number != tjurina_number:
            raise NotImplementedError(
                'the current ADE certificate requires equality of the Milnor and Tjurina numbers'
            )

        if variable_count == 3:
            if (
                self.local_dimension() == 2
                and hessian_rank == 3
                and milnor_number == 1
            ):
                return 'A1'
            raise NotImplementedError(
                'the current surface ADE classifier certifies only nondegenerate A1 hypersurface points'
            )

        if hessian_rank == 2 and milnor_number == 1:
            return 'A1'
        if hessian_rank == 1:
            return f'A{milnor_number}'
        if hessian_rank != 0:
            raise NotImplementedError(
                'the Hessian rank does not match a supported ADE case'
            )
        if self.multiplicity() != 3:
            raise NotImplementedError(
                'higher-corank ADE classification currently requires multiplicity three'
            )

        tangent_cone = self.tangent_cone_equation()
        variables = hypersurface_ring.gens()
        tangent_derivatives = tuple(
            tangent_cone.derivative(variable)
            for variable in variables
        )
        common_derivative_factor = tangent_derivatives[0].gcd(
            tangent_derivatives[1]
        )
        common_factor_degree = ZZ(
            common_derivative_factor.total_degree()
        )

        if common_factor_degree == 0 and milnor_number == 4:
            return 'D4'
        if common_factor_degree == 1 and milnor_number >= 5:
            return f'D{milnor_number}'
        if common_factor_degree == 2 and milnor_number in (6, 7, 8):
            return f'E{milnor_number}'
        raise NotImplementedError(
            'the multiplicity, tangent cone, and Milnor number do not certify a supported simple ADE type'
        )

    def normal_form_equation(self):
        singularity_type = self.ADE_type()
        if singularity_type is None:
            return None
        family = singularity_type[0]
        index = ZZ(singularity_type[1:])
        variable_count = self.local_equation().parent().ngens()
        if variable_count == 3:
            if singularity_type != 'A1':
                raise NotImplementedError
            normal_form_ring = PolynomialRing(
                self._scheme.base_ring(),
                names=('u', 'v', 'w'),
            )
            u, v, w = normal_form_ring.gens()
            return u**2 + v**2 + w**2

        normal_form_ring = PolynomialRing(
            self._scheme.base_ring(),
            names=('u', 'v'),
        )
        u, v = normal_form_ring.gens()
        if family == 'A':
            return u**2 + v**(index + 1)
        if family == 'D':
            return u**2 * v + v**(index - 1)
        if singularity_type == 'E6':
            return u**3 + v**4
        if singularity_type == 'E7':
            return u**3 + u * v**3
        if singularity_type == 'E8':
            return u**3 + v**5
        raise NotImplementedError

    equation_of_normal_form = normal_form_equation


class LocalHypersurfacePresentation(SageObject):
    def __init__(self, germ):
        if not isinstance(germ, LocalSchemeGerm):
            raise TypeError(
                'a hypersurface presentation must be constructed from a LocalSchemeGerm'
            )
        if germ.embedding_dimension() != germ.local_dimension() + 1:
            raise NotImplementedError(
                'the local germ does not have hypersurface embedding dimension'
            )

        self._germ = germ
        current_ring = germ.presentation_ring()
        current_ideal = current_ring.ideal(
            germ.equations()
        )
        eliminated_variables = []
        target_variable_count = germ.embedding_dimension()

        while current_ring.ngens() > target_variable_count:
            equations = tuple(
                equation
                for equation in current_ideal.groebner_basis()
                if equation != 0
            )
            pivot = None
            for equation_index, equation in enumerate(equations):
                for variable_index, variable in enumerate(
                    current_ring.gens()
                ):
                    derivative = equation.derivative(variable)
                    if (
                        derivative in current_ring.base_ring()
                        and derivative != 0
                    ):
                        pivot = (
                            equation_index,
                            variable_index,
                            equation,
                            variable,
                            current_ring.base_ring()(derivative),
                        )
                        break
                if pivot is not None:
                    break

            if pivot is None:
                raise NotImplementedError(
                    'no exact affine-linear variable elimination is available'
                )

            (
                equation_index,
                variable_index,
                equation,
                variable,
                coefficient,
            ) = pivot
            remainder = equation - coefficient * variable
            if remainder.degree(variable) > 0:
                raise ArithmeticError(
                    'the chosen pivot equation is not affine-linear'
                )

            remaining_variables = tuple(
                generator
                for index, generator in enumerate(
                    current_ring.gens()
                )
                if index != variable_index
            )
            new_ring = PolynomialRing(
                current_ring.base_ring(),
                names=tuple(
                    str(generator)
                    for generator in remaining_variables
                ),
            )
            remaining_to_new = {
                generator: new_ring.gen(index)
                for index, generator in enumerate(
                    remaining_variables
                )
            }
            zero_substitution = current_ring.hom(
                tuple(
                    new_ring.zero()
                    if index == variable_index
                    else remaining_to_new[generator]
                    for index, generator in enumerate(
                        current_ring.gens()
                    )
                ),
                new_ring,
            )
            solution = (
                -zero_substitution(remainder) / coefficient
            )
            substitution = current_ring.hom(
                tuple(
                    solution
                    if index == variable_index
                    else remaining_to_new[generator]
                    for index, generator in enumerate(
                        current_ring.gens()
                    )
                ),
                new_ring,
            )
            new_equations = tuple(
                substitution(other_equation)
                for index, other_equation in enumerate(equations)
                if index != equation_index
            )
            current_ring = new_ring
            current_ideal = current_ring.ideal(
                new_equations
            )
            eliminated_variables.append(
                (str(variable), solution)
            )

        final_generators = tuple(
            generator
            for generator in current_ideal.groebner_basis()
            if generator != 0
        )
        if len(final_generators) != 1:
            raise NotImplementedError(
                'the reduced local presentation is not a hypersurface'
            )
        if current_ring.ngens() != germ.local_dimension() + 1:
            raise NotImplementedError(
                'the reduced presentation does not have hypersurface embedding dimension'
            )

        self._ring = current_ring
        self._equation = final_generators[0]
        self._eliminated_variables = tuple(
            eliminated_variables
        )
        self._maximal_ideal = current_ring.ideal(
            current_ring.gens()
        )

    def _repr_(self):
        return (
            f'Local hypersurface presentation '
            f'{self._equation} = 0 in {self._ring}'
        )

    def germ(self):
        return self._germ

    def ring(self):
        return self._ring

    def equation(self):
        return self._equation

    def eliminated_variables(self):
        return self._eliminated_variables

    def maximal_ideal(self):
        return self._maximal_ideal

    def dimension(self):
        return ZZ(self._ring.ngens() - 1)

    def embedding_dimension(self):
        return ZZ(self._ring.ngens())


def _germ_hypersurface_presentation(self):
    if self._hypersurface_presentation_cache is None:
        self._hypersurface_presentation_cache = (
            LocalHypersurfacePresentation(self)
        )
    return self._hypersurface_presentation_cache


def _point_local_germ(self):
    germ = getattr(
        self,
        '_projective_framework_local_germ',
        None,
    )
    if germ is None:
        germ = LocalSchemeGerm(self)
        try:
            self._projective_framework_local_germ = germ
        except AttributeError:
            pass
    return germ


def _point_is_singular(self):
    return self.local_germ().is_singular()


def _point_hypersurface_presentation(self):
    return self.local_germ().hypersurface_presentation()


def _point_local_equation(self):
    return self.local_germ().local_equation()


def _point_multiplicity(self):
    return self.local_germ().multiplicity()


def _point_tangent_cone_equation(self):
    return self.local_germ().tangent_cone_equation()


def _point_embedding_dimension(self):
    return self.local_germ().embedding_dimension()


def _point_milnor_algebra(self):
    return self.local_germ().milnor_algebra()


def _point_tjurina_algebra(self):
    return self.local_germ().tjurina_algebra()


def _point_milnor_number(self):
    return self.local_germ().milnor_number()


def _point_tjurina_number(self):
    return self.local_germ().tjurina_number()


def _point_ADE_type(self):
    return self.local_germ().ADE_type()


def _point_equation_of_normal_form(self):
    return self.local_germ().normal_form_equation()


PointMorphismClass = _generic_point_morphism_module.SchemeMorphism_point
PointMorphismClass.local_germ = _point_local_germ
PointMorphismClass.is_singular = _point_is_singular
PointMorphismClass.hypersurface_presentation = (
    _point_hypersurface_presentation
)
PointMorphismClass.local_equation = _point_local_equation
PointMorphismClass.multiplicity = _point_multiplicity
PointMorphismClass.tangent_cone_equation = (
    _point_tangent_cone_equation
)
PointMorphismClass.tangent_cone = (
    _point_tangent_cone_equation
)
PointMorphismClass.embedding_dimension = (
    _point_embedding_dimension
)
PointMorphismClass.embedding_dim = (
    _point_embedding_dimension
)
PointMorphismClass.milnor_algebra = _point_milnor_algebra
PointMorphismClass.Milnor_algebra = _point_milnor_algebra
PointMorphismClass.tjurina_algebra = _point_tjurina_algebra
PointMorphismClass.Tjurina_algebra = _point_tjurina_algebra
PointMorphismClass.milnor_number = _point_milnor_number
PointMorphismClass.tjurina_number = _point_tjurina_number
PointMorphismClass.ADE_type = _point_ADE_type
PointMorphismClass.equation_of_normal_form = (
    _point_equation_of_normal_form
)

print('Installed local germs and point-local hypersurface singularity methods.')

## Local rings at rational points

Let $p\in X$ be a rational point and let

$$
B=k[t_1,\ldots,t_r]/I
$$

be the translated affine presentation supplied by `p.local_germ()`, with $p$ represented by the maximal ideal $\mathfrak m=(t_1,\ldots,t_r)$. The method `p.local_ring()` returns the genuine localization

$$
\mathcal O_{X,p}=B_{\mathfrak m}.
$$

An element is represented by a fraction $a/b$ with $b\notin\mathfrak m$. For possibly reducible or nonreduced $B$, equality is tested by the exact criterion

$$
\frac ab=\frac cd
\quad\Longleftrightarrow\quad
\bigl(I:(ad-bc)\bigr)+\mathfrak m=k[t_1,\ldots,t_r].
$$

The implementation therefore does not assume that the germ is integral. It currently requires an exact base field because ideal quotients are used to decide equality.

In [ ]:
from sage.structure.parent import Parent
from sage.structure.element import CommutativeRingElement


class AffineLocalRingElement(CommutativeRingElement):
    def __init__(self, parent, numerator, denominator):
        CommutativeRingElement.__init__(self, parent)
        presentation_ring = parent.presentation_ring()
        numerator = presentation_ring(numerator)
        denominator = presentation_ring(denominator)
        if denominator == 0:
            raise ZeroDivisionError('the denominator is zero')
        if parent.is_in_maximal_ideal(denominator):
            raise ValueError(
                'the denominator lies in the maximal ideal'
            )
        self._numerator = numerator
        self._denominator = denominator

    def numerator(self):
        return self._numerator

    def denominator(self):
        return self._denominator

    def _repr_(self):
        if self._denominator == 1:
            return repr(self._numerator)
        return (
            f'({self._numerator})/'
            f'({self._denominator})'
        )

    def _latex_(self):
        if self._denominator == 1:
            return str(latex(self._numerator))
        return (
            r'\frac{'
            + str(latex(self._numerator))
            + r'}{'
            + str(latex(self._denominator))
            + r'}'
        )

    def _coerce_other(self, other):
        if (
            isinstance(other, AffineLocalRingElement)
            and other.parent() is self.parent()
        ):
            return other
        return self.parent()(other)

    def __add__(self, other):
        other = self._coerce_other(other)
        return self.parent()(
            self._numerator * other._denominator
            + other._numerator * self._denominator,
            self._denominator * other._denominator,
        )

    __radd__ = __add__

    def __neg__(self):
        return self.parent()(
            -self._numerator,
            self._denominator,
        )

    def __sub__(self, other):
        return self.__add__(-self._coerce_other(other))

    def __rsub__(self, other):
        return self.parent()(other).__sub__(self)

    def __mul__(self, other):
        other = self._coerce_other(other)
        return self.parent()(
            self._numerator * other._numerator,
            self._denominator * other._denominator,
        )

    __rmul__ = __mul__

    def __invert__(self):
        if not self.is_unit():
            raise ZeroDivisionError(
                'the element is not a unit in the local ring'
            )
        return self.parent()(
            self._denominator,
            self._numerator,
        )

    def inverse(self):
        return ~self

    def __truediv__(self, other):
        return self.__mul__(~self._coerce_other(other))

    def __rtruediv__(self, other):
        return self.parent()(other).__truediv__(self)

    def __pow__(self, exponent):
        exponent = ZZ(exponent)
        if exponent < 0:
            return (~self)**(-exponent)
        result = self.parent().one()
        power = self
        while exponent:
            if exponent % 2:
                result = result * power
            power = power * power
            exponent //= 2
        return result

    def __eq__(self, other):
        try:
            other = self._coerce_other(other)
        except (TypeError, ValueError):
            return False
        return self.parent().fractions_are_equal(
            self._numerator,
            self._denominator,
            other._numerator,
            other._denominator,
        )

    def __ne__(self, other):
        return not self.__eq__(other)

    def is_unit(self):
        return not self.parent().is_in_maximal_ideal(
            self._numerator
        )

    def residue(self):
        parent = self.parent()
        return (
            parent.residue_of(self._numerator)
            / parent.residue_of(self._denominator)
        )


class AffineLocalMaximalIdeal(SageObject):
    def __init__(self, local_ring):
        self._local_ring = local_ring

    def _repr_(self):
        return f'Maximal ideal of {self._local_ring}'

    def ring(self):
        return self._local_ring

    def gens(self):
        return self._local_ring.maximal_ideal_generators()

    generators = gens

    def __contains__(self, element):
        element = self._local_ring(element)
        return element.residue() == 0


class AffineLocalResidueMap(SageObject):
    def __init__(self, local_ring):
        self._local_ring = local_ring

    def _repr_(self):
        return (
            f'Residue map from {self._local_ring} '
            f'to {self.codomain()}'
        )

    def domain(self):
        return self._local_ring

    def codomain(self):
        return self._local_ring.residue_field()

    def __call__(self, element):
        return self._local_ring(element).residue()


print('Loaded local-ring elements, maximal ideals, and residue maps.')

In [ ]:
class AffineLocalRing(Parent):
    Element = AffineLocalRingElement

    def __init__(self, germ):
        if not isinstance(germ, LocalSchemeGerm):
            raise TypeError(
                'a local ring must be constructed from a LocalSchemeGerm'
            )
        ambient_ring = germ.presentation_ring()
        if not ambient_ring.base_ring().is_exact():
            raise NotImplementedError(
                'the local-ring backend requires an exact base field'
            )

        self._germ = germ
        self._ambient_ring = ambient_ring
        self._defining_ideal = germ.defining_ideal()
        self._presentation_ring = ambient_ring.quotient(
            self._defining_ideal,
            names=ambient_ring.variable_names(),
        )
        self._ambient_maximal_ideal = ambient_ring.ideal(
            ambient_ring.gens()
        )
        self._maximal_ideal_cache = None
        self._residue_map_cache = None
        self._integral_domain_cache = None
        Parent.__init__(
            self,
            base=ambient_ring.base_ring(),
            category=CommutativeRings(),
        )

    def _repr_(self):
        return (
            f'Local ring of {self.scheme()} '
            f'at {self.point()}'
        )

    def germ(self):
        return self._germ

    def scheme(self):
        return self._germ.scheme()

    def point(self):
        return self._germ.point()

    def ambient_polynomial_ring(self):
        return self._ambient_ring

    def defining_ideal(self):
        return self._defining_ideal

    def presentation_ring(self):
        return self._presentation_ring

    def residue_field(self):
        return self.base_ring()

    def residue_map(self):
        if self._residue_map_cache is None:
            self._residue_map_cache = AffineLocalResidueMap(
                self
            )
        return self._residue_map_cache

    def residue_of(self, element):
        lift = self._presentation_ring(element).lift()
        return self.base_ring()(
            lift(*([0] * self._ambient_ring.ngens()))
        )

    def is_in_maximal_ideal(self, element):
        if (
            isinstance(element, AffineLocalRingElement)
            and element.parent() is self
        ):
            return element.residue() == 0
        return self.residue_of(element) == 0

    def maximal_ideal_generators(self):
        return tuple(
            self(self._presentation_ring.gen(index))
            for index in range(
                self._presentation_ring.ngens()
            )
        )

    def maximal_ideal(self):
        if self._maximal_ideal_cache is None:
            self._maximal_ideal_cache = (
                AffineLocalMaximalIdeal(self)
            )
        return self._maximal_ideal_cache

    def gens(self):
        return self.maximal_ideal_generators()

    def gen(self, index=0):
        return self.gens()[ZZ(index)]

    def ngens(self):
        return ZZ(len(self.gens()))

    def dimension(self):
        return self._germ.local_dimension()

    def embedding_dimension(self):
        return self._germ.embedding_dimension()

    embedding_dim = embedding_dimension

    def is_regular(self):
        return (
            self.dimension()
            == self.embedding_dimension()
        )

    def is_local(self):
        return True

    def is_noetherian(self):
        return True

    def is_integral_domain(self):
        if self._integral_domain_cache is None:
            self._integral_domain_cache = (
                self._defining_ideal.is_prime()
            )
        return self._integral_domain_cache

    def characteristic(self):
        return self.base_ring().characteristic()

    def fractions_are_equal(
        self,
        left_numerator,
        left_denominator,
        right_numerator,
        right_denominator,
    ):
        presentation_ring = self._presentation_ring
        difference = (
            presentation_ring(left_numerator)
            * presentation_ring(right_denominator)
            - presentation_ring(right_numerator)
            * presentation_ring(left_denominator)
        )
        if difference == 0:
            return True

        difference_lift = difference.lift()
        annihilator = self._defining_ideal.quotient(
            self._ambient_ring.ideal([
                difference_lift
            ])
        )
        return (
            annihilator + self._ambient_maximal_ideal
        ).is_one()

    def fraction(self, numerator, denominator):
        return self(numerator, denominator)

    def _element_constructor_(
        self,
        numerator=0,
        denominator=None,
    ):
        if (
            isinstance(numerator, AffineLocalRingElement)
            and numerator.parent() is self
            and denominator is None
        ):
            return numerator
        if (
            denominator is None
            and isinstance(numerator, (tuple, list))
            and len(numerator) == 2
        ):
            numerator, denominator = numerator
        if denominator is None:
            denominator = self._presentation_ring.one()
        return self.element_class(
            self,
            self._presentation_ring(numerator),
            self._presentation_ring(denominator),
        )

    def _coerce_map_from_(self, source):
        return self._presentation_ring.has_coerce_map_from(
            source
        )


def _germ_local_ring(self):
    local_ring = getattr(
        self,
        '_projective_framework_local_ring',
        None,
    )
    if local_ring is None:
        local_ring = AffineLocalRing(self)
        self._projective_framework_local_ring = local_ring
    return local_ring


def _point_local_ring(self):
    return self.local_germ().local_ring()


LocalSchemeGerm.local_ring = _germ_local_ring
PointMorphismClass.local_ring = _point_local_ring

print('Installed exact local rings at rational points.')

## Singular loci as closed subschemes

Let $X$ be an equidimensional closed subscheme of a smooth supported projective ambient space $A$ over an exact perfect field. If $c=\operatorname{codim}_A(X)$ and $J_X$ is the Jacobian matrix of a homogeneous generating set for the defining ideal, the Jacobian criterion gives

$$
\operatorname{Sing}(X)
=
V\!\left(I_X+I_c(J_X)\right)
\subseteq A,
$$

with multiprojective saturation by the irrelevant ideals. The next cell installs `X.singular_locus()` as this closed subscheme and `X.is_singular()` as the nonemptiness test. For rational points, `p.is_singular()` tests membership in the scheme-level singular locus and agrees with the local-germ regularity computation.

In [ ]:
def _empty_closed_subscheme(ambient):
    return ambient.subscheme([1])


def _point_coordinates_in_ambient(point):
    scheme = point.codomain()
    ambient = scheme.ambient_space()
    factors = _projective_factors(ambient)
    if len(factors) == 1:
        return tuple(point)
    return tuple(
        coordinate
        for factor_index in range(len(factors))
        for coordinate in tuple(point[factor_index])
    )


def _scheme_singular_locus(self):
    attribute = '_projective_framework_singular_locus'
    singular_locus = getattr(self, attribute, None)
    if singular_locus is not None:
        return singular_locus

    ambient = _require_supported_projective_ambient(
        self,
        'singular locus',
    )
    base_field = self.base_ring()
    if not base_field.is_field():
        raise NotImplementedError(
            'the Jacobian singular-locus backend requires a field base'
        )
    if not base_field.is_exact():
        raise NotImplementedError(
            'the Jacobian singular-locus backend requires an exact base field'
        )
    if not base_field.is_perfect():
        raise NotImplementedError(
            'the Jacobian criterion is currently asserted only over perfect fields'
        )

    if self == ambient or self.dimension() < 0:
        singular_locus = _empty_closed_subscheme(ambient)
        setattr(self, attribute, singular_locus)
        return singular_locus

    components = tuple(self.irreducible_components())
    component_dimensions = {
        ZZ(component.dimension())
        for component in components
    }
    if len(component_dimensions) > 1:
        raise NotImplementedError(
            'the current Jacobian singular-locus backend requires an equidimensional scheme'
        )

    codimension = ZZ(
        ambient.dimension() - self.dimension()
    )
    coordinate_ring = ambient.coordinate_ring()
    equations = tuple(_defining_equations(self))
    jacobian_matrix = matrix(
        coordinate_ring,
        [
            [
                equation.derivative(coordinate)
                for coordinate in coordinate_ring.gens()
            ]
            for equation in equations
        ],
    )
    if codimension == 0:
        jacobian_minors = tuple()
    else:
        jacobian_minors = tuple(
            jacobian_matrix.minors(codimension)
        )

    singular_ideal = coordinate_ring.ideal(
        equations + jacobian_minors
    )
    singular_ideal = _saturate_projective_ideal(
        self,
        singular_ideal,
        coordinate_ring.gens(),
    )
    singular_locus = ambient.subscheme(
        singular_ideal.gens()
    )
    setattr(self, attribute, singular_locus)
    return singular_locus


def _scheme_is_singular(self):
    return not self.singular_locus().defining_ideal().is_one()


def _point_is_singular_via_scheme(self):
    scheme = self.codomain()
    try:
        singular_locus = scheme.singular_locus()
    except (AttributeError, NotImplementedError):
        return self.local_germ().is_singular()

    coordinates = _point_coordinates_in_ambient(self)
    coordinate_ring = scheme.ambient_space().coordinate_ring()
    return all(
        coordinate_ring(equation)(*coordinates) == 0
        for equation in singular_locus.defining_polynomials()
    )


def _toric_irrelevant_ideal(ambient):
    coordinate_ring = ambient.coordinate_ring()
    fan = ambient.fan()
    all_ray_indices = set(range(len(fan.rays())))
    generators = []
    for cone in fan.generating_cones():
        cone_indices = set(cone.ambient_ray_indices())
        complement = sorted(all_ray_indices - cone_indices)
        generators.append(
            prod(
                coordinate_ring.gen(index)
                for index in complement
            )
        )
    return coordinate_ring.ideal(generators)


def _toric_scheme_singular_locus(self):
    attribute = '_projective_framework_singular_locus'
    singular_locus = getattr(self, attribute, None)
    if singular_locus is not None:
        return singular_locus

    ambient = self.ambient_space()
    base_field = self.base_ring()
    if not base_field.is_field():
        raise NotImplementedError(
            'the toric Jacobian backend requires a field base'
        )
    if not base_field.is_exact():
        raise NotImplementedError(
            'the toric Jacobian backend requires an exact base field'
        )
    if not base_field.is_perfect():
        raise NotImplementedError(
            'the Jacobian criterion is currently asserted only over perfect fields'
        )
    if not ambient.fan().is_smooth():
        raise NotImplementedError(
            'the toric Jacobian backend currently requires a smooth ambient toric variety'
        )

    components = tuple(self.irreducible_components())
    component_dimensions = {
        ZZ(component.dimension())
        for component in components
    }
    if len(component_dimensions) > 1:
        raise NotImplementedError(
            'the toric Jacobian backend requires an equidimensional scheme'
        )

    codimension = ZZ(
        ambient.dimension() - self.dimension()
    )
    coordinate_ring = ambient.coordinate_ring()
    equations = tuple(self.defining_polynomials())
    jacobian_matrix = matrix(
        coordinate_ring,
        [
            [
                equation.derivative(coordinate)
                for coordinate in coordinate_ring.gens()
            ]
            for equation in equations
        ],
    )
    jacobian_minors = (
        tuple(jacobian_matrix.minors(codimension))
        if codimension > 0
        else tuple()
    )
    singular_ideal = coordinate_ring.ideal(
        equations + jacobian_minors
    )
    singular_ideal = singular_ideal.saturation(
        _toric_irrelevant_ideal(ambient)
    )[0]
    singular_locus = ambient.subscheme(
        singular_ideal.gens()
    )
    setattr(self, attribute, singular_locus)
    return singular_locus


from sage.schemes.toric.toric_subscheme import (
    AlgebraicScheme_subscheme_toric,
)

for scheme_class in (
    ProjectiveSpace_ring,
    ProductProjectiveSpaces_ring,
    AlgebraicScheme_subscheme_projective,
    AlgebraicScheme_subscheme_product_projective,
):
    scheme_class.singular_locus = _scheme_singular_locus
    scheme_class.is_singular = _scheme_is_singular

AlgebraicScheme_subscheme_toric.singular_locus = (
    _toric_scheme_singular_locus
)
AlgebraicScheme_subscheme_toric.is_singular = (
    _scheme_is_singular
)

PointMorphismClass.is_singular = _point_is_singular_via_scheme

print(
    'Installed projective and toric scheme-level singular loci '
    'and point-membership singularity tests.'
)

## Finite group actions, linearisations, and isotypic decomposition

Let a finite group $G$ act on a projective product $X$ through a homomorphism

$$
\rho:G\longrightarrow\operatorname{Aut}(X).
$$

For a $G$-linearised line bundle $L$, the induced left action on sections is

$$
g\cdot s=(\rho(g^{-1}))^*s.
$$

The inverse ensures that the pullback action is a representation rather than an anti-representation. For the standard twists $\mathcal O_X(d_1,\ldots,d_r)$, the next cell constructs the coordinate linearisation whenever every automorphism preserves the multidegree. It then uses Sage's native representation parent. Sage's `twisted_invariant_module(chi)` is exposed under the standard name `isotypic_component(chi)`, and `isotypic_decomposition()` iterates over the irreducible characters.

In [20]:
from sage.structure.sage_object import SageObject
import sage.modules.with_basis.representation as _representation_module



class FiniteSchemeGroupAction(SageObject):
    def __init__(self, automorphism_group, group, generator_images):
        if not group.is_finite():
            raise NotImplementedError('the current action constructor requires a finite group')

        self._automorphism_group = automorphism_group
        self._scheme = automorphism_group.scheme()
        self._group = group
        generators = tuple(group.gens())

        if isinstance(generator_images, dict):
            images = tuple(generator_images[generator] for generator in generators)
        else:
            images = tuple(generator_images)

        if len(images) != len(generators):
            raise ValueError(
                f'expected {len(generators)} generator images, received {len(images)}'
            )
        for image in images:
            if image not in automorphism_group:
                raise ValueError('every generator image must lie in the automorphism group')

        image_by_element = {
            group.one(): self._scheme.identity_morphism(),
        }
        queue = [group.one()]
        while queue:
            element = queue.pop(0)
            for generator, generator_image in zip(generators, images):
                product_element = element * generator
                product_image = image_by_element[element] * generator_image
                if product_element in image_by_element:
                    if not (image_by_element[product_element] == product_image):
                        raise ValueError('the generator images do not satisfy the group relations')
                else:
                    image_by_element[product_element] = product_image
                    queue.append(product_element)

        if len(image_by_element) != group.order():
            raise ValueError('the supplied generators do not generate the finite group')

        for left in group:
            for right in group:
                if not (
                    image_by_element[left * right]
                    == image_by_element[left] * image_by_element[right]
                ):
                    raise ValueError('the supplied data do not define a group homomorphism')

        self._image_by_element = image_by_element

    def _repr_(self):
        return f'Action of {self._group} on {self._scheme}'

    def group(self):
        return self._group

    def scheme(self):
        return self._scheme

    def automorphism_group(self):
        return self._automorphism_group

    def automorphism(self, group_element):
        return self._image_by_element[self._group(group_element)]

    __call__ = automorphism

    def linearize(self, line_bundle):
        return ProductProjectiveCoordinateLinearization(line_bundle, self)


class ProductProjectiveCoordinateLinearization(SageObject):
    def __init__(self, line_bundle, action):
        if line_bundle.scheme() != action.scheme():
            raise ValueError('the line bundle and group action must live on the same scheme')

        self._line_bundle = line_bundle
        self._action = action
        self._section_space = line_bundle.H(0)
        self._representation_cache = {}

        for group_element in action.group():
            pulled_line_bundle = line_bundle.pullback(
                action(group_element)
            )
            if pulled_line_bundle != line_bundle:
                raise ValueError(
                    f'{group_element} pulls {line_bundle} back to '
                    f'{pulled_line_bundle}; the line-bundle class is not invariant'
                )

    def _repr_(self):
        return (
            f'Coordinate linearisation of {self._line_bundle} '
            f'for {self._action.group()}'
        )

    def line_bundle(self):
        return self._line_bundle

    def action(self):
        return self._action

    def group(self):
        return self._action.group()

    def scheme(self):
        return self._line_bundle.scheme()

    def section_space(self):
        return self._section_space

    def act_on_section(self, group_element, section):
        group_element = self.group()(group_element)
        inverse_automorphism = self._action(group_element.inverse())
        return self._section_space.pullback(
            inverse_automorphism
        )(section)

    def H_representation(self, degree):
        degree = ZZ(degree)
        if degree != 0:
            raise NotImplementedError(
                'the current coordinate linearisation constructs only the action on H^0'
            )
        if degree not in self._representation_cache:
            group = self.group()
            section_space = self._line_bundle.H(degree)

            def on_basis(group_element, basis_index):
                return self.act_on_section(
                    group_element,
                    section_space.basis()[basis_index],
                )

            representation = group.representation(
                section_space,
                on_basis,
                side='left',
            )
            representation._linearized_line_bundle = self
            representation._cohomological_degree = degree
            self._representation_cache[degree] = representation
        return self._representation_cache[degree]


def _automorphism_group_action(self, group, generator_images):
    return FiniteSchemeGroupAction(self, group, generator_images)


def _line_bundle_linearize(self, action):
    return ProductProjectiveCoordinateLinearization(self, action)


def _representation_underlying_module(self):
    return self._module


def _representation_linearized_line_bundle(self):
    try:
        return self._linearized_line_bundle
    except AttributeError as error:
        raise ValueError('this representation was not induced by a line-bundle linearisation') from error


class IsotypicComponent(SageObject):
    def __init__(self, representation, character):
        self._representation = representation
        self._character = character
        self._native_module = representation.twisted_invariant_module(
            character
        )
        self._basis_cache = None

    def _repr_(self):
        return (
            f'Isotypic component of dimension {self.dimension()} '
            f'for character {self._character}'
        )

    def representation(self):
        return self._representation

    def character(self):
        return self._character

    def module(self):
        return self._native_module

    def dimension(self):
        return ZZ(self._native_module.dimension())

    def basis(self):
        if self._basis_cache is None:
            underlying_module = self._representation.underlying_module()
            native_basis = self._native_module.basis()

            def realized_basis_element(index):
                lifted = self._native_module.lift(native_basis[index])
                return underlying_module._from_dict(
                    lifted.monomial_coefficients(),
                    remove_zeros=True,
                )

            self._basis_cache = Family(
                range(self.dimension()),
                realized_basis_element,
            )
        return self._basis_cache


class IsotypicDecomposition(SageObject):
    def __init__(self, representation):
        self._representation = representation
        group = representation.semigroup()
        if not group.is_finite():
            raise NotImplementedError(
                'isotypic decomposition currently requires a finite group'
            )
        characteristic = representation.base_ring().characteristic()
        if characteristic != 0 and group.order() % characteristic == 0:
            raise NotImplementedError(
                'isotypic decomposition is not asserted in modular characteristic'
            )
        self._components = tuple(
            IsotypicComponent(representation, character)
            for character in group.irreducible_characters()
        )

    def _repr_(self):
        dimensions = tuple(
            component.dimension() for component in self._components
        )
        return (
            f'Isotypic decomposition of {self._representation} '
            f'with component dimensions {dimensions}'
        )

    def representation(self):
        return self._representation

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __getitem__(self, index):
        return self._components[index]

    def components(self):
        return self._components

    def characters(self):
        return tuple(
            component.character() for component in self._components
        )

    def component(self, character):
        for component in self._components:
            if component.character() == character:
                return component
        raise KeyError('the character is not an irreducible character of this representation')

    def trivial_component(self):
        return self.component(
            self._representation.semigroup().trivial_character()
        )

    def nontrivial_components(self):
        trivial = self.trivial_component()
        return tuple(
            component
            for component in self._components
            if component is not trivial
        )

    def dimension(self):
        return sum(
            component.dimension() for component in self._components
        )


def _representation_isotypic_decomposition(self):
    decomposition = getattr(
        self,
        '_projective_framework_isotypic_decomposition',
        None,
    )
    if decomposition is None:
        decomposition = IsotypicDecomposition(self)
        self._projective_framework_isotypic_decomposition = decomposition
    return decomposition


def _representation_isotypic_component(self, character):
    return self.isotypic_decomposition().component(character)


def _representation_invariants(self):
    return self.isotypic_decomposition().trivial_component()


def _install_equivariant_section_interface():
    for morphism_class in _projective_morphism_classes:
        for obsolete_name in (
            'pullback_line_bundle',
            'pullback_section',
        ):
            if obsolete_name in morphism_class.__dict__:
                delattr(morphism_class, obsolete_name)

    _ProjectiveAutomorphismGroup.action = _automorphism_group_action
    ProductProjectiveLineBundle.linearize = _line_bundle_linearize
    _representation_module.Representation_abstract.underlying_module = (
        _representation_underlying_module
    )
    _representation_module.Representation_abstract.linearized_line_bundle = (
        _representation_linearized_line_bundle
    )
    _representation_module.Representation_abstract.isotypic_component = (
        _representation_isotypic_component
    )
    _representation_module.Representation_abstract.isotypic_decomposition = (
        _representation_isotypic_decomposition
    )
    _representation_module.Representation_abstract.invariants = (
        _representation_invariants
    )


_install_equivariant_section_interface()

print('Installed finite scheme actions, coordinate linearisations, and isotypic decomposition methods.')

Installed finite scheme actions, coordinate linearisations, and isotypic decomposition methods.


## Structured mathematical display

Sage's rich display calls `_latex_()`. The next cell equips the custom parents, elements, graded morphisms, representations, covers, affine schemes, and native scheme morphisms with structured mathematical LaTeX.

The display preserves defining data. For example, section spaces show their full bases, graded morphisms show generator or basis images, and scheme morphisms show their induced coordinate or ring maps. Arrays and aligned environments organize this information without suppressing it.

Each object owns its own LaTeX representation. A morphism renders its endpoints by calling `latex()` on its domain and codomain; it does not create alternative endpoint notation. Supported schemes may be assigned a mathematical name by `X.set_latex_name('X')`, which changes `latex(X)` itself. This layer changes only presentation.

In [27]:
def _latex_identifier(name):
    name = str(name)
    if '_' not in name:
        return name
    head, tail = name.split('_', 1)
    return head + r'_{' + tail + r'}'


def _line_bundle_symbol(line_bundle):
    entries = ','.join(
        str(latex(value))
        for value in line_bundle.multidegree()
    )
    return (
        r'\mathcal O_{'
        + _display_latex_symbol(line_bundle.scheme())
        + r'}\!\left('
        + entries
        + r'\right)'
    )


def _global_sections_symbol(section_space):
    return (
        r'H^{0}\!\left('
        + _display_latex_symbol(section_space.scheme())
        + ','
        + _line_bundle_symbol(section_space.line_bundle())
        + r'\right)'
    )


def _affine_qcoherent_module_symbol(module):
    custom = getattr(
        module,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    if hasattr(module, '_dual_of'):
        return (
            r'\left('
            + _affine_qcoherent_module_symbol(module._dual_of)
            + r'\right)^{\vee}'
        )
    source_sections = module.source_sections()
    if source_sections is not None:
        return r'p_{*}' + _line_bundle_symbol(
            source_sections.line_bundle()
        )
    return (
        r'\mathcal O_{'
        + _display_latex_symbol(module.base_scheme())
        + r'}^{\oplus '
        + str(latex(module.rank()))
        + r'}'
    )


def _affine_scheme_symbol(scheme):
    custom = getattr(
        scheme,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    if hasattr(scheme, '_vv_module'):
        return (
            r'\mathbf V_{'
            + _display_latex_symbol(scheme.base_scheme())
            + r'}\!\left('
            + _affine_qcoherent_module_symbol(
                scheme.vv_module()
            )
            + r'\right)'
        )
    return (
        r'\operatorname{Spec}\!\left('
        + str(latex(scheme.coordinate_ring()))
        + r'\right)'
    )


def _display_latex_symbol(obj):
    custom = getattr(
        obj,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    if isinstance(obj, ProductProjectiveLineBundle):
        return _line_bundle_symbol(obj)
    if isinstance(obj, ProductProjectiveGlobalSections):
        return _global_sections_symbol(obj)
    if isinstance(obj, AffineQuasiCoherentFreeModule):
        return _affine_qcoherent_module_symbol(obj)
    if isinstance(obj, AffineScheme):
        return _affine_scheme_symbol(obj)
    if isinstance(obj, ProductProjectiveCoxRing):
        return (
            r'\operatorname{Cox}\!\left('
            + _display_latex_symbol(obj.scheme())
            + r'\right)'
        )
    return str(latex(obj))


def _display_latex_name(obj):
    return _display_latex_symbol(obj)


def _set_latex_name(self, name):
    self._projective_framework_latex_name = str(name)
    return self


def _clear_latex_name(self):
    if hasattr(self, '_projective_framework_latex_name'):
        delattr(self, '_projective_framework_latex_name')
    return self


def _latex_mapping_array(pairs, columns=2):
    pairs = tuple(pairs)
    if len(pairs) == 0:
        return r'\varnothing'
    rows = []
    for start in range(0, len(pairs), columns):
        row_pairs = pairs[start:start + columns]
        cells = []
        for source, target in row_pairs:
            cells.extend((
                str(latex(source)),
                r'\longmapsto',
                str(latex(target)),
            ))
        while len(row_pairs) < columns:
            cells.extend(('', '', ''))
            row_pairs = row_pairs + ((None, None),)
        rows.append(' & '.join(cells))
    column_specification = r'@{}' + r'rcl@{\qquad}' * (columns - 1) + r'rcl@{}'
    return (
        r'\begin{array}{'
        + column_specification
        + r'}'
        + r'\\'.join(rows)
        + r'\end{array}'
    )


def _latex_basis_array(entries, columns=5):
    entries = tuple(entries)
    if len(entries) == 0:
        return r'()'
    rows = []
    for start in range(0, len(entries), columns):
        row = [str(latex(entry)) for entry in entries[start:start + columns]]
        row.extend([''] * (columns - len(row)))
        rows.append(' & '.join(row))
    return (
        r'\left(\begin{array}{'
        + 'c' * columns
        + r'}'
        + r'\\'.join(rows)
        + r'\end{array}\right)'
    )


import sage.schemes.generic.morphism as _generic_morphism_module


if '_projective_framework_original_projective_space_latex' not in globals():
    _projective_framework_original_projective_space_latex = (
        ProjectiveSpace_ring._latex_
    )
    _projective_framework_original_product_projective_latex = (
        ProductProjectiveSpaces_ring._latex_
    )
    _projective_framework_original_affine_space_latex = (
        AffineSpace_generic._latex_
    )


def _projective_space_latex(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    return _projective_framework_original_projective_space_latex(
        self
    )


def _product_projective_space_latex(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    return _projective_framework_original_product_projective_latex(
        self
    )


def _affine_space_latex(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    return _projective_framework_original_affine_space_latex(self)


def _affine_scheme_latex(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    if hasattr(self, '_vv_module'):
        base = _display_latex_symbol(self.base_scheme())
        module = _affine_qcoherent_module_symbol(
            self.vv_module()
        )
        return (
            r'\begin{aligned}'
            + r'\mathbf V_{'
            + base
            + r'}\!\left('
            + module
            + r'\right)'
            + r'&=\underline{\operatorname{Spec}}_{'
            + base
            + r'}\operatorname{Sym}\!\left('
            + module
            + r'\right)\\'
            + r'H^{0}\!\left(\mathbf V_{'
            + base
            + r'}\!\left('
            + module
            + r'\right),\mathcal O\right)'
            + r'&='
            + str(latex(self.coordinate_ring()))
            + r'\end{aligned}'
        )
    return (
        r'\operatorname{Spec}\!\left('
        + str(latex(self.coordinate_ring()))
        + r'\right)'
    )


def _line_bundle_latex(self):
    entries = ','.join(
        str(latex(value))
        for value in self.multidegree()
    )
    return (
        r'\mathcal O_{'
        + _display_latex_name(self.scheme())
        + r'}\!\left('
        + entries
        + r'\right)'
    )


def _picard_group_latex(self):
    return (
        r'\operatorname{Pic}\!\left('
        + _display_latex_name(self.scheme())
        + r'\right)'
    )


def _global_sections_latex(self):
    symbol = (
        r'H^{0}\!\left('
        + _display_latex_name(self.scheme())
        + ','
        + str(latex(self.line_bundle()))
        + r'\right)'
    )
    return (
        r'\begin{aligned}'
        + symbol
        + r'&\quad\text{with}\quad '
        + r'\dim '
        + symbol
        + r'='
        + str(latex(self.dimension()))
        + r'\\'
        + r'\operatorname{basis}\!\left('
        + symbol
        + r'\right)&='
        + _latex_basis_array(tuple(self.basis()), columns=5)
        + r'\end{aligned}'
    )


def _cohomology_group_latex(self):
    return (
        r'H^{'
        + str(latex(self.degree()))
        + r'}\!\left('
        + _display_latex_name(self.scheme())
        + ','
        + str(latex(self.line_bundle()))
        + r'\right)'
    )


def _total_cohomology_latex(self):
    return (
        r'H^{\bullet}\!\left('
        + _display_latex_name(self.scheme())
        + ','
        + str(latex(self.line_bundle()))
        + r'\right)'
    )


def _cox_ring_latex(self):
    return (
        r'\operatorname{Cox}\!\left('
        + _display_latex_name(self.scheme())
        + r'\right)'
    )


def _section_ring_latex(self):
    return (
        r'R\!\left('
        + _display_latex_name(self.scheme())
        + ','
        + str(latex(self.line_bundle()))
        + r'\right)'
    )


def _graded_component_latex(self):
    return (
        str(latex(self.graded_algebra().algebra()))
        + r'_{'
        + str(latex(self.degree()))
        + r'}'
    )


def _graded_structure_latex(self):
    return (
        str(latex(self.algebra()))
        + r'\;\text{graded by}\;'
        + str(latex(self.grading_group()))
    )


def _graded_morphism_symbol(self):
    custom = getattr(
        self,
        '_projective_framework_latex_symbol',
        None,
    )
    if custom is not None:
        return str(custom)
    if isinstance(self.domain(), ProductProjectiveCoxRing):
        return (
            r'\Phi_{'
            + _display_latex_name(self.domain().scheme())
            + r'}'
        )
    return r'\varphi'


def _graded_morphism_latex(self):
    arrow = (
        r'\xrightarrow{\sim}'
        if self._inverse_function is not None
        else r'\longrightarrow'
    )
    symbol = _graded_morphism_symbol(self)
    generator_pairs = tuple()
    if hasattr(self.domain(), 'gens'):
        try:
            generators = tuple(self.domain().gens())
            generator_pairs = tuple(
                (generator, self(generator))
                for generator in generators
            )
        except (TypeError, ValueError, NotImplementedError):
            generator_pairs = tuple()

    result = (
        r'\begin{aligned}'
        + symbol
        + r':\;'
        + str(latex(self.domain()))
        + arrow
        + str(latex(self.codomain()))
    )
    if generator_pairs:
        result += (
            r'\\'
            + symbol
            + r'\big|_{\operatorname{gens}}&:\quad '
            + _latex_mapping_array(generator_pairs, columns=2)
        )
    return result + r'\end{aligned}'


def _graded_component_morphism_latex(self):
    parent_symbol = _graded_morphism_symbol(
        self.graded_morphism()
    )
    symbol = (
        parent_symbol
        + r'_{'
        + str(latex(self.degree()))
        + r'}'
    )
    arrow = (
        r'\xrightarrow{\sim}'
        if self.graded_morphism()._inverse_function is not None
        else r'\longrightarrow'
    )
    basis = tuple(self.domain().basis())
    basis_pairs = tuple(
        (basis_element, self.to_ambient(basis_element))
        for basis_element in basis
    )
    return (
        r'\begin{aligned}'
        + symbol
        + r':\;'
        + str(latex(self.domain()))
        + arrow
        + str(latex(self.codomain_component()))
        + r'\\'
        + symbol
        + r'\big|_{\operatorname{basis}}&:\quad '
        + _latex_mapping_array(basis_pairs, columns=2)
        + r'\end{aligned}'
    )


def _automorphism_group_latex(self):
    return (
        r'\operatorname{Aut}\!\left('
        + _display_latex_name(self.scheme())
        + r'\right)'
    )


def _finite_group_latex(group):
    custom = getattr(
        group,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)
    try:
        if group.is_cyclic():
            return r'C_{' + str(latex(group.order())) + r'}'
    except (AttributeError, NotImplementedError):
        pass
    return str(latex(group))


def _finite_action_latex(self):
    return (
        r'\rho:\;'
        + _finite_group_latex(self.group())
        + r'\curvearrowright '
        + _display_latex_name(self.scheme())
    )


def _linearization_latex(self):
    return (
        r'\lambda:\;'
        + _finite_group_latex(self.group())
        + r'\curvearrowright '
        + str(latex(self.line_bundle()))
    )


def _character_latex(component):
    representation = component.representation()
    group = representation.semigroup()
    character = component.character()
    if character == group.trivial_character():
        return r'\chi_{\mathrm{triv}}'
    try:
        if group.is_cyclic() and group.order() == 2:
            return r'\chi_{\mathrm{sgn}}'
    except (AttributeError, NotImplementedError):
        pass
    decomposition = representation.isotypic_decomposition()
    index = tuple(decomposition.components()).index(component)
    return r'\chi_{' + str(index) + r'}'


def _isotypic_component_latex(self):
    return r'V_{' + _character_latex(self) + r'}'


def _isotypic_decomposition_latex(self):
    components = tuple(self.components())
    right_hand_side = r'\oplus'.join(
        str(latex(component))
        for component in components
    )
    return (
        str(latex(self.representation().underlying_module()))
        + r'\cong '
        + right_hand_side
    )


def _representation_latex(self):
    group = _finite_group_latex(self.semigroup())
    module_latex = str(latex(self.underlying_module()))
    return (
        r'\rho_{'
        + module_latex
        + r'}:\;'
        + group
        + r'\longrightarrow '
        + r'\operatorname{GL}\!\left('
        + module_latex
        + r'\right)'
    )


def _affine_qcoherent_module_latex(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        symbol = str(custom)
    elif hasattr(self, '_dual_of'):
        symbol = _affine_qcoherent_module_symbol(self)
    else:
        source_sections = self.source_sections()
        if source_sections is not None:
            symbol = (
                r'p_{*}'
                + str(latex(source_sections.line_bundle()))
            )
        else:
            symbol = (
                r'\mathcal O_{'
                + _display_latex_name(self.base_scheme())
                + r'}^{\oplus '
                + str(latex(self.rank()))
                + r'}'
            )

    basis_entries = tuple(
        LatexExpr(name)
        for name in self.basis_names()
    )
    return (
        r'\begin{aligned}'
        + symbol
        + r'&\cong '
        + r'\mathcal O_{'
        + _display_latex_name(self.base_scheme())
        + r'}^{\oplus '
        + str(latex(self.rank()))
        + r'}\\'
        + r'\operatorname{rank}('
        + symbol
        + r')&='
        + str(latex(self.rank()))
        + r'\\'
        + r'\operatorname{basis}('
        + symbol
        + r')&='
        + _latex_basis_array(basis_entries, columns=5)
        + r'\end{aligned}'
    )


def _scheme_morphism_symbol(self):
    custom = getattr(
        self,
        '_projective_framework_latex_name',
        None,
    )
    if custom is not None:
        return str(custom)

    role = getattr(
        self,
        '_projective_framework_morphism_role',
        None,
    )
    if role == 'generic_point':
        return r'\eta'
    if role == 'generic_base_change':
        return r'q'
    if role == 'vector_bundle_projection':
        return r'\pi'

    if isinstance(
        self,
        _generic_morphism_module.SchemeMorphism_id,
    ):
        return r'\operatorname{id}'

    if isinstance(
        self,
        _generic_morphism_module.SchemeMorphism_structure_map,
    ):
        return r'p'

    if self.is_endomorphism():
        return r'\varphi'
    return r'f'


def _scheme_morphism_latex(self):
    symbol = _scheme_morphism_symbol(self)
    result = (
        r'\begin{aligned}'
        + symbol
        + r':\;'
        + str(latex(self.domain()))
        + r'&\longrightarrow '
        + str(latex(self.codomain()))
    )

    if hasattr(self, 'defining_polynomials'):
        try:
            defining_polynomials = tuple(
                self.defining_polynomials()
            )
            target_ring = self.codomain().coordinate_ring()
            source_ring = self.domain().coordinate_ring()
            target_generators = tuple(target_ring.gens())
            if len(target_generators) == len(defining_polynomials):
                result += (
                    r'\\'
                    + symbol
                    + r'^{\#}:\;'
                    + str(latex(target_ring))
                    + r'&\longrightarrow '
                    + str(latex(source_ring))
                    + r'\\'
                    + r'&\quad '
                    + _latex_mapping_array(
                        tuple(zip(
                            target_generators,
                            defining_polynomials,
                        )),
                        columns=2,
                    )
                )
        except (AttributeError, TypeError, ValueError, NotImplementedError):
            pass
    elif hasattr(self, 'ring_homomorphism'):
        try:
            ring_map = self.ring_homomorphism()
            source_ring = ring_map.domain()
            target_ring = ring_map.codomain()
            source_generators = tuple(source_ring.gens())
            nontrivial_generators = tuple(
                generator
                for generator in source_generators
                if generator != source_ring.one()
            )
            result += (
                r'\\'
                + symbol
                + r'^{\#}:\;'
                + str(latex(source_ring))
                + r'&\longrightarrow '
                + str(latex(target_ring))
            )
            if nontrivial_generators:
                result += (
                    r'\\'
                    + r'&\quad '
                    + _latex_mapping_array(
                        tuple(
                            (
                                generator,
                                ring_map(generator),
                            )
                            for generator
                            in nontrivial_generators
                        ),
                        columns=2,
                    )
                )
            else:
                result += r'\quad\text{(canonical scalar map)}'
        except (AttributeError, TypeError, ValueError, NotImplementedError):
            pass
    elif isinstance(
        self,
        _generic_morphism_module.SchemeMorphism_structure_map,
    ):
        result += (
            r'\\'
            + symbol
            + r'^{\#}:\;'
            + str(latex(self.codomain().coordinate_ring()))
            + r'&\longrightarrow '
            + r'H^{0}\!\left('
            + str(latex(self.domain()))
            + r',\mathcal O_{'
            + str(latex(self.domain()))
            + r'}\right)'
            + r'\quad\text{(structure homomorphism)}'
        )

    return result + r'\end{aligned}'


def _affine_cover_latex(self):
    return (
        r'\mathfrak U_{'
        + _display_latex_name(self.scheme())
        + r'}='
        + r'\left(U_{\alpha}\hookrightarrow '
        + _display_latex_name(self.scheme())
        + r'\right)_{\alpha}'
    )


def _section_ring_element_latex(self):
    return str(latex(self.cox_element()))


def _projective_point_blocks_for_latex(point, ambient):
    factors = _projective_factors(ambient)
    if len(factors) == 1:
        return (tuple(point),)
    return tuple(
        tuple(point[index])
        for index in range(len(factors))
    )


def _projective_point_latex(point, ambient):
    blocks = _projective_point_blocks_for_latex(
        point,
        ambient,
    )
    if len(blocks) == 1:
        return str(latex(blocks[0]))
    return str(latex(blocks))


def _local_ring_point_latex(local_ring):
    return _projective_point_latex(
        local_ring.point(),
        local_ring.scheme().ambient_space(),
    )


def _affine_local_ring_latex(self):
    point_latex = _local_ring_point_latex(self)
    return (
        r'\begin{aligned}'
        + r'\mathcal O_{'
        + str(latex(self.scheme()))
        + r','
        + point_latex
        + r'}&='
        + r'\left('
        + str(latex(self.presentation_ring()))
        + r'\right)_{\mathfrak m}\\'
        + r'\mathfrak m&='
        + _latex_basis_array(
            self.maximal_ideal().gens(),
            columns=4,
        )
        + r'\\'
        + r'\dim\mathcal O_{X,p}&='
        + str(latex(self.dimension()))
        + r',\qquad '
        + r'\operatorname{edim}\mathcal O_{X,p}='
        + str(latex(self.embedding_dimension()))
        + r'\\'
        + r'\kappa(p)&='
        + str(latex(self.residue_field()))
        + r'\end{aligned}'
    )


def _affine_local_maximal_ideal_latex(self):
    return (
        r'\mathfrak m_{p}='
        + _latex_basis_array(
            self.gens(),
            columns=4,
        )
        + r'\subset '
        + str(latex(self.ring()))
    )


def _affine_local_residue_map_latex(self):
    return (
        r'\operatorname{res}_{p}:\;'
        + str(latex(self.domain()))
        + r'\longrightarrow '
        + str(latex(self.codomain()))
    )


def _local_hypersurface_presentation_latex(self):
    eliminated = self.eliminated_variables()
    if len(eliminated) == 0:
        elimination_latex = r'\varnothing'
    else:
        elimination_latex = _latex_basis_array(
            tuple(
                LatexExpr(
                    str(name)
                    + r'='
                    + str(latex(solution))
                )
                for name, solution in eliminated
            ),
            columns=2,
        )
    return (
        r'\begin{aligned}'
        + r'R_{\mathrm{hyp}}&='
        + str(latex(self.ring()))
        + r'\\'
        + r'f_p&='
        + str(latex(self.equation()))
        + r'\\'
        + r'\mathcal O_{X,p}&\cong '
        + r'\left(R_{\mathrm{hyp}}/(f_p)\right)_{\mathfrak m}\\'
        + r'\text{eliminated variables}&='
        + elimination_latex
        + r'\end{aligned}'
    )


def _local_scheme_germ_latex(self):
    point_latex = _projective_point_latex(
        self.point(),
        self.scheme().ambient_space(),
    )
    result = (
        r'\begin{aligned}'
        + r'(X,p)&=('
        + str(latex(self.scheme()))
        + r','
        + point_latex
        + r')\\'
        + r'U_p&='
        + str(latex(self.chart().domain()))
        + r'\\'
        + r'\widehat I_p&='
        + str(latex(self.defining_ideal()))
        + r'\\'
        + r'\dim T_pX&='
        + str(latex(self.embedding_dimension()))
        + r'\\'
        + r'p\in\operatorname{Sing}(X)&\iff '
        + str(latex(self.is_singular()))
    )
    try:
        equation = self.local_equation()
        result += (
            r'\\'
            + r'f_p&='
            + str(latex(equation))
        )
    except NotImplementedError:
        pass
    if self.is_singular():
        try:
            result += (
                r'\\'
                + r'\mu_p&='
                + str(latex(self.milnor_number()))
                + r',\qquad '
                + r'\tau_p='
                + str(latex(self.tjurina_number()))
                + r'\\'
                + r'\operatorname{ADE}(p)&='
                + str(latex(self.ADE_type()))
            )
        except NotImplementedError:
            pass
    return result + r'\end{aligned}'


def _finite_restriction_line_bundle_latex(self):
    return (
        r'\begin{aligned}'
        + str(latex(self.base_morphism()))
        + r'\\'
        + r'i^*L&='
        + str(latex(self.source_line_bundle()))
        + r'\big|_{'
        + str(latex(self.scheme()))
        + r'}\\'
        + r'\operatorname{trivialisations}&='
        + _latex_basis_array(
            tuple(
                LatexExpr(str(label))
                for label in self.trivialization_labels()
            ),
            columns=4,
        )
        + r'\end{aligned}'
    )


def _finite_restriction_sections_latex(self):
    return (
        r'\begin{aligned}'
        + r'H^0\!\left('
        + str(latex(self.scheme()))
        + r','
        + str(latex(self.restriction()))
        + r'\right)&\cong '
        + str(latex(self.base_ring()))
        + r'^{\oplus '
        + str(latex(self.dimension()))
        + r'}\\'
        + r'\operatorname{basis}&='
        + _latex_basis_array(tuple(self.basis()), columns=4)
        + r'\end{aligned}'
    )


def _section_restriction_morphism_latex(self):
    return (
        r'\begin{aligned}'
        + r'i^*:\;'
        + str(latex(self.domain()))
        + r'&\longrightarrow '
        + str(latex(self.codomain()))
        + r'\\'
        + r'\operatorname{rank}(i^*)&='
        + str(latex(self.rank()))
        + r'\\'
        + r'\operatorname{Mat}(i^*)&='
        + str(latex(self.matrix()))
        + r'\end{aligned}'
    )


def _complete_linear_system_latex(self):
    line_bundle = str(latex(self.line_bundle()))
    section_space = str(latex(self.section_space()))
    if self.is_empty():
        return (
            r'\begin{aligned}'
            + r'\lvert '
            + line_bundle
            + r'\rvert&=\varnothing\\'
            + r'H^0&='
            + section_space
            + r'\end{aligned}'
        )
    return (
        r'\begin{aligned}'
        + r'\lvert '
        + line_bundle
        + r'\rvert&=\mathbf P\!\left('
        + section_space
        + r'^{\vee}\right)\\'
        + r'\dim\lvert '
        + line_bundle
        + r'\rvert&='
        + str(latex(self.dimension()))
        + r'\\'
        + r'\operatorname{Bs}\lvert '
        + line_bundle
        + r'\rvert&='
        + str(latex(self.base_locus()))
        + r'\end{aligned}'
    )


def _pullback_diagram_latex(self):
    return (
        r'\begin{array}{ccc}'
        + str(latex(self.apex()))
        + r'&\xrightarrow{\;'
        + _scheme_morphism_symbol(self.right_projection())
        + r'\;}&'
        + str(latex(self.right_morphism().domain()))
        + r'\\[-2pt]'
        + r'{\scriptstyle '
        + _scheme_morphism_symbol(self.left_projection())
        + r'}\downarrow&&\downarrow{\scriptstyle '
        + _scheme_morphism_symbol(self.right_morphism())
        + r'}\\[-2pt]'
        + str(latex(self.left_morphism().domain()))
        + r'&\xrightarrow{\;'
        + _scheme_morphism_symbol(self.left_morphism())
        + r'\;}&'
        + str(latex(self.base()))
        + r'\end{array}'
    )


def _split_projective_bundle_latex(self):
    return (
        r'\begin{aligned}'
        + r'\mathbf P_{'
        + str(latex(self.base_scheme()))
        + r'}\!\left(\mathcal O\oplus '
        + str(latex(self.twisting_line_bundle().dual()))
        + r'\right)&='
        + str(latex(self.scheme()))
        + r'\\'
        + r'\operatorname{Cl}\text{-grading}&='
        + str(latex(self.grading_matrix()))
        + r'\end{aligned}'
    )


def _cyclic_cover_morphism_latex(self):
    return (
        r'\begin{aligned}'
        + r'\pi:\;'
        + str(latex(self.domain()))
        + r'&\longrightarrow '
        + str(latex(self.codomain()))
        + r'\\'
        + r'\deg(\pi)&='
        + str(latex(self.cover_degree()))
        + r'\\'
        + r'Y&=V\!\left('
        + str(latex(self.cover_equation()))
        + r'\right)\\'
        + r'\operatorname{Branch}(\pi)&='
        + str(latex(self.branch_subscheme()))
        + r'\\'
        + r'\operatorname{Ram}(\pi)&='
        + str(latex(self.ramification_subscheme()))
        + r'\end{aligned}'
    )


def _cyclic_cover_algebra_datum_latex(self):
    pieces = r'\oplus'.join(
        str(latex(piece))
        for piece in self.pieces()
    )
    return (
        r'\mathcal A='
        + pieces
        + r',\qquad '
        + r'\deg(\mathcal A)='
        + str(latex(self.degree()))
    )


def _cyclic_cover_datum_latex(self):
    return (
        r'\begin{aligned}'
        + r'n&='
        + str(latex(self.degree()))
        + r'\\'
        + r'M&='
        + str(latex(self.root_line_bundle()))
        + r'\\'
        + r's&\in '
        + str(latex(self.branch_section().parent()))
        + r'\\'
        + r'M^{\otimes n}&='
        + str(latex(self.branch_line_bundle()))
        + r'\\'
        + str(latex(self.cover_algebra_datum()))
        + r'\end{aligned}'
    )


def _ipython_latex_repr(self):
    return (
        r'$\displaystyle '
        + str(self._latex_())
        + r'$'
    )


def _sage_rich_latex_repr(self, display_manager, **kwds):
    from sage.repl.rich_output.output_catalog import OutputLatex

    if OutputLatex not in display_manager.supported_output():
        raise NotImplementedError
    return OutputLatex(str(self._latex_()))


def _install_normalized_latex_display():
    for named_class in (
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
        AffineSpace_generic,
        AffineScheme,
        AffineQuasiCoherentFreeModule,
        _generic_morphism_module.SchemeMorphism,
    ):
        named_class.set_latex_name = _set_latex_name
        named_class.clear_latex_name = _clear_latex_name

    ProjectiveSpace_ring._latex_ = _projective_space_latex
    ProductProjectiveSpaces_ring._latex_ = (
        _product_projective_space_latex
    )
    AffineSpace_generic._latex_ = _affine_space_latex
    AffineScheme._latex_ = _affine_scheme_latex
    _generic_morphism_module.SchemeMorphism._latex_ = (
        _scheme_morphism_latex
    )

    ProductProjectiveLineBundle._latex_ = (
        _line_bundle_latex
    )
    ProductProjectivePicardGroup._latex_ = (
        _picard_group_latex
    )
    ProductProjectiveGlobalSections._latex_ = (
        _global_sections_latex
    )
    ProductProjectiveCohomologyGroup._latex_ = (
        _cohomology_group_latex
    )
    ProductProjectiveLineBundleCohomology._latex_ = (
        _total_cohomology_latex
    )
    ProductProjectiveCoxRing._latex_ = _cox_ring_latex
    ProductProjectiveSectionRing._latex_ = (
        _section_ring_latex
    )
    ProductProjectiveSectionRingElement._latex_ = (
        _section_ring_element_latex
    )
    GradedAlgebraComponent._latex_ = (
        _graded_component_latex
    )
    GradedAlgebraStructure._latex_ = (
        _graded_structure_latex
    )
    GradedAlgebraMorphism._latex_ = (
        _graded_morphism_latex
    )
    GradedAlgebraComponentMorphism._latex_ = (
        _graded_component_morphism_latex
    )
    _ProjectiveAutomorphismGroup._latex_ = (
        _automorphism_group_latex
    )
    FiniteSchemeGroupAction._latex_ = (
        _finite_action_latex
    )
    ProductProjectiveCoordinateLinearization._latex_ = (
        _linearization_latex
    )
    _representation_module.Representation_abstract._latex_ = (
        _representation_latex
    )
    IsotypicComponent._latex_ = (
        _isotypic_component_latex
    )
    IsotypicDecomposition._latex_ = (
        _isotypic_decomposition_latex
    )
    AffineQuasiCoherentFreeModule._latex_ = (
        _affine_qcoherent_module_latex
    )
    StandardAffineCover._latex_ = _affine_cover_latex
    AffineLocalRing._latex_ = _affine_local_ring_latex
    AffineLocalMaximalIdeal._latex_ = (
        _affine_local_maximal_ideal_latex
    )
    AffineLocalResidueMap._latex_ = (
        _affine_local_residue_map_latex
    )
    LocalHypersurfacePresentation._latex_ = (
        _local_hypersurface_presentation_latex
    )
    LocalSchemeGerm._latex_ = _local_scheme_germ_latex
    FiniteReducedSubschemeLineBundle._latex_ = (
        _finite_restriction_line_bundle_latex
    )
    FiniteReducedSubschemeGlobalSections._latex_ = (
        _finite_restriction_sections_latex
    )
    SectionRestrictionMorphism._latex_ = (
        _section_restriction_morphism_latex
    )
    CompleteLinearSystem._latex_ = (
        _complete_linear_system_latex
    )
    SchemePullbackDiagram._latex_ = _pullback_diagram_latex
    SplitRankTwoProjectiveBundle._latex_ = (
        _split_projective_bundle_latex
    )
    CyclicCoverMorphism._latex_ = (
        _cyclic_cover_morphism_latex
    )
    CyclicCoverAlgebraDatum._latex_ = (
        _cyclic_cover_algebra_datum_latex
    )
    CyclicCoverDatum._latex_ = _cyclic_cover_datum_latex

    latex_classes = (
        ProjectiveSpace_ring,
        ProductProjectiveSpaces_ring,
        AffineSpace_generic,
        AffineScheme,
        _generic_morphism_module.SchemeMorphism,
        ProductProjectiveLineBundle,
        ProductProjectivePicardGroup,
        ProductProjectiveGlobalSections,
        ProductProjectiveSectionElement,
        ProductProjectiveCoxRingElement,
        ProductProjectiveCohomologyGroup,
        ProductProjectiveLineBundleCohomology,
        ProductProjectiveCoxRing,
        ProductProjectiveSectionRing,
        ProductProjectiveSectionRingElement,
        GradedAlgebraComponent,
        GradedAlgebraStructure,
        GradedAlgebraMorphism,
        GradedAlgebraComponentMorphism,
        _ProjectiveAutomorphismGroup,
        FiniteSchemeGroupAction,
        ProductProjectiveCoordinateLinearization,
        _representation_module.Representation_abstract,
        IsotypicComponent,
        IsotypicDecomposition,
        AffineQuasiCoherentFreeModule,
        StandardAffineCover,
        AffineLocalRing,
        AffineLocalRingElement,
        AffineLocalMaximalIdeal,
        AffineLocalResidueMap,
        LocalHypersurfacePresentation,
        LocalSchemeGerm,
        FiniteReducedSubschemeLineBundle,
        FiniteReducedSubschemeGlobalSections,
        SectionRestrictionMorphism,
        LinearSystem,
        CompleteLinearSystem,
        AffineLinearSystemFamily,
        AffineCyclicCoverFamily,
        AffineCyclicCoverChart,
        CertifiedQuotientLocalization,
        CertifiedQuotientLocalizationElement,
        CoveredScheme,
        CoveredSchemeOverlap,
        CoveredSchemeMorphism,
        CoveredSchemeChartwiseMorphism,
        CoveredDiagonalSignAutomorphism,
        DiagonalSignInvariantQuotientChart,
        DiagonalSignQuotientFamily,
        SchemePullbackDiagram,
        SchemeProductDiagram,
        SplitRankTwoProjectiveBundle,
        CyclicCoverMorphism,
        CyclicCoverAlgebraDatum,
        CyclicCoverDatum,
    )
    for latex_class in latex_classes:
        latex_class._repr_latex_ = _ipython_latex_repr
        latex_class._rich_repr_ = _sage_rich_latex_repr


_install_normalized_latex_display()

print('Installed structured Sage LaTeX display for framework objects and morphisms.')

Installed structured Sage LaTeX display for framework objects and morphisms.


## Picard, Cox, cohomology, representation, and base-change regressions

The next cell tests the semantic interfaces independently of the research narrative. In particular it verifies:

- the abstract Cox algebra and its chosen graded-polynomial model;
- general degree restriction of a graded-algebra isomorphism;
- `L.H(i)` and Poincaré series;
- representation-owned isotypic decomposition;
- base change along an explicit affine-scheme morphism;
- functorial pullback on Picard groups and $H^0$;
- Veronese section-ring multiplication and intersection theory.

In [28]:
import os as _os
if not _os.environ.get('PROJECTIVE_SCHEME_FRAMEWORK_SKIP_REGRESSIONS'):
    exec(r'''P1_sections_x = ProjectiveSpace(
    CC,
    1,
    names=('sx0', 'sx1'),
)
P1_sections_y = ProjectiveSpace(
    CC,
    1,
    names=('sy0', 'sy1'),
)
X_sections = P1_sections_x * P1_sections_y
X_sections.set_latex_name('X')
sx0, sx1, sy0, sy1 = X_sections.gens()

Pic_X_sections = X_sections.Pic()
L44_sections = X_sections.O(4, 4)
M12_sections = X_sections.O(1, 2)
K_X_sections = X_sections.canonical_bundle()
Hstar_L44_sections = L44_sections.cohomology()
H0_L44_sections = L44_sections.H(0)
Hstar_K_sections = K_X_sections.cohomology()

Cox_X_sections = X_sections.cox_ring()
Phi_Cox_sections = Cox_X_sections.polynomial_isomorphism()
Phi_L44_sections = Phi_Cox_sections.restrict_degree(
    L44_sections
)
polynomial_model_sections = Phi_Cox_sections.codomain()

latex_formatter_sections = get_ipython().display_formatter
for display_object in (
    Pic_X_sections,
    L44_sections,
    Hstar_L44_sections,
    H0_L44_sections,
    Cox_X_sections,
    Phi_Cox_sections,
    Phi_L44_sections,
    H0_L44_sections.basis()[0],
    Cox_X_sections.gens()[0],
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

assert str(latex(Pic_X_sections)) == r'\operatorname{Pic}\!\left(X\right)'
assert str(latex(L44_sections)) == r'\mathcal O_{X}\!\left(4,4\right)'
H0_L44_latex = str(latex(H0_L44_sections))
assert H0_L44_latex.startswith(
    r'\begin{aligned}H^{0}\!\left('
)
assert r'\dim H^{0}' in H0_L44_latex
assert r's_{24}' in H0_L44_latex
assert str(latex(Cox_X_sections)) == r'\operatorname{Cox}\!\left(X\right)'

expected_basis_L44 = tuple(
    sx0**(4 - i) * sx1**i * sy0**(4 - j) * sy1**j
    for i in range(5)
    for j in range(5)
)

assert L44_sections.parent() is Pic_X_sections
assert L44_sections.bidegree() == (4, 4)
assert K_X_sections == X_sections.O(-2, -2)
assert Hstar_L44_sections.dimensions() == (25, 0, 0)
assert L44_sections.H(0) is H0_L44_sections
assert L44_sections.H(1).dimension() == 0
assert L44_sections.H(2).dimension() == 0
assert H0_L44_sections.dimension() == 25
assert H0_L44_sections.basis().cardinality() == 25
assert Cox_X_sections.category().is_subcategory(
    Algebras(CC).Commutative().WithBasis()
)
assert polynomial_model_sections is X_sections.coordinate_ring()
assert Phi_Cox_sections is Cox_X_sections.polynomial_isomorphism()
assert Phi_L44_sections is Phi_Cox_sections.restrict_degree(
    L44_sections
)
assert Phi_L44_sections.domain() is H0_L44_sections
assert tuple(
    section.to_polynomial()
    for section in H0_L44_sections.basis()
) == expected_basis_L44

sample_section_L44 = (
    H0_L44_sections.basis()[0]
    + 2 * H0_L44_sections.basis()[1]
)
assert Phi_L44_sections.preimage(
    Phi_L44_sections(sample_section_L44)
) == sample_section_L44
assert H0_L44_sections.from_polynomial(
    sample_section_L44.to_polynomial()
) == sample_section_L44

linear_system_L44_sections = (
    L44_sections.complete_linear_system()
)
phi_L44_sections = linear_system_L44_sections.morphism()
assert linear_system_L44_sections.dimension() == 24
assert linear_system_L44_sections.is_basepoint_free()
assert linear_system_L44_sections.base_locus().defining_ideal().is_one()
assert phi_L44_sections.codomain().dimension_relative() == 24
assert phi_L44_sections.codomain().O(1).pullback(
    phi_L44_sections
) == L44_sections

M22_sections = X_sections.O(2, 2)
cyclic_datum_sections = M22_sections.cyclic_cover_datum(
    sample_section_L44,
    2,
)
assert cyclic_datum_sections.degree() == 2
assert cyclic_datum_sections.branch_line_bundle() == L44_sections
assert cyclic_datum_sections.cover_algebra_datum().pieces() == (
    X_sections.O(0, 0),
    -M22_sections,
)
assert cyclic_datum_sections.cover_algebra_datum().product_degree(
    1,
    1,
) == (0, True)
assert cyclic_datum_sections.branch_subscheme().dimension() == 1

cyclic_morphism_sections = M22_sections.cyclic_cover(
    sample_section_L44,
    2,
)
cyclic_projective_bundle_sections = (
    cyclic_morphism_sections.projective_bundle()
)
cyclic_deck_sections = (
    cyclic_morphism_sections.deck_transformation()
)
assert isinstance(
    cyclic_morphism_sections,
    CyclicCoverMorphism,
)
assert cyclic_morphism_sections.domain().dimension() == 2
assert cyclic_morphism_sections.codomain() == X_sections
assert cyclic_morphism_sections.cover_degree() == 2
assert cyclic_morphism_sections.is_finite()
assert cyclic_morphism_sections.branch_subscheme() == (
    cyclic_datum_sections.branch_subscheme()
)
assert cyclic_morphism_sections.ramification_subscheme().dimension() == 1
assert cyclic_projective_bundle_sections.fan().is_complete()
assert cyclic_projective_bundle_sections.fan().is_smooth()
assert cyclic_projective_bundle_sections.scheme().is_homogeneous(
    cyclic_morphism_sections.cover_equation()
)
assert _morphisms_have_identical_coordinates(
    cyclic_deck_sections * cyclic_deck_sections,
    cyclic_morphism_sections.domain().identity_morphism(),
)
assert _morphisms_have_identical_coordinates(
    cyclic_morphism_sections * cyclic_deck_sections,
    cyclic_morphism_sections,
)

tau_lift_sections = X_sections.hom(
    [
        sx0,
        -sx1,
        sy0,
        -sy1,
    ],
    X_sections,
)
invariant_branch_lift_sections = (
    H0_L44_sections.basis()[0]
    + H0_L44_sections.basis()[2]
    + H0_L44_sections.basis()[4]
    + H0_L44_sections.basis()[20]
    + H0_L44_sections.basis()[24]
)
cyclic_lift_morphism_sections = M22_sections.cyclic_cover(
    invariant_branch_lift_sections,
    2,
)
cyclic_lift_deck_sections = (
    cyclic_lift_morphism_sections.deck_transformation()
)
lifted_tau_sections = (
    cyclic_lift_morphism_sections.lift_automorphisms(
        tau_lift_sections
    )
)
assert cyclic_lift_morphism_sections.branch_scaling(
    tau_lift_sections
) == 1
assert len(lifted_tau_sections) == 2
assert {
    lift._cyclic_cover_fiber_scalar
    for lift in lifted_tau_sections
} == {CC(1), CC(-1)}
for lift in lifted_tau_sections:
    assert _morphisms_have_identical_coordinates(
        cyclic_lift_morphism_sections * lift,
        tau_lift_sections * cyclic_lift_morphism_sections,
    )
    assert _morphisms_have_identical_coordinates(
        lift * lift,
        cyclic_lift_morphism_sections.domain().identity_morphism(),
    )
deck_composite_lift_sections = (
    cyclic_lift_deck_sections * lifted_tau_sections[0]
)
assert any(
    _morphisms_have_identical_coordinates(
        deck_composite_lift_sections,
        lift,
    )
    for lift in lifted_tau_sections
)
assert not _morphisms_have_identical_coordinates(
    deck_composite_lift_sections,
    lifted_tau_sections[0],
)
for lift in lifted_tau_sections + (
    cyclic_lift_deck_sections,
):
    assert isinstance(lift, CyclicCoverLiftMorphism)
    assert lift.is_automorphism()
    assert _morphisms_have_identical_coordinates(
        lift.inverse() * lift,
        cyclic_lift_morphism_sections.domain().identity_morphism(),
    )

try:
    cyclic_lift_deck_sections.fixed_subscheme()
except NotImplementedError:
    pass
else:
    raise AssertionError(
        'the inexact-field fixed-subscheme backend was used without an exactness certificate'
    )
try:
    cyclic_morphism_sections.lift_automorphisms(
        tau_lift_sections
    )
except ValueError:
    pass
else:
    raise AssertionError(
        'a base automorphism was lifted despite not preserving the branch section up to scalar'
    )
try:
    M22_sections.cyclic_cover_datum(
        M22_sections.H(0).basis()[0],
        2,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        'an invalid cyclic-cover nth-root datum was accepted'
    )

for display_object in (
    linear_system_L44_sections,
    cyclic_datum_sections,
    cyclic_datum_sections.cover_algebra_datum(),
    cyclic_projective_bundle_sections,
    cyclic_morphism_sections,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

cox_generators_sections = Cox_X_sections.gens()
assert (
    Phi_Cox_sections(
        cox_generators_sections[0]
        * cox_generators_sections[-1]
    )
    == Phi_Cox_sections(cox_generators_sections[0])
       * Phi_Cox_sections(cox_generators_sections[-1])
)
assert Cox_X_sections.homogeneous_degree(
    cox_generators_sections[0]
    * cox_generators_sections[-1]
) == X_sections.O(1, 1)

G_tau_sections = CyclicPermutationGroup(2)
tau_sections = X_sections.hom(
    [sx0, -sx1, sy0, -sy1],
    X_sections,
)
action_tau_sections = X_sections.Aut().action(
    G_tau_sections,
    [tau_sections],
)
linearized_L44_sections = L44_sections.linearize(
    action_tau_sections
)
representation_L44_sections = (
    linearized_L44_sections.H_representation(0)
)
isotypic_L44_sections = (
    representation_L44_sections.isotypic_decomposition()
)
trivial_component_sections = (
    isotypic_L44_sections.trivial_component()
)
nontrivial_components_sections = (
    isotypic_L44_sections.nontrivial_components()
)

assert representation_L44_sections.underlying_module() is (
    H0_L44_sections
)
assert len(isotypic_L44_sections) == 2
assert len(nontrivial_components_sections) == 1
assert trivial_component_sections.dimension() == 13
assert nontrivial_components_sections[0].dimension() == 12
assert isotypic_L44_sections.dimension() == 25
assert sorted(
    component.dimension()
    for component in isotypic_L44_sections
) == [12, 13]
for basis_element in trivial_component_sections.basis():
    assert linearized_L44_sections.act_on_section(
        G_tau_sections.gen(),
        basis_element,
    ) == basis_element
for basis_element in nontrivial_components_sections[0].basis():
    assert linearized_L44_sections.act_on_section(
        G_tau_sections.gen(),
        basis_element,
    ) == -basis_element

assert Hstar_K_sections.dimensions() == (0, 0, 1)
assert K_X_sections.H(2).dimension() == 1
assert Hstar_K_sections.poincare_series() == (
    PowerSeriesRing(ZZ, 't').gen()**2
)

structure_map_generic_sections = X_sections.structure_morphism()
pushforward_L44_sections = structure_map_generic_sections.pushforward(
    L44_sections
)
parameter_module_L44_sections = pushforward_L44_sections.dual()
section_parameter_space = VV(parameter_module_L44_sections)
eta_section = section_parameter_space.generic_point()
K_section = section_parameter_space.function_field()
q_section = eta_section.domain().base_morphism()
projection_section = section_parameter_space.projection()

for display_object in (
    structure_map_generic_sections,
    pushforward_L44_sections,
    parameter_module_L44_sections,
    section_parameter_space,
    projection_section,
    eta_section,
    q_section,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

assert str(latex(X_sections)) == 'X'
assert str(latex(structure_map_generic_sections)).startswith(
    r'\begin{aligned}p:\;X&\longrightarrow '
)
assert str(latex(structure_map_generic_sections.domain())) in (
    str(latex(structure_map_generic_sections))
)
assert str(latex(structure_map_generic_sections.codomain())) in (
    str(latex(structure_map_generic_sections))
)
assert str(latex(projection_section)).startswith(
    r'\begin{aligned}\pi:\;'
)
assert str(latex(eta_section)).startswith(
    r'\begin{aligned}\eta:\;'
)
assert str(latex(q_section)).startswith(
    r'\begin{aligned}q:\;'
)
for morphism in (
    projection_section,
    eta_section,
    q_section,
):
    morphism_latex = str(latex(morphism))
    assert str(latex(morphism.domain())) in morphism_latex
    assert str(latex(morphism.codomain())) in morphism_latex
assert 'Ring morphism' not in str(latex(eta_section))
assert 'Defn' not in str(latex(structure_map_generic_sections))
assert r'\Gamma' not in str(latex(structure_map_generic_sections))
assert r'\Gamma' not in str(latex(section_parameter_space))
assert r'H^{0}' in str(latex(structure_map_generic_sections))
assert r'H^{0}' in str(latex(section_parameter_space))

X_eta_sections = X_sections.base_change(q_section)
L44_eta_sections = L44_sections.base_change(q_section)
H0_L44_eta_sections = L44_eta_sections.H(0)
eta_coefficients = tuple(
    eta_section.ring_homomorphism()(generator)
    for generator in section_parameter_space.coordinate_ring().gens()
)
section_eta = H0_L44_eta_sections.from_vector(
    vector(K_section, eta_coefficients)
)
F_eta = section_eta.to_polynomial()

assert section_parameter_space.base_scheme() == (
    structure_map_generic_sections.codomain()
)
assert X_eta_sections.base_scheme() == q_section.domain()
assert L44_eta_sections.scheme().base_scheme() == q_section.domain()
assert H0_L44_eta_sections.dimension() == 25
assert tuple(section_eta.to_vector()) == eta_coefficients
assert set(F_eta.coefficients()) == set(eta_coefficients)
assert len(F_eta.monomials()) == 25

P1_pullback_a = PP(
    QQ,
    1,
    names=('pa0_pullback', 'pa1_pullback'),
)
P1_pullback_b = PP(
    QQ,
    1,
    names=('pb0_pullback', 'pb1_pullback'),
)
X_pullback = P1_pullback_a * P1_pullback_b
Y_pullback = PP(
    QQ,
    1,
    names=('y0_pullback', 'y1_pullback'),
)
Z_pullback = PP(
    QQ,
    1,
    names=('z0_pullback', 'z1_pullback'),
)
pa0_pullback, pa1_pullback, pb0_pullback, pb1_pullback = X_pullback.gens()
y0_pullback, y1_pullback = Y_pullback.gens()
z0_pullback, z1_pullback = Z_pullback.gens()

f_pullback = X_pullback.hom(
    [pa0_pullback, pa1_pullback],
    Y_pullback,
)
g_pullback = Y_pullback.hom(
    [y0_pullback**2, y1_pullback**2],
    Z_pullback,
)
h_pullback = g_pullback * f_pullback

H_Z_pullback = Z_pullback.O(1)
f_star_Pic = Y_pullback.Pic().pullback(f_pullback)
g_star_Pic = Z_pullback.Pic().pullback(g_pullback)
h_star_Pic = Z_pullback.Pic().pullback(h_pullback)
assert g_star_Pic(H_Z_pullback) == Y_pullback.O(2)
assert f_star_Pic(g_star_Pic(H_Z_pullback)) == X_pullback.O(2, 0)
assert h_star_Pic(H_Z_pullback) == X_pullback.O(2, 0)

H0_Z_pullback = H_Z_pullback.H(0)
g_star_H0 = H0_Z_pullback.pullback(g_pullback)
f_star_H0 = g_star_H0.codomain().pullback(f_pullback)
h_star_H0 = H0_Z_pullback.pullback(h_pullback)
section_pullback = H0_Z_pullback.from_polynomial(
    z0_pullback + 3 * z1_pullback
)
assert f_star_H0(g_star_H0(section_pullback)) == (
    h_star_H0(section_pullback)
)
assert h_star_H0(section_pullback).to_polynomial() == (
    pa0_pullback**2 + 3 * pa1_pullback**2
)

P1_restriction_x = PP(
    QQ,
    1,
    names=('rx0_restriction', 'rx1_restriction'),
)
P1_restriction_y = PP(
    QQ,
    1,
    names=('ry0_restriction', 'ry1_restriction'),
)
X_restriction = P1_restriction_x * P1_restriction_y
rx0_restriction, rx1_restriction, ry0_restriction, ry1_restriction = (
    X_restriction.gens()
)
L22_restriction = X_restriction.O(2, 2)
H0_L22_restriction = L22_restriction.H(0)
Z_restriction = X_restriction.subscheme([
    rx0_restriction * rx1_restriction,
    ry0_restriction * ry1_restriction,
])
i_restriction = Z_restriction.embedding_morphism()
L22_on_Z = L22_restriction.pullback(i_restriction)
restriction_H0 = H0_L22_restriction.pullback(i_restriction)

assert L22_on_Z.source_line_bundle() is L22_restriction
assert L22_on_Z.base_morphism() is i_restriction
assert L22_on_Z.scheme() == Z_restriction
assert restriction_H0.domain() is H0_L22_restriction
assert restriction_H0.base_morphism() is i_restriction
assert restriction_H0.codomain() is L22_on_Z.H(0)
assert restriction_H0.codomain().dimension() == 4
assert restriction_H0.rank() == 4
assert restriction_H0.kernel().dimension() == 5
assert restriction_H0.image().dimension() == 4
assert restriction_H0.cokernel().dimension() == 0
assert restriction_H0.matrix().nrows() == 4
assert restriction_H0.matrix().ncols() == 9
assert len(restriction_H0.trivialization_labels()) == 4
for display_object in (
    L22_on_Z,
    restriction_H0.codomain(),
    restriction_H0,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

P1_local_x = PP(
    QQ,
    1,
    names=('lx0_local', 'lx1_local'),
)
P1_local_y = PP(
    QQ,
    1,
    names=('ly0_local', 'ly1_local'),
)
X_local = P1_local_x * P1_local_y
lx0_local, lx1_local, ly0_local, ly1_local = X_local.gens()

L22_subsystem_local = X_local.O(2, 2)
tau_subsystem_local = X_local.hom(
    [
        lx0_local,
        -lx1_local,
        ly0_local,
        -ly1_local,
    ],
    X_local,
)
G_subsystem_local = CyclicPermutationGroup(2)
action_subsystem_local = X_local.Aut().action(
    G_subsystem_local,
    [tau_subsystem_local],
)
linearized_L22_subsystem_local = (
    L22_subsystem_local.linearize(
        action_subsystem_local
    )
)
rho_L22_subsystem_local = (
    linearized_L22_subsystem_local.H_representation(0)
)
V_plus_L22_subsystem_local = (
    rho_L22_subsystem_local.isotypic_decomposition()
    .trivial_component()
)
linear_system_V_plus_local = (
    L22_subsystem_local.linear_system(
        V_plus_L22_subsystem_local,
        coordinate_names=(
            'A_subsystem',
            'B_subsystem',
            'C_subsystem',
            'D_subsystem',
            'E_subsystem',
        ),
        projective_latex_name=r'\mathbf P(V_+^\vee)',
        morphism_latex_name=r'q_{V_+}',
    )
)
q_V_plus_local = linear_system_V_plus_local.morphism()
assert linear_system_V_plus_local.line_bundle() is (
    L22_subsystem_local
)
assert linear_system_V_plus_local.section_space() is (
    V_plus_L22_subsystem_local
)
assert linear_system_V_plus_local.ambient_section_space() is (
    L22_subsystem_local.H(0)
)
assert linear_system_V_plus_local.dimension() == 4
assert not linear_system_V_plus_local.is_complete()
assert linear_system_V_plus_local.is_basepoint_free()
assert linear_system_V_plus_local.base_locus().defining_ideal().is_one()
assert q_V_plus_local.codomain().dimension_relative() == 4
assert q_V_plus_local.codomain().O(1).pullback(
    q_V_plus_local
) == L22_subsystem_local
assert tuple(
    section.to_polynomial()
    for section in linear_system_V_plus_local.basis()
) == (
    lx0_local**2 * ly0_local**2,
    lx0_local**2 * ly1_local**2,
    lx0_local * lx1_local * ly0_local * ly1_local,
    lx1_local**2 * ly0_local**2,
    lx1_local**2 * ly1_local**2,
)
for display_object in (
    linear_system_V_plus_local,
    q_V_plus_local,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

A1_product_local = AA(
    QQ,
    1,
    names=('u_product_local',),
)
A2_product_local = AA(
    QQ,
    2,
    names=('v_product_local', 'w_product_local'),
)
affine_product_local = A1_product_local.product(
    A2_product_local
)
assert affine_product_local.backend() == (
    'affine-presentation tensor product'
)
assert affine_product_local.apex().dimension() == 3

P1_repeat_left_local = PP(
    QQ,
    1,
    names=('z0_repeat_local', 'z1_repeat_local'),
)
P1_repeat_right_local = PP(
    QQ,
    1,
    names=('z0_repeat_local', 'z1_repeat_local'),
)
projective_product_local = (
    P1_repeat_left_local.product(
        P1_repeat_right_local
    )
)
assert projective_product_local.backend() == (
    'projective-presentation product'
)
assert projective_product_local.apex().dimension() == 2

T_mixed_local = AA(
    QQ,
    2,
    names=('a_mixed_local', 'b_mixed_local'),
)
mixed_product_local = X_local.product(T_mixed_local)
assert mixed_product_local.backend() == (
    'affine-base-change product'
)
assert mixed_product_local.apex().base_scheme() is (
    T_mixed_local
)
assert mixed_product_local.left_projection().codomain() is (
    X_local
)
assert mixed_product_local.right_projection().codomain() is (
    T_mixed_local
)

L11_family_local = X_local.O(1, 1)
affine_family_L11_local = (
    L11_family_local.complete_linear_system()
    .affine_family(
        parameter_names=(
            'c0_family_local',
            'c1_family_local',
            'c2_family_local',
            'c3_family_local',
        )
    )
)
c0_family_local, c1_family_local, c2_family_local, c3_family_local = (
    affine_family_L11_local.parameter_coordinates()
)
assert affine_family_L11_local.parameter_space().dimension() == 4
assert affine_family_L11_local.product_diagram().backend() == (
    'affine-base-change product'
)
assert tuple(
    affine_family_L11_local.universal_section().to_vector()
) == affine_family_L11_local.parameter_coordinates()
assert affine_family_L11_local.projection().codomain() is (
    affine_family_L11_local.parameter_space()
)
assert affine_family_L11_local.discriminant_ideal() == (
    affine_family_L11_local.coefficient_ring().ideal([
        c1_family_local * c2_family_local
        - c0_family_local * c3_family_local
    ])
)
assert affine_family_L11_local.relative_singular_locus().base_scheme() is (
    affine_family_L11_local.parameter_space()
)

family_V_plus_local = (
    linear_system_V_plus_local.affine_family(
        parameter_names=(
            'a00_family_local',
            'a02_family_local',
            'a11_family_local',
            'a20_family_local',
            'a22_family_local',
        )
    )
)
Fix_tau_subsystem_local = (
    tau_subsystem_local.fixed_subscheme()
)
avoidance_V_plus_local = (
    family_V_plus_local.avoidance_polynomial(
        Fix_tau_subsystem_local.embedding_morphism()
    )
)
assert family_V_plus_local.parameter_space().dimension() == 5
assert avoidance_V_plus_local != 0
for display_object in (
    affine_product_local,
    projective_product_local,
    mixed_product_local,
    affine_family_L11_local,
    family_V_plus_local,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

cyclic_family_V_plus_local = (
    linear_system_V_plus_local.cyclic_cover_family(
        X_local.O(1, 1),
        2,
        parameter_names=(
            'a00_cover_family_local',
            'a02_cover_family_local',
            'a11_cover_family_local',
            'a20_cover_family_local',
            'a22_cover_family_local',
        ),
    )
)
covered_scheme_V_plus_local = (
    cyclic_family_V_plus_local.cover_scheme()
)
covering_morphism_V_plus_local = (
    cyclic_family_V_plus_local.covering_morphism()
)
family_morphism_V_plus_local = (
    cyclic_family_V_plus_local.family_morphism()
)
assert len(cyclic_family_V_plus_local.charts()) == 4
assert len(cyclic_family_V_plus_local.overlaps()) == 6
assert covered_scheme_V_plus_local.gluing_verified()
assert covered_scheme_V_plus_local.is_separated()
assert covered_scheme_V_plus_local.base_scheme() is (
    cyclic_family_V_plus_local.parameter_space()
)
assert covering_morphism_V_plus_local.domain() is (
    covered_scheme_V_plus_local
)
assert covering_morphism_V_plus_local.codomain() is (
    cyclic_family_V_plus_local.divisor_family().ambient_family()
)
assert covering_morphism_V_plus_local.is_compatible()
assert family_morphism_V_plus_local.domain() is (
    covered_scheme_V_plus_local
)
assert family_morphism_V_plus_local.codomain() is (
    cyclic_family_V_plus_local.parameter_space()
)
assert family_morphism_V_plus_local.is_compatible()
assert covered_scheme_V_plus_local.base_morphism() is (
    family_morphism_V_plus_local
)
assert cyclic_family_V_plus_local.global_morphism() is (
    covering_morphism_V_plus_local
)
branch_section_V_plus_local = (
    V_plus_L22_subsystem_local.basis()[0]
    + V_plus_L22_subsystem_local.basis()[1]
    + V_plus_L22_subsystem_local.basis()[2]
    + V_plus_L22_subsystem_local.basis()[3]
    + V_plus_L22_subsystem_local.basis()[4]
)
parameter_point_V_plus_local = (
    cyclic_family_V_plus_local.parameter_point(
        branch_section_V_plus_local
    )
)
specialized_branch_V_plus_local = (
    cyclic_family_V_plus_local.specialize_branch_section(
        parameter_point_V_plus_local
    )
)
specialized_cover_V_plus_local = (
    cyclic_family_V_plus_local.specialize(
        parameter_point_V_plus_local
    )
)
assert specialized_branch_V_plus_local == (
    branch_section_V_plus_local
)
assert cyclic_family_V_plus_local.divisor_family().fiber_divisor(
    parameter_point_V_plus_local
) == branch_section_V_plus_local.zero_subscheme()
assert specialized_cover_V_plus_local.cover_degree() == 2
assert specialized_cover_V_plus_local.codomain() is X_local
assert specialized_cover_V_plus_local.cyclic_cover_datum().branch_section() == (
    branch_section_V_plus_local
)
for display_object in (
    cyclic_family_V_plus_local,
    covered_scheme_V_plus_local,
    covering_morphism_V_plus_local,
    family_morphism_V_plus_local,
) + cyclic_family_V_plus_local.overlaps():
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

constant_quotient_bundle_local = X_local.O(0, 0)
constant_quotient_linear_system_local = (
    constant_quotient_bundle_local.complete_linear_system()
)
cyclic_quotient_family_local = (
    constant_quotient_linear_system_local
    .cyclic_cover_family(
        X_local.O(0, 0),
        2,
        parameter_names=(
            'a_constant_quotient_family_local',
        ),
    )
)
enriques_lift_family_local = (
    cyclic_quotient_family_local.enriques_lift(
        tau_subsystem_local,
        fiber_scalar=-1,
    )
)
enriques_quotient_family_local = (
    enriques_lift_family_local.quotient()
)
covered_enriques_family_local = (
    enriques_quotient_family_local.covered_scheme()
)
enriques_family_projection_local = (
    enriques_quotient_family_local.family_morphism()
)
enriques_quotient_morphism_local = (
    enriques_quotient_family_local.quotient_morphism()
)
assert enriques_lift_family_local.is_automorphism()
assert enriques_lift_family_local.is_compatible()
assert enriques_lift_family_local.inverse() is (
    enriques_lift_family_local
)
assert len(enriques_quotient_family_local.charts()) == 4
assert len(enriques_quotient_family_local.overlaps()) == 6
assert covered_enriques_family_local.dimension() == 3
assert covered_enriques_family_local.gluing_verified()
assert covered_enriques_family_local.is_separated()
assert covered_enriques_family_local.base_scheme() is (
    cyclic_quotient_family_local.parameter_space()
)
assert enriques_family_projection_local.is_compatible()
assert covered_enriques_family_local.base_morphism() is (
    enriques_family_projection_local
)
assert enriques_quotient_morphism_local.domain() is (
    cyclic_quotient_family_local.cover_scheme()
)
assert enriques_quotient_morphism_local.codomain() is (
    covered_enriques_family_local
)
assert enriques_quotient_morphism_local.is_compatible()
for quotient_chart_local in (
    enriques_quotient_family_local.charts()
):
    assert quotient_chart_local.quotient_morphism().domain() is (
        quotient_chart_local.cover_chart().cover_scheme()
    )
    assert quotient_chart_local.coordinate_ring().krull_dimension() == 3
    assert quotient_chart_local.coordinate_ring().is_integral_domain()
for quotient_overlap_local in (
    enriques_quotient_family_local.overlaps()
):
    certificate = quotient_overlap_local.certificate()
    assert 'cover_factor_morphism' in certificate
    assert certificate['cover_factor_morphism'].domain() is (
        certificate['cover_overlap'].scheme()
    )
for display_object in (
    enriques_lift_family_local,
    enriques_quotient_family_local,
    covered_enriques_family_local,
    enriques_family_projection_local,
    enriques_quotient_morphism_local,
) + enriques_quotient_family_local.charts() + (
    enriques_quotient_family_local.overlaps()
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

C_A1_local = X_local.subscheme([
    lx1_local**2 * ly0_local**2
    + lx0_local**2 * ly1_local**2
])
p_A1_local = C_A1_local((1, 0, 1, 0))
germ_A1_local = p_A1_local.local_germ()
assert p_A1_local.is_singular()
assert p_A1_local.ADE_type() == 'A1'
assert p_A1_local.milnor_number() == 1
assert p_A1_local.tjurina_number() == 1
assert germ_A1_local.embedding_dimension() == 2
assert germ_A1_local.local_dimension() == 1
assert p_A1_local.equation_of_normal_form() == (
    p_A1_local.equation_of_normal_form().parent().gen(0)**2
    + p_A1_local.equation_of_normal_form().parent().gen(1)**2
)

C_A3_local = X_local.subscheme([
    lx0_local**2 * lx1_local**2 * ly0_local**4
    + lx0_local**4 * ly1_local**4
    + lx1_local**4 * ly0_local**4
    + lx1_local**4 * ly1_local**4
])
p_A3_local = C_A3_local((1, 0, 1, 0))
germ_A3_local = p_A3_local.local_germ()
assert p_A3_local.is_singular()
assert p_A3_local.ADE_type() == 'A3'
assert p_A3_local.milnor_number() == 3
assert p_A3_local.tjurina_number() == 3
assert germ_A3_local.hessian_at_origin().rank() == 1
for display_object in (
    germ_A1_local,
    germ_A3_local,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

higher_corank_polynomials_local = {
    'D4': (
        lx1_local**2 * ly1_local * ly0_local**2
        + lx0_local**2 * ly1_local**3
    ),
    'D5': (
        lx1_local**2 * ly1_local * ly0_local**3
        + lx0_local**2 * ly1_local**4
    ),
    'E6': (
        lx1_local**3 * ly0_local**4
        + lx0_local**3 * ly1_local**4
    ),
    'E7': (
        lx1_local**3 * ly0_local**3
        + lx1_local * lx0_local**2 * ly1_local**3
    ),
    'E8': (
        lx1_local**3 * ly0_local**5
        + lx0_local**3 * ly1_local**5
    ),
}
higher_corank_points_local = {}
for singularity_type, polynomial in (
    higher_corank_polynomials_local.items()
):
    curve = X_local.subscheme([polynomial])
    point = curve((1, 0, 1, 0))
    germ = point.local_germ()
    index = ZZ(singularity_type[1:])
    higher_corank_points_local[singularity_type] = point
    assert point.is_singular()
    assert point.multiplicity() == 3
    assert point.embedding_dim() == 2
    assert germ.hessian_at_origin().rank() == 0
    assert point.milnor_number() == index
    assert point.tjurina_number() == index
    assert point.ADE_type() == singularity_type
    assert point.Milnor_algebra().cover_ring() == (
        point.milnor_algebra().cover_ring()
    )
    assert point.Tjurina_algebra().cover_ring() == (
        point.tjurina_algebra().cover_ring()
    )
    normal_form = point.equation_of_normal_form()
    u_normal, v_normal = normal_form.parent().gens()
    if singularity_type.startswith('D'):
        expected_normal_form = (
            u_normal**2 * v_normal
            + v_normal**(index - 1)
        )
    elif singularity_type == 'E6':
        expected_normal_form = u_normal**3 + v_normal**4
    elif singularity_type == 'E7':
        expected_normal_form = (
            u_normal**3 + u_normal * v_normal**3
        )
    else:
        expected_normal_form = u_normal**3 + v_normal**5
    assert normal_form == expected_normal_form

local_ring_A3 = p_A3_local.local_ring()
assert local_ring_A3 is p_A3_local.local_germ().local_ring()
assert local_ring_A3.scheme() == C_A3_local
assert local_ring_A3.point() == p_A3_local
assert local_ring_A3.dimension() == 1
assert local_ring_A3.embedding_dimension() == 2
assert not local_ring_A3.is_regular()
assert local_ring_A3.is_local()
assert local_ring_A3.is_noetherian()
assert local_ring_A3.is_integral_domain()
assert local_ring_A3.residue_field() is QQ

local_A3_u, local_A3_v = local_ring_A3.gens()
local_A3_unit = 1 + local_A3_u
local_A3_fraction = (
    local_A3_u / (1 + local_A3_v)
)
assert local_A3_u in local_ring_A3.maximal_ideal()
assert local_A3_unit not in local_ring_A3.maximal_ideal()
assert local_A3_unit.is_unit()
assert local_A3_unit * local_A3_unit.inverse() == (
    local_ring_A3.one()
)
assert local_A3_fraction.residue() == 0
assert local_ring_A3.residue_map()(
    local_A3_unit
) == 1
try:
    local_ring_A3.fraction(
        local_A3_u.numerator(),
        local_A3_v.numerator(),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        'a denominator in the maximal ideal was accepted'
    )

reducible_curve_local = X_local.subscheme([
    (lx0_local + lx1_local) * ly1_local
])
reducible_point_local = reducible_curve_local((
    1,
    0,
    1,
    0,
))
reducible_local_ring = reducible_point_local.local_ring()
reducible_local_x, reducible_local_y = (
    reducible_local_ring.gens()
)
assert reducible_local_y == reducible_local_ring.zero()
assert reducible_local_x != reducible_local_ring.zero()
assert reducible_local_ring.is_regular()
assert not reducible_local_ring.is_integral_domain()

nonreduced_curve_local = X_local.subscheme([
    ly1_local**2
])
nonreduced_point_local = nonreduced_curve_local((
    1,
    0,
    1,
    0,
))
nonreduced_local_ring = nonreduced_point_local.local_ring()
nonreduced_local_x, nonreduced_local_y = (
    nonreduced_local_ring.gens()
)
assert nonreduced_local_y != nonreduced_local_ring.zero()
assert nonreduced_local_y**2 == nonreduced_local_ring.zero()
assert not nonreduced_local_ring.is_integral_domain()
assert not nonreduced_local_ring.is_regular()

for display_object in (
    local_ring_A3,
    local_ring_A3.gen(0),
    local_ring_A3.maximal_ideal(),
    local_ring_A3.residue_map(),
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

P4_surface_A1_local = PP(
    QQ,
    4,
    names=(
        'A_surface_A1',
        'B_surface_A1',
        'C_surface_A1',
        'D_surface_A1',
        'E_surface_A1',
    ),
)
(
    A_surface_A1,
    B_surface_A1,
    C_surface_A1,
    D_surface_A1,
    E_surface_A1,
) = P4_surface_A1_local.gens()
W_surface_A1_local = P4_surface_A1_local.subscheme([
    B_surface_A1 * D_surface_A1
    - A_surface_A1 * E_surface_A1,
    C_surface_A1**2
    - A_surface_A1 * E_surface_A1,
])
p_surface_A1_local = W_surface_A1_local((
    1,
    0,
    0,
    0,
    0,
))
germ_surface_A1_local = p_surface_A1_local.local_germ()
hypersurface_surface_A1_local = (
    p_surface_A1_local.hypersurface_presentation()
)
local_ring_surface_A1 = p_surface_A1_local.local_ring()
assert germ_surface_A1_local.local_dimension() == 2
assert germ_surface_A1_local.embedding_dimension() == 3
assert hypersurface_surface_A1_local.dimension() == 2
assert hypersurface_surface_A1_local.embedding_dimension() == 3
assert len(
    hypersurface_surface_A1_local.eliminated_variables()
) == 1
assert p_surface_A1_local.local_equation().parent() is (
    hypersurface_surface_A1_local.ring()
)
assert p_surface_A1_local.local_equation().total_degree() == 2
assert germ_surface_A1_local.hessian_at_origin().rank() == 3
assert p_surface_A1_local.milnor_number() == 1
assert p_surface_A1_local.tjurina_number() == 1
assert p_surface_A1_local.ADE_type() == 'A1'
normal_form_surface_A1 = (
    p_surface_A1_local.equation_of_normal_form()
)
u_surface_A1, v_surface_A1, w_surface_A1 = (
    normal_form_surface_A1.parent().gens()
)
assert normal_form_surface_A1 == (
    u_surface_A1**2
    + v_surface_A1**2
    + w_surface_A1**2
)
assert local_ring_surface_A1.dimension() == 2
assert local_ring_surface_A1.embedding_dimension() == 3
assert not local_ring_surface_A1.is_regular()
assert len(
    W_surface_A1_local.singular_locus().rational_points()
) == 4
for display_object in (
    germ_surface_A1_local,
    hypersurface_surface_A1_local,
    local_ring_surface_A1,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

embedded_Pic_W_surface_A1 = (
    W_surface_A1_local.embedded_picard_group()
)
complete_intersection_W_surface_A1 = (
    W_surface_A1_local.complete_intersection_certificate()
)
O1_W_surface_A1 = W_surface_A1_local.O(1)
K_W_surface_A1 = W_surface_A1_local.canonical_bundle()
minus_K_W_surface_A1 = (
    W_surface_A1_local.anticanonical_bundle()
)
assert embedded_Pic_W_surface_A1.rank() == 1
assert embedded_Pic_W_surface_A1.gen() == O1_W_surface_A1
assert W_surface_A1_local.is_complete_intersection()
assert W_surface_A1_local.complete_intersection_degrees() == (
    2,
    2,
)
assert complete_intersection_W_surface_A1.degree_matches()
assert K_W_surface_A1 == W_surface_A1_local.O(-1)
assert minus_K_W_surface_A1 == O1_W_surface_A1
assert minus_K_W_surface_A1.is_ample()
assert minus_K_W_surface_A1**2 == 4
assert W_surface_A1_local.is_normal()
assert W_surface_A1_local.is_gorenstein()
assert W_surface_A1_local.is_del_Pezzo()
assert W_surface_A1_local.anticanonical_degree() == 4
assert W_surface_A1_local.del_Pezzo_degree() == 4
for display_object in (
    embedded_Pic_W_surface_A1,
    O1_W_surface_A1,
    complete_intersection_W_surface_A1,
    K_W_surface_A1,
    minus_K_W_surface_A1,
):
    display_data, display_metadata = (
        latex_formatter_sections.format(display_object)
    )
    assert 'text/latex' in display_data

smooth_curve_local = X_local.subscheme([
    lx0_local * ly0_local
    + lx1_local * ly1_local
])
singular_locus_smooth_local = (
    smooth_curve_local.singular_locus()
)
singular_locus_A1_local = C_A1_local.singular_locus()
nonreduced_curve_local = X_local.subscheme([
    (
        lx0_local * ly0_local
        + lx1_local * ly1_local
    )**2
])
singular_locus_nonreduced_local = (
    nonreduced_curve_local.singular_locus()
)
finite_regular_local = X_local.subscheme([
    lx0_local * lx1_local,
    ly0_local * ly1_local,
])
singular_locus_finite_local = (
    finite_regular_local.singular_locus()
)
regular_point_local = smooth_curve_local((
    1,
    1,
    1,
    -1,
))

assert singular_locus_smooth_local.defining_ideal().is_one()
assert not smooth_curve_local.is_singular()
assert singular_locus_A1_local.dimension() == 0
assert len(singular_locus_A1_local.rational_points()) == 2
assert C_A1_local.is_singular()
assert singular_locus_nonreduced_local.dimension() == (
    nonreduced_curve_local.dimension()
)
assert nonreduced_curve_local.is_singular()
assert singular_locus_finite_local.defining_ideal().is_one()
assert not finite_regular_local.is_singular()
assert p_A1_local.is_singular() == (
    p_A1_local.local_germ().is_singular()
)
assert regular_point_local.is_singular() == (
    regular_point_local.local_germ().is_singular()
)
assert not regular_point_local.is_singular()

smooth_branch_toric_local = X_local.O(4, 4).H(0).from_polynomial(
    lx0_local**4 * ly0_local**4
    + lx0_local**4 * ly0_local**2 * ly1_local**2
    + lx0_local**4 * ly1_local**4
    + lx1_local**4 * ly0_local**4
    + lx1_local**4 * ly1_local**4
)
smooth_cover_toric_local = X_local.O(2, 2).cyclic_cover(
    smooth_branch_toric_local,
    2,
)
singular_locus_smooth_cover_toric_local = (
    smooth_cover_toric_local.domain().singular_locus()
)
assert (
    singular_locus_smooth_cover_toric_local.defining_ideal().is_one()
)
assert not smooth_cover_toric_local.domain().is_singular()

A1_branch_toric_local = X_local.O(4, 4).H(0).from_polynomial(
    lx0_local**2 * lx1_local**2 * ly0_local**4
    + lx0_local**4 * ly0_local**2 * ly1_local**2
    + lx0_local**4 * ly1_local**4
    + lx1_local**4 * ly0_local**4
    + lx1_local**4 * ly1_local**4
)
A1_cover_toric_local = X_local.O(2, 2).cyclic_cover(
    A1_branch_toric_local,
    2,
)
singular_locus_A1_cover_toric_local = (
    A1_cover_toric_local.domain().singular_locus()
)
assert not singular_locus_A1_cover_toric_local.defining_ideal().is_one()
assert singular_locus_A1_cover_toric_local.dimension() == 0
assert A1_cover_toric_local.domain().is_singular()

tau_cover_toric_local = X_local.hom(
    [
        lx0_local,
        -lx1_local,
        ly0_local,
        -ly1_local,
    ],
    X_local,
)
lifted_tau_toric_local = (
    smooth_cover_toric_local.lift_automorphisms(
        tau_cover_toric_local
    )
)
lifted_tau_by_scalar_toric_local = {
    lift.fiber_scalar(): lift
    for lift in lifted_tau_toric_local
}
deck_toric_local = (
    smooth_cover_toric_local.deck_transformation()
)
fixed_deck_toric_local = (
    deck_toric_local.fixed_subscheme()
)
fixed_En_toric_local = (
    lifted_tau_by_scalar_toric_local[QQ(-1)].fixed_subscheme()
)
fixed_Nik_toric_local = (
    lifted_tau_by_scalar_toric_local[QQ(1)].fixed_subscheme()
)
ramification_saturated_toric_local = (
    smooth_cover_toric_local.ramification_subscheme()
    .defining_ideal()
    .saturation(
        _toric_irrelevant_ideal(
            smooth_cover_toric_local.projective_bundle().scheme()
        )
    )[0]
)
assert fixed_deck_toric_local.defining_ideal() == (
    ramification_saturated_toric_local
)
assert fixed_En_toric_local.defining_ideal().is_one()
assert fixed_Nik_toric_local.dimension() == 0
for lift in lifted_tau_toric_local + (deck_toric_local,):
    assert _morphisms_have_identical_coordinates(
        lift.inverse() * lift,
        smooth_cover_toric_local.domain().identity_morphism(),
    )

Picard_gram_X_sections = Pic_X_sections.intersection_form()
Picard_lattice_X_sections = X_sections.Picard_lattice()
assert Picard_gram_X_sections == matrix(ZZ, [[0, 1], [1, 0]])
assert Picard_lattice_X_sections.gram_matrix() == (
    Picard_gram_X_sections
)
assert L44_sections * M12_sections == 12
assert L44_sections**2 == 32

R_L44_sections = L44_sections.section_ring()
R_L44_generators = R_L44_sections.gens()
assert R_L44_sections.category().is_subcategory(
    Algebras(CC).Commutative()
)
assert len(R_L44_generators) == 25
assert R_L44_sections.graded_piece(1) is H0_L44_sections
assert R_L44_sections.graded_piece(2).dimension() == 81
assert R_L44_generators[0].homogeneous_degree() == 1
assert (
    R_L44_generators[0] * R_L44_generators[-1]
).homogeneous_degree() == 2
assert (
    R_L44_generators[0] * R_L44_generators[-1]
).to_polynomial() == (
    R_L44_generators[0].to_polynomial()
    * R_L44_generators[-1].to_polynomial()
)

P2_sections = ProjectiveSpace(
    QQ,
    2,
    names=('u0_sections', 'u1_sections', 'u2_sections'),
)
H_P2_sections = P2_sections.O(1)
assert P2_sections.Pic().rank() == 1
assert H_P2_sections.H(0).dimension() == 3
assert H_P2_sections**2 == 1
assert P2_sections.canonical_bundle() == P2_sections.O(-3)

P1_product_sections = ProjectiveSpace(
    QQ,
    1,
    names=('v0_sections', 'v1_sections'),
)
P2_product_sections = ProjectiveSpace(
    QQ,
    2,
    names=('w0_sections', 'w1_sections', 'w2_sections'),
)
X12_sections = P1_product_sections * P2_product_sections
D21_sections = X12_sections.O(2, 1)
assert X12_sections.Pic().rank() == 2
assert D21_sections.H(0).dimension() == 9
assert D21_sections.top_self_intersection() == 6

picard_section_regression_summary = {
    'Picard_rank_P1xP1': Pic_X_sections.rank(),
    'O44_bidegree': L44_sections.bidegree(),
    'H0_O44_dimension': H0_L44_sections.dimension(),
    'H0_O44_basis_cardinality': (
        H0_L44_sections.basis().cardinality()
    ),
    'Cox_polynomial_model': polynomial_model_sections,
    'degree_44_restriction_codomain_dimension': (
        Phi_L44_sections.codomain().dimension()
    ),
    'isotypic_dimensions': tuple(
        component.dimension()
        for component in isotypic_L44_sections
    ),
    'generic_section_coefficient_field': K_section,
    'functorial_Picard_pullback': h_star_Pic(H_Z_pullback),
    'functorial_H0_pullback_polynomial': (
        h_star_H0(section_pullback).to_polynomial()
    ),
    'O44_self_intersection': L44_sections**2,
    'Picard_gram_P1xP1': Picard_gram_X_sections,
    'section_ring_degree_two_dimension': (
        R_L44_sections.graded_piece(2).dimension()
    ),
    'complete_linear_system_dimension': (
        linear_system_L44_sections.dimension()
    ),
    'complete_linear_system_target_dimension': (
        phi_L44_sections.codomain().dimension_relative()
    ),
    'cyclic_cover_degree': cyclic_datum_sections.degree(),
    'cyclic_cover_algebra_pieces': (
        cyclic_datum_sections.cover_algebra_datum().pieces()
    ),
    'cyclic_cover_domain_dimension': (
        cyclic_morphism_sections.domain().dimension()
    ),
    'cyclic_cover_ramification_dimension': (
        cyclic_morphism_sections.ramification_subscheme().dimension()
    ),
    'finite_restriction_rank': restriction_H0.rank(),
    'finite_restriction_kernel_dimension': (
        restriction_H0.kernel().dimension()
    ),
    'A1_type': p_A1_local.ADE_type(),
    'A3_type': p_A3_local.ADE_type(),
}

print('Semantic algebraic-geometry regressions passed:')
for key, value in picard_section_regression_summary.items():
    print(key, '=', value)
''', globals())
else:
    picard_section_regression_summary = {'skipped': True}
    print('Skipped Picard, Cox, cohomology, and representation regressions during import.')

Semantic algebraic-geometry regressions passed:
Picard_rank_P1xP1 = 2
O44_bidegree = (4, 4)
H0_O44_dimension = 25
H0_O44_basis_cardinality = 25
Cox_polynomial_model = Multivariate Polynomial Ring in sx0, sx1, sy0, sy1 over Complex Field with 53 bits of precision
degree_44_restriction_codomain_dimension = 25
isotypic_dimensions = (13, 12)
generic_section_coefficient_field = Fraction Field of Multivariate Polynomial Ring in c_0, c_1, c_2, c_3, c_4, c_5, c_6, c_7, c_8, c_9, c_10, c_11, c_12, c_13, c_14, c_15, c_16, c_17, c_18, c_19, c_20, c_21, c_22, c_23, c_24 over Complex Field with 53 bits of precision
functorial_Picard_pullback = O(2, 0) on Product of projective spaces P^1 x P^1 over Rational Field
functorial_H0_pullback_polynomial = pa0_pullback^2 + 3*pa1_pullback^2
O44_self_intersection = 32
Picard_gram_P1xP1 = [0 1]
[1 0]
section_ring_degree_two_dimension = 81
complete_linear_system_dimension = 24
complete_linear_system_target_dimension = 24
cyclic_cover_degree = 2
cyclic_cover_alg

## Constructor and point-interface regression computations

The next cell verifies that the aliases return native Sage spaces and line bundles, that products and powers use Sage's existing constructions, and that points remain morphisms in the rational-point hom-set while admitting affine-chart and local-equation expressions.

In [29]:
import os as _os
if not _os.environ.get('PROJECTIVE_SCHEME_FRAMEWORK_SKIP_REGRESSIONS'):
    P2_alias = PP(
        QQ,
        2,
        names=('ap0', 'ap1', 'ap2'),
    )
    P2_alias_power = PP(
        QQ,
        names=('bp0', 'bp1', 'bp2'),
    )**2
    P2_alias_fixed_dimension = (PP**2)(
        QQ,
        names=('cp0', 'cp1', 'cp2'),
    )

    A3_alias = AA(
        QQ,
        3,
        names=('aa0', 'aa1', 'aa2'),
    )
    A3_alias_power = AA(
        QQ,
        names=('ba0', 'ba1', 'ba2'),
    )**3
    A3_alias_fixed_dimension = (AA**3)(
        QQ,
        names=('ca0', 'ca1', 'ca2'),
    )

    assert isinstance(P2_alias, ProjectiveSpace_ring)
    assert P2_alias.dimension_relative() == 2
    assert P2_alias_power.dimension_relative() == 2
    assert P2_alias_fixed_dimension.dimension_relative() == 2
    assert isinstance(A3_alias, AffineSpace_generic)
    assert A3_alias.dimension_relative() == 3
    assert A3_alias_power.dimension_relative() == 3
    assert A3_alias_fixed_dimension.dimension_relative() == 3

    P1_alias_x = PP(QQ, 1, names=('px0', 'px1'))
    P1_alias_y = PP(QQ, 1, names=('py0', 'py1'))
    X_alias = P1_alias_x * P1_alias_y
    L_alias = OO(X_alias, 4, 4)

    assert OO(X_alias) == X_alias.O(0, 0)
    assert OO(L_alias) is L_alias
    assert OO(X_alias, L_alias) is L_alias
    assert L_alias == X_alias.O(4, 4)
    assert (P1_alias_x**3).n_components() == 3
    assert (X_alias**2).n_components() == 4
    assert (AA(QQ, 2)**3).dimension_relative() == 6

    projective_point_alias = P2_alias((1, 2, 3))
    product_point_alias = X_alias((1, 2, 3, 4))
    affine_point_alias = A3_alias((4, 5, 6))

    assert projective_point_alias.parent() is P2_alias(QQ)
    assert product_point_alias.parent() is X_alias(QQ)
    assert affine_point_alias.parent() is A3_alias(QQ)
    assert not hasattr(P2_alias, 'points_over')
    assert not hasattr(P2_alias, 'point_from')
    assert not hasattr(projective_point_alias, 'factor_coordinates')
    assert not hasattr(projective_point_alias, 'affine_expression')

    projective_cover_alias = P2_alias.affine_cover()
    product_cover_alias = X_alias.affine_cover()
    affine_cover_alias = A3_alias.affine_cover()

    assert len(projective_cover_alias) == 3
    assert projective_cover_alias.labels() == (0, 1, 2)
    assert projective_cover_alias.labels_containing(projective_point_alias) == (0, 1, 2)
    assert projective_cover_alias.canonical_label(projective_point_alias) == 0
    projective_preimage_alias = projective_cover_alias.preimage(
        projective_point_alias
    )
    assert tuple(projective_preimage_alias) == (2, 3)
    assert (
        projective_cover_alias.canonical_chart(projective_point_alias)(
            projective_preimage_alias
        )
        == projective_point_alias
    )

    assert len(product_cover_alias) == 4
    assert product_cover_alias.labels() == (
        (0, 0),
        (0, 1),
        (1, 0),
        (1, 1),
    )
    assert product_cover_alias.labels_containing(product_point_alias) == (
        (0, 0),
        (0, 1),
        (1, 0),
        (1, 1),
    )
    assert product_cover_alias.canonical_label(product_point_alias) == (0, 0)
    product_preimages_alias = tuple(
        product_cover_alias.preimage(product_point_alias, label)
        for label in product_cover_alias.labels_containing(product_point_alias)
    )
    assert tuple(tuple(point) for point in product_preimages_alias) == (
        (QQ(2), QQ(4) / 3),
        (QQ(2), QQ(3) / 4),
        (QQ(1) / 2, QQ(4) / 3),
        (QQ(1) / 2, QQ(3) / 4),
    )

    for chart in product_cover_alias:
        assert chart.is_standard_affine_chart()
        assert chart.is_open_immersion()
        assert chart.cover_label() in product_cover_alias.labels()
        assert chart.contains_point(product_point_alias)
        assert chart(chart.preimage_point(product_point_alias)) == product_point_alias

    assert len(affine_cover_alias) == 1
    assert affine_cover_alias.labels() == (tuple(),)
    assert affine_cover_alias.preimage(affine_point_alias) == affine_point_alias

    product_point_subscheme_alias = product_point_alias.as_subscheme()
    assert product_point_subscheme_alias.dimension() == 0

    constructor_point_regression_summary = {
        'PP_dimension': P2_alias.dimension_relative(),
        'AA_dimension': A3_alias.dimension_relative(),
        'OO_bidegree': L_alias.bidegree(),
        'projective_power_factor_count': (P1_alias_x**3).n_components(),
        'product_power_factor_count': (X_alias**2).n_components(),
        'projective_point_parent': projective_point_alias.parent(),
        'projective_cover_labels': projective_cover_alias.labels(),
        'projective_point_containing_labels': (
            projective_cover_alias.labels_containing(projective_point_alias)
        ),
        'projective_point_canonical_affine_coordinates': tuple(
            projective_preimage_alias
        ),
        'product_point_all_affine_coordinates': tuple(
            tuple(point) for point in product_preimages_alias
        ),
    }

    print('Constructor aliases, standard affine covers, and native point tests passed:')
    for key, value in constructor_point_regression_summary.items():
        print(key, '=', value)
else:
    constructor_point_regression_summary = {'skipped': True}
    print('Skipped constructor, affine-cover, and point regression computations during import.')

Constructor aliases, standard affine covers, and native point tests passed:
PP_dimension = 2
AA_dimension = 3
OO_bidegree = (4, 4)
projective_power_factor_count = 3
product_power_factor_count = 4
projective_point_parent = Set of rational points of Projective Space of dimension 2 over Rational Field
projective_cover_labels = (0, 1, 2)
projective_point_containing_labels = (0, 1, 2)
projective_point_canonical_affine_coordinates = (2, 3)
product_point_all_affine_coordinates = ((2, 4/3), (2, 3/4), (1/2, 4/3), (1/2, 3/4))


## Scope and import contract

This notebook extends Sage's projective spaces, products, hom-sets, morphisms, Picard groups, line bundles, cohomology modules, abstract Cox algebras, section rings, finite group actions, and representations.

The installed public interface includes:

- `f.image()`, `f.graph_morphism()`, `f.pullback(g)`, and `f.fixed_subscheme()`;
- `f.ambient_category().pullback(f,g)` returning a categorical pullback diagram;
- `X.ambient_category().product(X,Y,base=S)` and `X.product(Y,base=S)` with affine, projective-presentation, and affine-base-change backends;
- `f.is_automorphism()`, `~f`, and `f.inverse()`;
- `X.End()`, `X.Aut()`, and `X.diagonal_morphism()`, whose codomain is the diagonal scheme;
- `X.Pic()`, `X.O(d_1,...,d_r)`, `X.canonical_bundle()`, and `X.Picard_lattice()`;
- `Y.Pic().pullback(f)`, `L.pullback(f)`, and `L.H(0).pullback(f)` for the contravariant maps associated to $f:X\to Y$;
- finite reduced restrictions `L.H(0).pullback(i)` with intrinsic target fibers, matrix, kernel, image, and cokernel;
- scheme-level `X.singular_locus()` and `X.is_singular()` under the equidimensional Jacobian-criterion hypotheses;
- point-local methods `p.local_germ()`, `p.local_ring()`, `p.hypersurface_presentation()`, `p.local_equation()`, `p.multiplicity()`, `p.tangent_cone()`, `p.Milnor_algebra()`, `p.Tjurina_algebra()`, and `p.ADE_type()` under explicit hypotheses;
- `L.cohomology()` and `L.H(i)`;
- `X.cox_ring()` as the abstract $\operatorname{Pic}(X)$-graded algebra of sections;
- `X.cox_ring().polynomial_isomorphism()` as a chosen graded-algebra model map;
- general `GradedAlgebraMorphism.restrict_degree(d)`;
- `L.section_ring()` as the abstract Veronese algebra $\bigoplus_{n\geq0}H^0(X,L^{\otimes n})$;
- `L.complete_linear_system()` and `L.linear_system(V)` with their parameter spaces, base loci, and associated morphisms;
- `system.affine_family()` with the affine coefficient space, universal section/divisor, relative singular scheme, discriminant, specialization, and fixed-locus avoidance polynomial;
- `system.cyclic_cover_family(M,n)` returning a globally glued `CoveredScheme`, its covering morphism, family morphism, affine overlap atlas, and specialized fibers;
- `family.diagonal_sign_automorphism(tau)`, `family.enriques_lift(tau)`, and `lift.quotient()` returning a global covered invariant quotient, its parameter morphism, and its overlap-compatible quotient morphism;
- exact polynomial-coefficient quotient patches for saturation, normal forms, equality, integral-domain certificates, Krull dimension, and certified localization;
- `L.cyclic_cover_datum(s,n)` for validated cyclic-cover root data and `L.cyclic_cover(s,n)` for the resulting native finite morphism;
- `pi.lift_automorphism(f,zeta)`, `pi.lift_automorphisms(f)`, and lift methods `inverse()` and `fixed_subscheme()`;
- `X.Aut().action(G, images)`, `L.linearize(action)`, and `linearization.H_representation(i)`;
- `rho.isotypic_decomposition()` with component characters and bases in the underlying $G$-module;
- `PP`, `AA`, and `OO` as aliases returning native Sage objects;
- `X.affine_cover()` as a cached finite family of affine-chart open immersions;
- relative `Spec`, affine quasi-coherent modules, `VV(E)`, and morphism-based `base_change(q)`.

Automorphism-group elements remain ordinary Sage morphisms in `X.Hom(X)`. Rational points remain the native elements of `X(R)`. Polynomial expressions of sections occur only through the explicit graded-algebra isomorphism to the chosen polynomial model.

A dependent notebook imports these extensions with `%run /home/dzack/research/computations/notebooks/Projective_Scheme_Framework.ipynb`.

## End of framework

The former duplicated tail has been retired. The canonical equivariant, display, and regression cells are Cells 43--50.

In [20]:
pass

Installed finite scheme actions, coordinate linearisations, and isotypic decomposition methods.


pass

In [27]:
pass

Installed structured Sage LaTeX display for framework objects and morphisms.


pass

In [28]:
pass

Semantic algebraic-geometry regressions passed:
Picard_rank_P1xP1 = 2
O44_bidegree = (4, 4)
H0_O44_dimension = 25
H0_O44_basis_cardinality = 25
Cox_polynomial_model = Multivariate Polynomial Ring in sx0, sx1, sy0, sy1 over Complex Field with 53 bits of precision
degree_44_restriction_codomain_dimension = 25
isotypic_dimensions = (13, 12)
generic_section_coefficient_field = Fraction Field of Multivariate Polynomial Ring in c_0, c_1, c_2, c_3, c_4, c_5, c_6, c_7, c_8, c_9, c_10, c_11, c_12, c_13, c_14, c_15, c_16, c_17, c_18, c_19, c_20, c_21, c_22, c_23, c_24 over Complex Field with 53 bits of precision
functorial_Picard_pullback = O(2, 0) on Product of projective spaces P^1 x P^1 over Rational Field
functorial_H0_pullback_polynomial = pa0_pullback^2 + 3*pa1_pullback^2
O44_self_intersection = 32
Picard_gram_P1xP1 = [0 1]
[1 0]
section_ring_degree_two_dimension = 81
complete_linear_system_dimension = 24
complete_linear_system_target_dimension = 24
cyclic_cover_degree = 2
cyclic_cover_alg

pass

In [29]:
pass

Constructor aliases, standard affine covers, and native point tests passed:
PP_dimension = 2
AA_dimension = 3
OO_bidegree = (4, 4)
projective_power_factor_count = 3
product_power_factor_count = 4
projective_point_parent = Set of rational points of Projective Space of dimension 2 over Rational Field
projective_cover_labels = (0, 1, 2)
projective_point_containing_labels = (0, 1, 2)
projective_point_canonical_affine_coordinates = (2, 3)
product_point_all_affine_coordinates = ((2, 4/3), (2, 3/4), (1/2, 4/3), (1/2, 3/4))
